# FreteScript — Cálculo de Frete e Logística

Trabalho acadêmico da disciplina de Compiladores. Este notebook apresenta um compilador completo para uma linguagem de logística simulada, desde a ficha e os programas-exemplo até a execução e a interface Jupyter.

**Integrantes:**

- Bruno dos Reis Ferreira Santos — RA 2564952
- Vinicius de Freitas Ferreira — RA 2373516
- Matheus Francisco — RA 2693112
- Matheus de Oliveira — RA 2369955

As cotações, os despachos, as recusas e os rastreamentos são simulados localmente, sem integração com transportadoras.

## Referência da ficha

Foi consultado `material_apoio_treinoscript_professor.txt`, encontrado na raiz, não em `referencias/`. Seguimos o Passo 0 da TreinoScript: escrever o contrato e os exemplos antes do compilador.

O material recebido apresenta `FICHA_TREINO`, mas não contém `MINHA_FICHA` nem a função `conferir_ficha`. A função abaixo é uma implementação própria dos mínimos solicitados, inspirada nessa metodologia; não é uma transcrição de código ausente do professor.

O exemplo de frete do material usa três argumentos em `cotar`. Neste projeto, adotamos os quatro argumentos solicitados: modalidade, peso, valor e prazo. O material original é somente para consulta local e não deve acompanhar a publicação.

## Ambiente

O projeto utiliza Python, Lark para os analisadores, dataclasses para a AST e o TAC e ipywidgets para o painel. A ficha e os exemplos abaixo iniciam a apresentação da linguagem.

In [1]:
import sys

if sys.version_info < (3, 10):
    raise RuntimeError("Use Python 3.10 ou superior.")
print("Python", sys.version.split()[0])

Python 3.13.17


## Contrato inicial

Declarações usam `tipo nome = expressao;`, atribuições usam `nome = expressao;` e saída usa `mostrar expressao;`. Blocos são delimitados por `{ }`; comandos simples terminam com `;`. Comentários começam com `//`, textos usam aspas duplas e decimais usam ponto. Valores lógicos: `verdadeiro` e `falso`.

Operadores previstos: `+ - * /`, `> < >= <= == !=`, `e ou nao`. Controles: `se condicao { ... } senao { ... }` e `enquanto condicao { ... }`.

`cotar modalidade, peso, valor, prazo;` apresenta uma cotação simulada com o valor calculado pelas expressões do programa (peso em kg, valor em reais, prazo em dias). Modalidade é texto, peso e valor são numéricos, prazo é inteiro. Não há tarifa universal implícita.

`despachar codigo;` registra um envio, `rastrear codigo;` consulta seu estado e `recusar motivo;` informa a recusa. Código e motivo são textos. O estado das entregas é acompanhado pela máquina virtual.

Peso deve ser maior que zero e prazo deve ser de pelo menos um dia (erros). Valor acima de R$ 1.000 gera aviso. Valores determináveis na compilação são conferidos pelo analisador semântico; valores conhecidos apenas durante a execução são conferidos pela VM. O limite de 25 kg do terceiro exemplo pertence apenas àquele cenário, não é regra universal da linguagem.

## Exemplo 1 — Cotação normal

Intenção: cotar quando o cálculo estiver dentro do orçamento. O comportamento correspondente é demonstrado adiante pela máquina virtual.

In [2]:
PROGRAMA_1 = r'''// Cotação de uma encomenda comum
decimal peso = 4.5;
inteiro distancia = 180;
decimal valor = 15.0 + peso * 2.0 + distancia * 0.08;
inteiro prazo = 3;
se valor <= 80.0 {
    cotar "Economico", peso, valor, prazo;
} senao {
    mostrar "Cotacao acima do orcamento";
}'''
print(PROGRAMA_1)

// Cotação de uma encomenda comum
decimal peso = 4.5;
inteiro distancia = 180;
decimal valor = 15.0 + peso * 2.0 + distancia * 0.08;
inteiro prazo = 3;
se valor <= 80.0 {
    cotar "Economico", peso, valor, prazo;
} senao {
    mostrar "Cotacao acima do orcamento";
}


## Exemplo 2 — Despacho com rastreamento

Intenção: despachar um envio autorizado e consultar seu estado duas vezes. O comportamento correspondente é demonstrado adiante pela máquina virtual.

In [3]:
PROGRAMA_2 = r'''// Despacho e duas consultas do mesmo envio
texto codigo = "FS-2026-001";
logico autorizado = verdadeiro;
inteiro consultas = 0;
se autorizado {
    cotar "Expresso", 2.0, 45.0, 1;
    despachar codigo;
    enquanto consultas < 2 {
        rastrear codigo;
        consultas = consultas + 1;
    }
} senao {
    recusar "Envio sem autorizacao";
}'''
print(PROGRAMA_2)

// Despacho e duas consultas do mesmo envio
texto codigo = "FS-2026-001";
logico autorizado = verdadeiro;
inteiro consultas = 0;
se autorizado {
    cotar "Expresso", 2.0, 45.0, 1;
    despachar codigo;
    enquanto consultas < 2 {
        rastrear codigo;
        consultas = consultas + 1;
    }
} senao {
    recusar "Envio sem autorizacao";
}


## Exemplo 3 — Recusa por restrição logística

Intenção: recusar a carga que excede a capacidade escolhida para o veículo. O comportamento correspondente é demonstrado adiante pela máquina virtual.

In [4]:
PROGRAMA_3 = r'''// Limite de peso escolhido para esta operação
decimal peso = 42.0;
decimal limite = 25.0;
se peso > limite {
    recusar "Carga acima da capacidade do veiculo";
} senao {
    cotar "Rodoviario", peso, 120.0, 5;
}'''
print(PROGRAMA_3)

// Limite de peso escolhido para esta operação
decimal peso = 42.0;
decimal limite = 25.0;
se peso > limite {
    recusar "Carga acima da capacidade do veiculo";
} senao {
    cotar "Rodoviario", peso, 120.0, 5;
}


## Ficha da linguagem

Os seis campos abaixo seguem os nomes solicitados para esta entrega.

In [5]:
MINHA_FICHA = {
    "nome": "FreteScript",
    "dominio": "Cálculo de fretes e operações logísticas simuladas",
    "tipos": ["inteiro", "decimal", "texto", "logico"],
    "comandos_dominio": [
        "cotar modalidade, peso, valor, prazo;",
        "despachar codigo;",
        "recusar motivo;",
        "rastrear codigo;",
    ],
    "regras_de_negocio": [
        "peso > 0 (erro)",
        "prazo >= 1 dia (erro)",
        "valor > R$ 1.000 (aviso, não erro)",
    ],
    "programas_exemplo": [PROGRAMA_1, PROGRAMA_2, PROGRAMA_3],
}

for campo, valor in MINHA_FICHA.items():
    if campo != "programas_exemplo":
        print(f"{campo}: {valor}")
print("programas_exemplo:", len(MINHA_FICHA["programas_exemplo"]))

nome: FreteScript
dominio: Cálculo de fretes e operações logísticas simuladas
tipos: ['inteiro', 'decimal', 'texto', 'logico']
comandos_dominio: ['cotar modalidade, peso, valor, prazo;', 'despachar codigo;', 'recusar motivo;', 'rastrear codigo;']
regras_de_negocio: ['peso > 0 (erro)', 'prazo >= 1 dia (erro)', 'valor > R$ 1.000 (aviso, não erro)']
programas_exemplo: 3


## Conferência da ficha

A conferência verifica campos, quantidades mínimas, controle de fluxo aparente e presença dos quatro comandos no conjunto. A busca textual desconsidera textos entre aspas e comentários para evitar contar palavras dentro deles. Ela não valida gramática, tipos, argumentos ou comportamento: não substitui os analisadores e os testes apresentados nas seções seguintes.

In [6]:
import re

def conferir_ficha(ficha):
    """Confere os mínimos documentais; não compila os exemplos."""
    erros = []
    campos = ("nome", "dominio", "tipos", "comandos_dominio",
              "regras_de_negocio", "programas_exemplo")
    if not isinstance(ficha, dict):
        raise ValueError("A ficha precisa ser um dicionário.")
    for campo in campos:
        if campo not in ficha:
            erros.append(f"Campo obrigatório ausente: {campo}.")

    for campo in ("nome", "dominio"):
        valor = ficha.get(campo)
        if not isinstance(valor, str) or not valor.strip():
            erros.append(f"{campo}: informe um texto não vazio.")

    minimos = {"tipos": 3, "comandos_dominio": 2,
               "regras_de_negocio": 2, "programas_exemplo": 3}
    for campo, minimo in minimos.items():
        itens = ficha.get(campo)
        if not isinstance(itens, list) or not all(
            isinstance(item, str) and item.strip() for item in itens
        ):
            erros.append(f"{campo}: informe uma lista de textos não vazios.")
        elif len(set(itens)) < minimo:
            erros.append(f"{campo}: mínimo de {minimo} itens distintos.")

    programas = ficha.get("programas_exemplo")
    fontes = []
    if isinstance(programas, list):
        for numero, programa in enumerate(programas, 1):
            if not isinstance(programa, str):
                continue
            # Apenas inspeção textual da ficha, sem produção de tokens.
            fonte = re.sub(r'"[^"\n]*"|//[^\n]*', " ", programa)
            fontes.append(fonte)
            tem_laco = re.search(r"\benquanto\b", fonte)
            tem_decisao = (re.search(r"\bse\b", fonte)
                           and re.search(r"\bsenao\b", fonte))
            if not (tem_laco or tem_decisao):
                erros.append(f"Exemplo {numero}: inclua se/senao ou enquanto.")

    conjunto = "\n".join(fontes)
    for comando in ("cotar", "despachar", "recusar", "rastrear"):
        if not re.search(rf"\b{comando}\b", conjunto):
            erros.append(f"Comando ausente dos exemplos: {comando}.")

    if erros:
        raise ValueError("\n".join(erros))
    print("Ficha conferida: mínimos documentais atendidos.")
    return True

In [7]:
assert conferir_ficha(MINHA_FICHA)
assert len(MINHA_FICHA["programas_exemplo"]) == 3
assert set(MINHA_FICHA["tipos"]) == {"inteiro", "decimal", "texto", "logico"}
print("3 exemplos conferidos; os 4 comandos estão representados.")
print("Esta conferência não afirma que os programas compilam.")

Ficha conferida: mínimos documentais atendidos.
3 exemplos conferidos; os 4 comandos estão representados.
Esta conferência não afirma que os programas compilam.


## Continuação do notebook

As seções seguintes apresentam, em ordem, análise léxica, análise sintática e AST, análise semântica, TAC, otimização, máquina virtual, testes de integração e painel Jupyter. A conferência acima verifica especificamente a ficha inicial.

# Etapa 02 — Analisador léxico

Nesta etapa, o Lark transforma o código-fonte em tokens. O analisador reconhece palavras reservadas, identificadores, literais, operadores e delimitadores, preservando o lexema, a linha e a coluna. Comentários e espaços são ignorados. As análises sintática e semântica são apresentadas nas seções seguintes.

## Definição dos tokens

As palavras reservadas possuem limite de palavra para não dividir identificadores semelhantes. Números decimais e operadores compostos recebem prioridade sobre suas formas menores. A lista contém somente tokens reais da FreteScript.

In [8]:
from lark import Lark
from lark.exceptions import UnexpectedCharacters

TERMINAIS_FRETESCRIPT = r'''
// Tipos de dados
TIPO_INTEIRO.3: /inteiro(?![A-Za-z0-9_])/
TIPO_DECIMAL.3: /decimal(?![A-Za-z0-9_])/
TIPO_TEXTO.3: /texto(?![A-Za-z0-9_])/
TIPO_LOGICO.3: /logico(?![A-Za-z0-9_])/

// Estruturas e comandos
SE.3: /se(?![A-Za-z0-9_])/
SENAO.3: /senao(?![A-Za-z0-9_])/
ENQUANTO.3: /enquanto(?![A-Za-z0-9_])/
MOSTRAR.3: /mostrar(?![A-Za-z0-9_])/
COTAR.3: /cotar(?![A-Za-z0-9_])/
DESPACHAR.3: /despachar(?![A-Za-z0-9_])/
RECUSAR.3: /recusar(?![A-Za-z0-9_])/
RASTREAR.3: /rastrear(?![A-Za-z0-9_])/

// Valores e operadores lógicos
VERDADEIRO.3: /verdadeiro(?![A-Za-z0-9_])/
FALSO.3: /falso(?![A-Za-z0-9_])/
E_LOGICO.3: /e(?![A-Za-z0-9_])/
OU_LOGICO.3: /ou(?![A-Za-z0-9_])/
NAO_LOGICO.3: /nao(?![A-Za-z0-9_])/

// Literais e identificadores
NUM_DECIMAL.2: /[0-9]+\.[0-9]+/
NUM_INTEIRO: /[0-9]+/
TEXTO: /"[^"\n]*"/
IDENTIFICADOR: /[A-Za-z_][A-Za-z0-9_]*/

// Operadores compostos antes dos simples
MAIOR_IGUAL.2: ">="
MENOR_IGUAL.2: "<="
IGUAL.2: "=="
DIFERENTE.2: "!="
MAIOR: ">"
MENOR: "<"
ATRIBUICAO: "="
MAIS: "+"
MENOS: "-"
MULTIPLICACAO: "*"
DIVISAO: "/"

// Delimitadores
ABRE_CHAVE: "{"
FECHA_CHAVE: "}"
ABRE_PARENTESE: "("
FECHA_PARENTESE: ")"
VIRGULA: ","
PONTO_VIRGULA: ";"

// Elementos ignorados
COMENTARIO.4: /\/\/[^\n]*/
%import common.WS
%ignore WS
%ignore COMENTARIO
'''

PALAVRAS_RESERVADAS = {
    "inteiro", "decimal", "texto", "logico", "se", "senao",
    "enquanto", "mostrar", "cotar", "despachar", "recusar", "rastrear",
    "verdadeiro", "falso", "e", "ou", "nao",
}

## Inicialização do analisador

A regra inicial apenas reúne os terminais, pois esta etapa realiza somente a análise léxica. Os diagnósticos informam o trecho encontrado e uma dica específica, sem tentar decidir se a sequência de tokens forma um programa válido.

In [9]:
NOMES_TOKENS = [
    nome for nome in re.findall(
        r"^([A-Z][A-Z0-9_]*)(?:\.\d+)?:", TERMINAIS_FRETESCRIPT, re.MULTILINE
    )
    if nome != "COMENTARIO"
]

GRAMATICA_LEXICA = (
    "inicio: token*\n"
    "token: " + " | ".join(NOMES_TOKENS) + "\n" + TERMINAIS_FRETESCRIPT
)

ANALISADOR_LEXICO = Lark(
    GRAMATICA_LEXICA,
    start="inicio",
    parser="lalr",
    lexer="basic",
    propagate_positions=True,
)


class ErroLexico(Exception):
    def __init__(self, linha, coluna, trecho, descricao, dica):
        self.linha = linha
        self.coluna = coluna
        self.trecho = trecho
        self.descricao = descricao
        self.dica = dica
        super().__init__(str(self))

    def __str__(self):
        return (
            f"Erro léxico — linha {self.linha}, coluna {self.coluna}\n"
            f"Trecho: {self.trecho!r}\n"
            f"Descrição: {self.descricao}\n"
            f"Dica: {self.dica}"
        )


def _trecho_na_linha(codigo, linha, coluna, caractere):
    linhas = codigo.splitlines()
    texto_linha = linhas[linha - 1] if linha <= len(linhas) else ""
    restante = texto_linha[coluna - 1:]
    if caractere == "'":
        fechamento = restante.find("'", 1)
        return restante[:fechamento + 1] if fechamento >= 0 else restante
    if caractere == '"':
        return restante
    return caractere


def _converter_erro(codigo, erro):
    caractere = codigo[erro.pos_in_stream]
    trecho = _trecho_na_linha(codigo, erro.line, erro.column, caractere)
    if caractere == "'":
        descricao = "Aspas simples não delimitam textos na FreteScript."
        dica = 'Use aspas duplas, por exemplo: "Expresso".'
    elif caractere == '"':
        descricao = "Texto não finalizado com aspas duplas na mesma linha."
        dica = 'Acrescente uma aspa dupla (") ao final do texto.'
    elif caractere.isalpha():
        descricao = f"O caractere {caractere!r} não é permitido em identificadores."
        dica = "Use identificadores sem acentos ou cedilha."
    else:
        descricao = f"O caractere {caractere!r} não é válido na FreteScript."
        if caractere == "@":
            dica = "Remova '@' ou use um operador válido: +, -, *, /, >, <, >=, <=, == ou !=."
        else:
            dica = "Remova o caractere ou substitua-o por um símbolo definido pela linguagem."
    return ErroLexico(erro.line, erro.column, trecho, descricao, dica)


def analisar_lexico(codigo):
    """Retorna os tokens com posição e lexema, ou levanta ErroLexico."""
    if not isinstance(codigo, str):
        raise TypeError("O código-fonte precisa ser uma string.")
    try:
        tokens_lark = list(ANALISADOR_LEXICO.lex(codigo))
    except UnexpectedCharacters as erro:
        raise _converter_erro(codigo, erro) from None
    return [
        {
            "linha": token.line,
            "coluna": token.column,
            "token": token.type,
            "lexema": str(token),
        }
        for token in tokens_lark
    ]


def mostrar_tabela_tokens(codigo, limite=None):
    """Imprime uma tabela simples e devolve a lista produzida pelo analisador."""
    tokens = analisar_lexico(codigo)
    exibidos = tokens if limite is None else tokens[:limite]
    print(f"{'Linha':>5} | {'Coluna':>6} | {'Token':<18} | Lexema")
    print("-" * 58)
    for item in exibidos:
        print(
            f"{item['linha']:>5} | {item['coluna']:>6} | "
            f"{item['token']:<18} | {item['lexema']}"
        )
    if limite is not None and len(tokens) > limite:
        print(f"... mais {len(tokens) - limite} token(s)")
    return tokens


print("Analisador léxico inicializado com Lark.")
print("Tipos de token definidos:", len(NOMES_TOKENS))
print("Palavras reservadas:", len(PALAVRAS_RESERVADAS))

Analisador léxico inicializado com Lark.
Tipos de token definidos: 38
Palavras reservadas: 17


## Exemplo de análise

O primeiro programa aprovado na Etapa 01 é analisado sem qualquer alteração. A chamada abaixo confirma que a entrada inteira pode ser convertida em tokens.

In [10]:
TOKENS_PROGRAMA_1 = analisar_lexico(PROGRAMA_1)
print(f"Programa 1 reconhecido: {len(TOKENS_PROGRAMA_1)} tokens.")
print("Primeiro token:", TOKENS_PROGRAMA_1[0])
print("Último token:", TOKENS_PROGRAMA_1[-1])

Programa 1 reconhecido: 49 tokens.
Primeiro token: {'linha': 2, 'coluna': 1, 'token': 'TIPO_DECIMAL', 'lexema': 'decimal'}
Último token: {'linha': 10, 'coluna': 1, 'token': 'FECHA_CHAVE', 'lexema': '}'}


## Tabela dos tokens encontrados

A tabela é gerada pelos dados reais do analisador. Para manter a saída breve, são exibidos os primeiros 25 tokens do exemplo.

In [11]:
mostrar_tabela_tokens(PROGRAMA_1, limite=25)

Linha | Coluna | Token              | Lexema
----------------------------------------------------------
    2 |      1 | TIPO_DECIMAL       | decimal
    2 |      9 | IDENTIFICADOR      | peso
    2 |     14 | ATRIBUICAO         | =
    2 |     16 | NUM_DECIMAL        | 4.5
    2 |     19 | PONTO_VIRGULA      | ;
    3 |      1 | TIPO_INTEIRO       | inteiro
    3 |      9 | IDENTIFICADOR      | distancia
    3 |     19 | ATRIBUICAO         | =
    3 |     21 | NUM_INTEIRO        | 180
    3 |     24 | PONTO_VIRGULA      | ;
    4 |      1 | TIPO_DECIMAL       | decimal
    4 |      9 | IDENTIFICADOR      | valor
    4 |     15 | ATRIBUICAO         | =
    4 |     17 | NUM_DECIMAL        | 15.0
    4 |     22 | MAIS               | +
    4 |     24 | IDENTIFICADOR      | peso
    4 |     29 | MULTIPLICACAO      | *
    4 |     31 | NUM_DECIMAL        | 2.0
    4 |     35 | MAIS               | +
    4 |     37 | IDENTIFICADOR      | distancia
    4 |     47 | MULTIPLICACAO      | *
   

[{'linha': 2, 'coluna': 1, 'token': 'TIPO_DECIMAL', 'lexema': 'decimal'},
 {'linha': 2, 'coluna': 9, 'token': 'IDENTIFICADOR', 'lexema': 'peso'},
 {'linha': 2, 'coluna': 14, 'token': 'ATRIBUICAO', 'lexema': '='},
 {'linha': 2, 'coluna': 16, 'token': 'NUM_DECIMAL', 'lexema': '4.5'},
 {'linha': 2, 'coluna': 19, 'token': 'PONTO_VIRGULA', 'lexema': ';'},
 {'linha': 3, 'coluna': 1, 'token': 'TIPO_INTEIRO', 'lexema': 'inteiro'},
 {'linha': 3, 'coluna': 9, 'token': 'IDENTIFICADOR', 'lexema': 'distancia'},
 {'linha': 3, 'coluna': 19, 'token': 'ATRIBUICAO', 'lexema': '='},
 {'linha': 3, 'coluna': 21, 'token': 'NUM_INTEIRO', 'lexema': '180'},
 {'linha': 3, 'coluna': 24, 'token': 'PONTO_VIRGULA', 'lexema': ';'},
 {'linha': 4, 'coluna': 1, 'token': 'TIPO_DECIMAL', 'lexema': 'decimal'},
 {'linha': 4, 'coluna': 9, 'token': 'IDENTIFICADOR', 'lexema': 'valor'},
 {'linha': 4, 'coluna': 15, 'token': 'ATRIBUICAO', 'lexema': '='},
 {'linha': 4, 'coluna': 17, 'token': 'NUM_DECIMAL', 'lexema': '15.0'},
 {'l

## Demonstração de erros léxicos

Os três casos abaixo são falhas de formação de tokens. Cada mensagem utiliza a linha e a coluna informadas pelo Lark e oferece uma correção relacionada ao problema encontrado.

In [12]:
EXEMPLOS_ERROS_LEXICOS = {
    "Caractere inválido": "inteiro peso = 10 @ 2;",
    "Aspas simples": "texto modalidade = 'Expresso';",
    "Texto não finalizado": 'mostrar "Frete calculado;',
}

for nome, fonte in EXEMPLOS_ERROS_LEXICOS.items():
    print(f"\n--- {nome} ---")
    print(fonte)
    try:
        analisar_lexico(fonte)
    except ErroLexico as erro:
        print(erro)


--- Caractere inválido ---
inteiro peso = 10 @ 2;
Erro léxico — linha 1, coluna 19
Trecho: '@'
Descrição: O caractere '@' não é válido na FreteScript.
Dica: Remova '@' ou use um operador válido: +, -, *, /, >, <, >=, <=, == ou !=.

--- Aspas simples ---
texto modalidade = 'Expresso';
Erro léxico — linha 1, coluna 20
Trecho: "'Expresso'"
Descrição: Aspas simples não delimitam textos na FreteScript.
Dica: Use aspas duplas, por exemplo: "Expresso".

--- Texto não finalizado ---
mostrar "Frete calculado;
Erro léxico — linha 1, coluna 9
Trecho: '"Frete calculado;'
Descrição: Texto não finalizado com aspas duplas na mesma linha.
Dica: Acrescente uma aspa dupla (") ao final do texto.


## Testes e resultados

Esta bateria verifica apenas o requisito léxico R2. Ela inclui os três programas da Etapa 01, todas as categorias de tokens, posições e diagnósticos. A bateria de integração do compilador é apresentada posteriormente.

In [13]:
def tipos_de(fonte):
    return [item["token"] for item in analisar_lexico(fonte)]


def lexemas_de(fonte):
    return [item["lexema"] for item in analisar_lexico(fonte)]


def gera_erro(fonte, trecho_descricao):
    try:
        analisar_lexico(fonte)
    except ErroLexico as erro:
        return trecho_descricao in erro.descricao and erro.linha > 0 and erro.coluna > 0
    return False


FONTE_RESERVADAS = " ".join(sorted(PALAVRAS_RESERVADAS))
FONTE_OPERADORES_SIMPLES = "+ - * / > < ="
FONTE_OPERADORES_COMPOSTOS = ">= <= == !="
FONTE_DELIMITADORES = "{ } ( ) , ;"

CASOS_TESTE_LEXICO = [
    ("Programa 1 da Etapa 01", lambda: len(analisar_lexico(PROGRAMA_1)) > 0),
    ("Programa 2 da Etapa 01", lambda: len(analisar_lexico(PROGRAMA_2)) > 0),
    ("Programa 3 da Etapa 01", lambda: len(analisar_lexico(PROGRAMA_3)) > 0),
    (
        "17 palavras reservadas",
        lambda: len(analisar_lexico(FONTE_RESERVADAS)) == len(PALAVRAS_RESERVADAS)
        and "IDENTIFICADOR" not in tipos_de(FONTE_RESERVADAS),
    ),
    (
        "Identificadores semelhantes",
        lambda: tipos_de("cotacao cotarFrete enquantoEntrega senao_ se2")
        == ["IDENTIFICADOR"] * 5,
    ),
    (
        "Números inteiros",
        lambda: tipos_de("0 10 2026") == ["NUM_INTEIRO"] * 3,
    ),
    (
        "Números decimais",
        lambda: tipos_de("0.5 4.5 120.0") == ["NUM_DECIMAL"] * 3,
    ),
    (
        "Operadores simples",
        lambda: tipos_de(FONTE_OPERADORES_SIMPLES)
        == ["MAIS", "MENOS", "MULTIPLICACAO", "DIVISAO", "MAIOR", "MENOR", "ATRIBUICAO"],
    ),
    (
        "Operadores compostos",
        lambda: tipos_de(FONTE_OPERADORES_COMPOSTOS)
        == ["MAIOR_IGUAL", "MENOR_IGUAL", "IGUAL", "DIFERENTE"],
    ),
    (
        "Delimitadores",
        lambda: tipos_de(FONTE_DELIMITADORES)
        == ["ABRE_CHAVE", "FECHA_CHAVE", "ABRE_PARENTESE", "FECHA_PARENTESE", "VIRGULA", "PONTO_VIRGULA"],
    ),
    (
        "Comentários ignorados",
        lambda: lexemas_de("inteiro // comentário\npeso") == ["inteiro", "peso"],
    ),
    (
        "Linha e coluna",
        lambda: analisar_lexico("\n  decimal peso = 4.5;")[0]
        == {"linha": 2, "coluna": 3, "token": "TIPO_DECIMAL", "lexema": "decimal"},
    ),
    (
        "Lexema original do texto",
        lambda: analisar_lexico('texto nome = "Frete Expresso";')[3]["lexema"]
        == '"Frete Expresso"',
    ),
    (
        "Caractere inválido",
        lambda: gera_erro("inteiro peso = 10 @ 2;", "não é válido"),
    ),
    (
        "Aspas simples",
        lambda: gera_erro("texto modalidade = 'Expresso';", "Aspas simples"),
    ),
    (
        "Texto não finalizado",
        lambda: gera_erro('mostrar "Frete calculado;', "Texto não finalizado"),
    ),
    (
        "Quantidade real de tokens",
        lambda: len(NOMES_TOKENS) >= 20 and len(set(NOMES_TOKENS)) == len(NOMES_TOKENS),
    ),
]

TESTES_APROVADOS = 0
for nome, teste in CASOS_TESTE_LEXICO:
    try:
        passou = bool(teste())
    except Exception as erro:
        passou = False
        print(f"FALHOU — {nome}: {erro}")
    else:
        print(f"{'OK' if passou else 'FALHOU'} — {nome}")
    TESTES_APROVADOS += int(passou)

TESTES_EXECUTADOS = len(CASOS_TESTE_LEXICO)
TESTES_REPROVADOS = TESTES_EXECUTADOS - TESTES_APROVADOS
print(
    f"\nResultado: {TESTES_EXECUTADOS} executados, "
    f"{TESTES_APROVADOS} aprovados e {TESTES_REPROVADOS} reprovados."
)
assert TESTES_REPROVADOS == 0, "Há testes léxicos reprovados."

OK — Programa 1 da Etapa 01
OK — Programa 2 da Etapa 01
OK — Programa 3 da Etapa 01
OK — 17 palavras reservadas
OK — Identificadores semelhantes
OK — Números inteiros
OK — Números decimais
OK — Operadores simples
OK — Operadores compostos
OK — Delimitadores
OK — Comentários ignorados
OK — Linha e coluna
OK — Lexema original do texto
OK — Caractere inválido
OK — Aspas simples
OK — Texto não finalizado
OK — Quantidade real de tokens

Resultado: 17 executados, 17 aprovados e 0 reprovados.


# Etapa 03 — Análise sintática e AST

O analisador sintático confere como os tokens formam comandos e expressões. O parser LALR produz uma árvore sintática abstrata (AST) com objetos simples, utilizados pela análise semântica, pelo gerador TAC e pela VM. A verificação de tipos e a execução aparecem nas seções posteriores.

## Gramática Lark

A gramática reutiliza `TERMINAIS_FRETESCRIPT`, definidos no R2. Cada nível de expressão representa uma precedência: `ou`, `e`, `nao`, comparação, soma/subtração, multiplicação/divisão e operações unárias/primárias.

In [14]:
REGRAS_SINTATICAS = r'''
?inicio: programa
programa: comando*

?comando: declaracao
        | atribuicao
        | mostrar
        | cotar
        | despachar
        | recusar
        | rastrear
        | condicional
        | enquanto

declaracao: tipo IDENTIFICADOR (ATRIBUICAO expr)? PONTO_VIRGULA
tipo: TIPO_INTEIRO | TIPO_DECIMAL | TIPO_TEXTO | TIPO_LOGICO
atribuicao: IDENTIFICADOR ATRIBUICAO expr PONTO_VIRGULA
mostrar: MOSTRAR expr PONTO_VIRGULA

cotar: COTAR expr VIRGULA expr VIRGULA expr VIRGULA expr PONTO_VIRGULA
despachar: DESPACHAR expr PONTO_VIRGULA
recusar: RECUSAR expr PONTO_VIRGULA
rastrear: RASTREAR expr PONTO_VIRGULA

condicional: SE expr bloco (SENAO bloco)?
enquanto: ENQUANTO expr bloco
bloco: ABRE_CHAVE comando* FECHA_CHAVE

// Quanto mais profunda a regra, maior a precedência.
?expr: expr_ou
?expr_ou: expr_e
        | expr_ou OU_LOGICO expr_e              -> binop
?expr_e: negacao
       | expr_e E_LOGICO negacao                -> binop
?negacao: comparacao
        | NAO_LOGICO negacao                    -> unop
?comparacao: soma
           | soma operador_relacional soma      -> binop
?soma: produto
     | soma (MAIS | MENOS) produto              -> binop
?produto: unario
        | produto (MULTIPLICACAO | DIVISAO) unario -> binop
?unario: primario
       | MENOS unario                           -> unop
?primario: NUM_INTEIRO                          -> lit_inteiro
         | NUM_DECIMAL                          -> lit_decimal
         | TEXTO                                -> lit_texto
         | VERDADEIRO                           -> lit_logico
         | FALSO                                -> lit_logico
         | IDENTIFICADOR                        -> var
         | ABRE_PARENTESE expr FECHA_PARENTESE -> agrupamento

operador_relacional: MAIOR
                    | MENOR
                    | MAIOR_IGUAL
                    | MENOR_IGUAL
                    | IGUAL
                    | DIFERENTE
'''

GRAMATICA_FRETESCRIPT = REGRAS_SINTATICAS + TERMINAIS_FRETESCRIPT
print("Gramática sintática definida com 7 níveis de precedência.")

Gramática sintática definida com 7 níveis de precedência.


## Classes da AST

Todos os comandos possuem linha e coluna. As expressões também reservam o campo `tipo`, inicialmente vazio, para a análise semântica. Cada comando logístico possui sua própria classe.

In [15]:
from dataclasses import dataclass, field
from lark import Transformer, v_args
from lark.exceptions import UnexpectedEOF, UnexpectedToken


@dataclass
class Expr:
    linha: int
    coluna: int
    tipo: str | None = field(default=None, init=False)


@dataclass
class Lit(Expr):
    valor: object


@dataclass
class Var(Expr):
    nome: str


@dataclass
class BinOp(Expr):
    op: str
    esq: Expr
    dir: Expr


@dataclass
class UnOp(Expr):
    op: str
    expr: Expr


@dataclass
class Decl:
    linha: int
    coluna: int
    tipo: str
    nome: str
    expr: Expr | None = None


@dataclass
class Atrib:
    linha: int
    coluna: int
    nome: str
    expr: Expr


@dataclass
class Mostrar:
    linha: int
    coluna: int
    expr: Expr


@dataclass
class Se:
    linha: int
    coluna: int
    condicao: Expr
    entao: list
    senao: list | None = None


@dataclass
class Enquanto:
    linha: int
    coluna: int
    condicao: Expr
    corpo: list


@dataclass
class Cotar:
    linha: int
    coluna: int
    modalidade: Expr
    peso: Expr
    valor: Expr
    prazo: Expr


@dataclass
class Despachar:
    linha: int
    coluna: int
    codigo: Expr


@dataclass
class Recusar:
    linha: int
    coluna: int
    motivo: Expr


@dataclass
class Rastrear:
    linha: int
    coluna: int
    codigo: Expr


@dataclass
class Programa:
    comandos: list
    linha: int = 1
    coluna: int = 1


CLASSES_AST = (
    Programa, Decl, Atrib, Mostrar, Se, Enquanto, Lit, Var, BinOp, UnOp,
    Cotar, Despachar, Recusar, Rastrear,
)
print("Classes da AST:", ", ".join(classe.__name__ for classe in CLASSES_AST))

Classes da AST: Programa, Decl, Atrib, Mostrar, Se, Enquanto, Lit, Var, BinOp, UnOp, Cotar, Despachar, Recusar, Rastrear


## Transformer

O `MontaAST` é chamado a cada redução da gramática. Os métodos recebem os tokens nomeados, retiram somente a pontuação e preservam comandos, blocos, argumentos e subárvores de expressão.

In [16]:
@v_args(inline=True)
class MontaAST(Transformer):
    def programa(self, *comandos):
        if comandos:
            return Programa(list(comandos), comandos[0].linha, comandos[0].coluna)
        return Programa([])

    def bloco(self, _abre, *itens):
        return list(itens[:-1])  # o último item é FECHA_CHAVE

    def tipo(self, token):
        return token

    def declaracao(self, tipo, nome, *resto):
        expr = resto[1] if len(resto) == 3 else None
        return Decl(tipo.line, tipo.column, str(tipo), str(nome), expr)

    def atribuicao(self, nome, _atribuicao, expr, _ponto_virgula):
        return Atrib(nome.line, nome.column, str(nome), expr)

    def mostrar(self, palavra, expr, _ponto_virgula):
        return Mostrar(palavra.line, palavra.column, expr)

    def cotar(self, palavra, modalidade, _v1, peso, _v2, valor, _v3, prazo, _pv):
        return Cotar(palavra.line, palavra.column, modalidade, peso, valor, prazo)

    def despachar(self, palavra, codigo, _ponto_virgula):
        return Despachar(palavra.line, palavra.column, codigo)

    def recusar(self, palavra, motivo, _ponto_virgula):
        return Recusar(palavra.line, palavra.column, motivo)

    def rastrear(self, palavra, codigo, _ponto_virgula):
        return Rastrear(palavra.line, palavra.column, codigo)

    def condicional(self, palavra, condicao, entao, *resto):
        senao = resto[1] if len(resto) == 2 else None
        return Se(palavra.line, palavra.column, condicao, entao, senao)

    def enquanto(self, palavra, condicao, corpo):
        return Enquanto(palavra.line, palavra.column, condicao, corpo)

    def operador_relacional(self, token):
        return token

    def binop(self, esquerda, operador, direita):
        return BinOp(operador.line, operador.column, str(operador), esquerda, direita)

    def unop(self, operador, expr):
        return UnOp(operador.line, operador.column, str(operador), expr)

    def agrupamento(self, _abre, expr, _fecha):
        return expr

    def lit_inteiro(self, token):
        return Lit(token.line, token.column, int(token))

    def lit_decimal(self, token):
        return Lit(token.line, token.column, float(token))

    def lit_texto(self, token):
        return Lit(token.line, token.column, str(token)[1:-1])

    def lit_logico(self, token):
        return Lit(token.line, token.column, token.type == "VERDADEIRO")

    def var(self, token):
        return Var(token.line, token.column, str(token))

## Parser e diagnósticos

Antes do parser contextual, `analisar_sintaxe` chama o léxico do R2. Assim, caracteres realmente inválidos continuam gerando `ErroLexico`; sequências válidas de tokens em uma ordem incorreta geram `ErroSintatico`.

In [17]:
PARSER_FRETESCRIPT = Lark(
    GRAMATICA_FRETESCRIPT,
    start="inicio",
    parser="lalr",
    lexer="contextual",
    propagate_positions=True,
)

ROTULOS_TERMINAIS = {
    "PONTO_VIRGULA": "';'", "VIRGULA": "','", "ABRE_CHAVE": "'{'",
    "FECHA_CHAVE": "'}'", "ABRE_PARENTESE": "'('",
    "FECHA_PARENTESE": "')'", "ATRIBUICAO": "'='",
    "IDENTIFICADOR": "um identificador", "NUM_INTEIRO": "um número inteiro",
    "NUM_DECIMAL": "um número decimal", "TEXTO": "um texto entre aspas duplas",
    "VERDADEIRO": "'verdadeiro'", "FALSO": "'falso'", "MENOS": "'-'",
    "NAO_LOGICO": "'nao'",
}
INICIOS_EXPRESSAO = {
    "IDENTIFICADOR", "NUM_INTEIRO", "NUM_DECIMAL", "TEXTO", "VERDADEIRO",
    "FALSO", "MENOS", "NAO_LOGICO", "ABRE_PARENTESE",
}


class ErroSintatico(Exception):
    fase = "sintática"

    def __init__(self, linha, coluna, encontrado, esperados, descricao, dica):
        self.linha = linha
        self.coluna = coluna
        self.encontrado = encontrado
        self.esperados = list(esperados)
        self.descricao = descricao
        self.dica = dica
        super().__init__(str(self))

    def __str__(self):
        esperado = ", ".join(self.esperados) if self.esperados else "continuação válida"
        return (
            f"Erro sintático — linha {self.linha}, coluna {self.coluna}\n"
            f"Encontrado: {self.encontrado!r}\n"
            f"Descrição: {self.descricao}\n"
            f"Esperado: {esperado}\n"
            f"Dica: {self.dica}"
        )


def _esperados_do_erro(erro):
    try:
        nomes = erro.interactive_parser.accepts() - {"$END"}
    except (AttributeError, TypeError):
        nomes = set()
    nomes = nomes or set(getattr(erro, "expected", set())) - {"$END"}
    return nomes


def _diagnostico_sintatico(codigo, erro):
    nomes = _esperados_do_erro(erro)
    esperados = sorted({ROTULOS_TERMINAIS.get(nome, nome.lower()) for nome in nomes})

    token = getattr(erro, "token", None)
    fim = token is None or token.type == "$END"
    if fim:
        linhas = codigo.splitlines() or [""]
        linha = len(linhas)
        coluna = len(linhas[-1]) + 1
        encontrado = "<fim do código>"
    else:
        linha = token.line
        coluna = token.column
        encontrado = str(token)

    if "PONTO_VIRGULA" in nomes:
        descricao = "O comando anterior não foi finalizado com ponto e vírgula."
        dica = "Adicione ';' ao final do comando anterior."
    elif fim and "FECHA_CHAVE" in nomes:
        descricao = "O programa terminou antes do fechamento do bloco."
        dica = "Feche o bloco aberto adicionando '}'."
    elif nomes & INICIOS_EXPRESSAO:
        descricao = "A expressão está incompleta ou começa com um token inválido."
        dica = "Escreva uma expressão válida após o operador ou a atribuição."
    elif "FECHA_CHAVE" in nomes:
        descricao = "O conteúdo do bloco não está completo."
        dica = "Confira os comandos e feche o bloco com '}'."
    else:
        descricao = "O token encontrado não pode aparecer nesta posição."
        dica = "Confira a estrutura do comando e os elementos esperados."

    return ErroSintatico(linha, coluna, encontrado, esperados, descricao, dica)


def analisar_sintaxe(codigo):
    """Valida a sintaxe e devolve a raiz Programa da AST."""
    analisar_lexico(codigo)  # mantém os diagnósticos léxicos do R2
    try:
        arvore_lark = PARSER_FRETESCRIPT.parse(codigo)
        return MontaAST().transform(arvore_lark)
    except (UnexpectedToken, UnexpectedEOF, UnexpectedCharacters) as erro:
        raise _diagnostico_sintatico(codigo, erro) from None


print("Parser LALR inicializado com lexer contextual.")

Parser LALR inicializado com lexer contextual.


## Visualização da AST

A função abaixo percorre os objetos produzidos pelo `MontaAST`. O desenho não é fixo: qualquer programa aceito pode ser exibido com a mesma função.

In [18]:
def _rotulo_ast(no):
    posicao = f" [{no.linha}:{no.coluna}]" if hasattr(no, "linha") else ""
    if isinstance(no, Programa):
        return "Programa"
    if isinstance(no, Decl):
        return f"Decl {no.tipo} {no.nome}{posicao}"
    if isinstance(no, Atrib):
        return f"Atrib {no.nome}{posicao}"
    if isinstance(no, BinOp):
        return f"BinOp {no.op!r}{posicao}"
    if isinstance(no, UnOp):
        return f"UnOp {no.op!r}{posicao}"
    if isinstance(no, Lit):
        return f"Lit {no.valor!r}{posicao}"
    if isinstance(no, Var):
        return f"Var {no.nome}{posicao}"
    return f"{type(no).__name__}{posicao}"


def _filhos_ast(no):
    if isinstance(no, Programa):
        return [(f"comando {i}: ", comando) for i, comando in enumerate(no.comandos, 1)]
    if isinstance(no, Decl):
        return [] if no.expr is None else [("valor: ", no.expr)]
    if isinstance(no, Atrib):
        return [("valor: ", no.expr)]
    if isinstance(no, Mostrar):
        return [("expr: ", no.expr)]
    if isinstance(no, Se):
        filhos = [("condição: ", no.condicao), ("então: ", no.entao)]
        return filhos + ([] if no.senao is None else [("senão: ", no.senao)])
    if isinstance(no, Enquanto):
        return [("condição: ", no.condicao), ("corpo: ", no.corpo)]
    if isinstance(no, Cotar):
        return [("modalidade: ", no.modalidade), ("peso: ", no.peso),
                ("valor: ", no.valor), ("prazo: ", no.prazo)]
    if isinstance(no, (Despachar, Rastrear)):
        return [("código: ", no.codigo)]
    if isinstance(no, Recusar):
        return [("motivo: ", no.motivo)]
    if isinstance(no, BinOp):
        return [("esq: ", no.esq), ("dir: ", no.dir)]
    if isinstance(no, UnOp):
        return [("expr: ", no.expr)]
    return []


def mostrar_ast(ast):
    """Exibe uma AST ou lista de comandos como árvore textual."""
    def visitar(no, prefixo="", ultimo=True, nome=""):
        conector = "└── " if ultimo else "├── "
        if isinstance(no, list):
            print(prefixo + conector + nome + f"Bloco ({len(no)} comando(s))")
            filhos = [("", item) for item in no]
        else:
            print(prefixo + conector + nome + _rotulo_ast(no))
            filhos = _filhos_ast(no)
        novo_prefixo = prefixo + ("    " if ultimo else "│   ")
        for indice, (rotulo, filho) in enumerate(filhos):
            visitar(filho, novo_prefixo, indice == len(filhos) - 1, rotulo)

    visitar(ast)


AST_PRECEDENCIA = analisar_sintaxe("decimal total = 10.0 + 2.0 * 5.0;")
mostrar_ast(AST_PRECEDENCIA)

└── Programa
    └── comando 1: Decl decimal total [1:1]
        └── valor: BinOp '+' [1:22]
            ├── esq: Lit 10.0 [1:17]
            └── dir: BinOp '*' [1:28]
                ├── esq: Lit 2.0 [1:24]
                └── dir: Lit 5.0 [1:30]


## Programas completos e erros sintáticos

Os três programas aprovados são transformados em ASTs. Em seguida, os três erros obrigatórios são apresentados como diagnósticos controlados, sem interromper a execução do notebook.

In [19]:
ASTS_EXEMPLOS = [analisar_sintaxe(programa) for programa in MINHA_FICHA["programas_exemplo"]]
for numero, ast in enumerate(ASTS_EXEMPLOS, 1):
    print(f"Programa {numero}: AST gerada com {len(ast.comandos)} comando(s) no nível principal.")

EXEMPLOS_ERROS_SINTATICOS = {
    "Ponto e vírgula ausente": "inteiro distancia = 450\nmostrar distancia;",
    "Chave de fechamento ausente": 'se distancia > 100 {\n    mostrar "Frete longo";',
    "Expressão incompleta": "decimal valor = 10.0 + ;",
}

for nome, fonte in EXEMPLOS_ERROS_SINTATICOS.items():
    print(f"\n--- {nome} ---")
    print(fonte)
    try:
        analisar_sintaxe(fonte)
    except ErroSintatico as erro:
        print(erro)

Programa 1: AST gerada com 5 comando(s) no nível principal.
Programa 2: AST gerada com 4 comando(s) no nível principal.
Programa 3: AST gerada com 3 comando(s) no nível principal.

--- Ponto e vírgula ausente ---
inteiro distancia = 450
mostrar distancia;
Erro sintático — linha 2, coluna 1
Encontrado: 'mostrar'
Descrição: O comando anterior não foi finalizado com ponto e vírgula.
Esperado: '-', ';', diferente, divisao, e_logico, igual, maior, maior_igual, mais, menor, menor_igual, multiplicacao, ou_logico
Dica: Adicione ';' ao final do comando anterior.

--- Chave de fechamento ausente ---
se distancia > 100 {
    mostrar "Frete longo";
Erro sintático — linha 2, coluna 27
Encontrado: '<fim do código>'
Descrição: O programa terminou antes do fechamento do bloco.
Esperado: '}', cotar, despachar, enquanto, mostrar, rastrear, recusar, se, tipo_decimal, tipo_inteiro, tipo_logico, tipo_texto, um identificador
Dica: Feche o bloco aberto adicionando '}'.

--- Expressão incompleta ---
decimal v

## Testes automatizados do R3

Os testes abaixo inspecionam classes, campos, blocos e subárvores; não verificam somente a ausência de exceções. A bateria léxica R2 já foi executada anteriormente no mesmo kernel e seu resultado também é conferido.

In [20]:
def todos_comandos(no):
    encontrados = []
    if isinstance(no, Programa):
        for comando in no.comandos:
            encontrados.extend(todos_comandos(comando))
    elif isinstance(no, list):
        for comando in no:
            encontrados.extend(todos_comandos(comando))
    else:
        encontrados.append(no)
        if isinstance(no, Se):
            encontrados.extend(todos_comandos(no.entao))
            if no.senao is not None:
                encontrados.extend(todos_comandos(no.senao))
        elif isinstance(no, Enquanto):
            encontrados.extend(todos_comandos(no.corpo))
    return encontrados


def erro_sintatico(fonte):
    try:
        analisar_sintaxe(fonte)
    except ErroSintatico as erro:
        return erro
    return None


def erro_lexico_preservado():
    try:
        analisar_sintaxe("inteiro peso = 10 @ 2;")
    except ErroLexico as erro:
        return erro.linha == 1 and erro.coluna == 19 and "não é válido" in erro.descricao
    return False


def teste_tipos_declaracao():
    ast = analisar_sintaxe(
        'inteiro a = 1; decimal b = 2.0; texto c = "x"; logico d = falso;'
    )
    return [no.tipo for no in ast.comandos] == ["inteiro", "decimal", "texto", "logico"]


def teste_precedencia_sem_parenteses():
    expr = analisar_sintaxe("decimal total = 10.0 + 2.0 * 5.0;").comandos[0].expr
    return isinstance(expr, BinOp) and expr.op == "+" and isinstance(expr.dir, BinOp) and expr.dir.op == "*"


def teste_precedencia_com_parenteses():
    expr = analisar_sintaxe("decimal total = (10.0 + 2.0) * 5.0;").comandos[0].expr
    return isinstance(expr, BinOp) and expr.op == "*" and isinstance(expr.esq, BinOp) and expr.esq.op == "+"


def teste_logicos():
    expr = analisar_sintaxe("logico permitido = peso > 0.0 e prazo >= 1 ou falso;").comandos[0].expr
    return (
        isinstance(expr, BinOp) and expr.op == "ou"
        and isinstance(expr.esq, BinOp) and expr.esq.op == "e"
        and isinstance(expr.esq.esq, BinOp) and expr.esq.esq.op == ">"
        and isinstance(expr.esq.dir, BinOp) and expr.esq.dir.op == ">="
    )


def teste_associatividade():
    expr = analisar_sintaxe("inteiro x = 10 - 3 - 2;").comandos[0].expr
    return isinstance(expr, BinOp) and expr.op == "-" and isinstance(expr.esq, BinOp) and expr.esq.op == "-"


def teste_erros_e_posicoes(fonte, trecho_dica):
    erro = erro_sintatico(fonte)
    return (
        erro is not None and erro.fase == "sintática" and erro.linha > 0
        and erro.coluna > 0 and bool(erro.esperados) and trecho_dica in erro.dica
    )


CASOS_TESTE_SINTATICO = [
    ("Programa 1 completo", lambda: len(todos_comandos(ASTS_EXEMPLOS[0])) == 7
     and any(isinstance(no, Cotar) for no in todos_comandos(ASTS_EXEMPLOS[0]))),
    ("Programa 2 completo", lambda: len(todos_comandos(ASTS_EXEMPLOS[1])) == 10
     and {Cotar, Despachar, Rastrear, Recusar}.issubset({type(no) for no in todos_comandos(ASTS_EXEMPLOS[1])})),
    ("Programa 3 completo", lambda: len(todos_comandos(ASTS_EXEMPLOS[2])) == 5
     and any(isinstance(no, Recusar) for no in todos_comandos(ASTS_EXEMPLOS[2]))),
    ("Declarações dos quatro tipos", teste_tipos_declaracao),
    ("Declaração sem inicialização", lambda: analisar_sintaxe("inteiro tentativas;").comandos[0].expr is None),
    ("Atribuição", lambda: isinstance(analisar_sintaxe("tentativas = tentativas + 1;").comandos[0], Atrib)),
    ("Comando mostrar", lambda: isinstance(analisar_sintaxe('mostrar "Calculando";').comandos[0], Mostrar)),
    ("Cotar com quatro argumentos", lambda: isinstance(analisar_sintaxe('cotar "Expresso", peso, valor, prazo;').comandos[0], Cotar)
     and len(_filhos_ast(analisar_sintaxe('cotar "Expresso", peso, valor, prazo;').comandos[0])) == 4),
    ("Comando despachar", lambda: isinstance(analisar_sintaxe("despachar codigo;").comandos[0], Despachar)),
    ("Comando recusar", lambda: isinstance(analisar_sintaxe('recusar "Motivo";').comandos[0], Recusar)),
    ("Comando rastrear", lambda: isinstance(analisar_sintaxe("rastrear codigo;").comandos[0], Rastrear)),
    ("Condicional se/senao", lambda: isinstance(analisar_sintaxe('se peso > 30.0 { recusar "x"; } senao { mostrar peso; }').comandos[0], Se)
     and len(analisar_sintaxe('se peso > 30.0 { recusar "x"; } senao { mostrar peso; }').comandos[0].senao) == 1),
    ("Laço enquanto", lambda: isinstance(analisar_sintaxe("enquanto tentativas < 3 { tentativas = tentativas + 1; }").comandos[0], Enquanto)),
    ("Blocos aninhados", lambda: isinstance(analisar_sintaxe("enquanto verdadeiro { se falso { mostrar 1; } }").comandos[0].corpo[0], Se)),
    ("Precedência entre + e *", teste_precedencia_sem_parenteses),
    ("Precedência com parênteses", teste_precedencia_com_parenteses),
    ("Operadores lógicos e comparações", teste_logicos),
    ("Negação lógica", lambda: isinstance(analisar_sintaxe("logico x = nao falso;").comandos[0].expr, UnOp)
     and analisar_sintaxe("logico x = nao falso;").comandos[0].expr.op == "nao"),
    ("Associatividade à esquerda", teste_associatividade),
    ("Sinal negativo unário", lambda: isinstance(analisar_sintaxe("decimal x = -2.5;").comandos[0].expr, UnOp)),
    ("Localização nos nós", lambda: analisar_sintaxe("\n  inteiro x = 1;").comandos[0].linha == 2
     and analisar_sintaxe("\n  inteiro x = 1;").comandos[0].coluna == 3),
    ("Erro: ponto e vírgula", lambda: teste_erros_e_posicoes("inteiro distancia = 450\nmostrar distancia;", ";")),
    ("Erro: chave não fechada", lambda: teste_erros_e_posicoes('se distancia > 100 {\n mostrar "longo";', "}")),
    ("Erro: expressão incompleta", lambda: teste_erros_e_posicoes("decimal valor = 10.0 + ;", "expressão")),
    ("Erro léxico preservado", erro_lexico_preservado),
]

TESTES_R3_APROVADOS = 0
for nome, teste in CASOS_TESTE_SINTATICO:
    try:
        passou = bool(teste())
    except Exception as erro:
        passou = False
        print(f"FALHOU — {nome}: {type(erro).__name__}: {erro}")
    else:
        print(f"{'OK' if passou else 'FALHOU'} — {nome}")
    TESTES_R3_APROVADOS += int(passou)

TESTES_R3_EXECUTADOS = len(CASOS_TESTE_SINTATICO)
TESTES_R3_REPROVADOS = TESTES_R3_EXECUTADOS - TESTES_R3_APROVADOS
print(
    f"\nR3: {TESTES_R3_EXECUTADOS} executados, "
    f"{TESTES_R3_APROVADOS} aprovados e {TESTES_R3_REPROVADOS} reprovados."
)
print(
    f"Regressão R2: {TESTES_EXECUTADOS} executados, "
    f"{TESTES_APROVADOS} aprovados e {TESTES_REPROVADOS} reprovados."
)
assert TESTES_R3_REPROVADOS == 0, "Há testes sintáticos reprovados."
assert TESTES_REPROVADOS == 0, "A regressão léxica apresentou falhas."

OK — Programa 1 completo
OK — Programa 2 completo
OK — Programa 3 completo
OK — Declarações dos quatro tipos
OK — Declaração sem inicialização
OK — Atribuição
OK — Comando mostrar
OK — Cotar com quatro argumentos
OK — Comando despachar
OK — Comando recusar
OK — Comando rastrear
OK — Condicional se/senao
OK — Laço enquanto
OK — Blocos aninhados
OK — Precedência entre + e *
OK — Precedência com parênteses
OK — Operadores lógicos e comparações
OK — Negação lógica
OK — Associatividade à esquerda
OK — Sinal negativo unário
OK — Localização nos nós
OK — Erro: ponto e vírgula
OK — Erro: chave não fechada
OK — Erro: expressão incompleta
OK — Erro léxico preservado

R3: 25 executados, 25 aprovados e 0 reprovados.
Regressão R2: 17 executados, 17 aprovados e 0 reprovados.


# Etapa 04 — Análise semântica

A análise semântica percorre a AST do R3 para conferir declarações, escopos, inicialização, tipos e regras logísticas. Ela não executa o programa. Erros impedem a aprovação semântica; avisos apenas chamam atenção para uma situação válida, mas relevante.

## Tabela de símbolos e escopos

A tabela usa uma pilha de escopos. O primeiro é global; cada bloco de `se`, `senao` ou `enquanto` abre um escopo. A busca começa no bloco atual e segue para os externos, permitindo sombreamento sem transformar atribuições externas em declarações locais.

In [21]:
DESCONHECIDO = object()


@dataclass
class Simbolo:
    nome: str
    tipo: str
    escopo: str
    linha: int
    coluna: int
    inicializada: bool = False
    valor_conhecido: object = field(default=DESCONHECIDO, repr=False)


class TabelaSimbolos:
    """Pilha de escopos com histórico para exibição posterior."""
    def __init__(self):
        self._pilha = [("global", {})]
        self._contador = 0
        self.simbolos = []

    @property
    def escopo_atual(self):
        return self._pilha[-1][0]

    def abrir(self, rotulo):
        self._contador += 1
        nome = f"{rotulo}#{self._contador}"
        self._pilha.append((nome, {}))
        return nome

    def fechar(self):
        if len(self._pilha) == 1:
            raise RuntimeError("O escopo global não pode ser fechado.")
        return self._pilha.pop()

    def declarar(self, nome, tipo, linha, coluna):
        _, atual = self._pilha[-1]
        if nome in atual:
            return None
        simbolo = Simbolo(nome, tipo, self.escopo_atual, linha, coluna)
        atual[nome] = simbolo
        self.simbolos.append(simbolo)
        return simbolo

    def buscar(self, nome):
        for _, escopo in reversed(self._pilha):
            if nome in escopo:
                return escopo[nome]
        return None


def _texto_valor(valor):
    return "desconhecido" if valor is DESCONHECIDO else repr(valor)


def mostrar_tabela_simbolos(tabela):
    print(f"{'Nome':<14} | {'Tipo':<8} | {'Escopo':<16} | {'Declaração':<10} | {'Inicializada':<11} | Valor")
    print("-" * 91)
    for simbolo in tabela.simbolos:
        print(
            f"{simbolo.nome:<14} | {simbolo.tipo:<8} | {simbolo.escopo:<16} | "
            f"{simbolo.linha}:{simbolo.coluna:<8} | "
            f"{'sim' if simbolo.inicializada else 'não':<11} | "
            f"{_texto_valor(simbolo.valor_conhecido)}"
        )

## Tipos, coerção e resultados

Os quatro tipos da linguagem são distintos. Somente `inteiro` pode ser convertido automaticamente para `decimal`. A coerção é registrada como um nó da AST para que o TAC e a VM a preservem. Cada análise devolve uma AST tipada, a tabela, erros, avisos e a situação de aprovação.

In [22]:
@dataclass
class Coercao(Expr):
    expr: Expr
    destino: str = "decimal"


@dataclass
class DiagnosticoSemantico:
    severidade: str
    codigo: str
    linha: int
    coluna: int
    descricao: str
    dica: str
    fase: str = field(default="semântica", init=False)

    def __str__(self):
        titulo = "Erro" if self.severidade == "erro" else "Aviso"
        return (
            f"{titulo} semântico — linha {self.linha}, coluna {self.coluna}\n"
            f"Descrição: {self.descricao}\n"
            f"Dica: {self.dica}"
        )


@dataclass
class ResultadoSemantico:
    ast: Programa
    tabela: TabelaSimbolos
    diagnosticos: list

    @property
    def erros(self):
        return [d for d in self.diagnosticos if d.severidade == "erro"]

    @property
    def avisos(self):
        return [d for d in self.diagnosticos if d.severidade == "aviso"]

    @property
    def aprovado(self):
        return not self.erros


TIPOS_NUMERICOS = {"inteiro", "decimal"}
OPERADORES_ARITMETICOS = {"+", "-", "*", "/"}
OPERADORES_ORDEM = {">", "<", ">=", "<="}
OPERADORES_IGUALDADE = {"==", "!="}


_rotulo_ast_r3 = _rotulo_ast
_filhos_ast_r3 = _filhos_ast


def _rotulo_ast(no):
    if isinstance(no, Coercao):
        return f"Coercao para {no.destino} [{no.linha}:{no.coluna}]"
    rotulo = _rotulo_ast_r3(no)
    if isinstance(no, Expr) and no.tipo is not None:
        rotulo += f" : {no.tipo}"
    return rotulo


def _filhos_ast(no):
    if isinstance(no, Coercao):
        return [("expr: ", no.expr)]
    return _filhos_ast_r3(no)

## Verificação de tipos e fluxo conservador

O analisador registra vários diagnósticos independentes na mesma passagem. Uma variável inicializada apenas dentro de uma decisão ou laço continua possivelmente não inicializada depois do bloco. Se uma variável já inicializada for alterada em um caminho incerto, ela permanece inicializada, mas seu valor constante passa a ser desconhecido.

In [23]:
class AnalisadorSemantico:
    def __init__(self):
        self.tabela = TabelaSimbolos()
        self.diagnosticos = []
        self._rastros_atribuicao = []

    def _diagnostico(self, severidade, codigo, no, descricao, dica):
        chave = (severidade, codigo, no.linha, no.coluna, descricao)
        existentes = {
            (d.severidade, d.codigo, d.linha, d.coluna, d.descricao)
            for d in self.diagnosticos
        }
        if chave not in existentes:
            self.diagnosticos.append(
                DiagnosticoSemantico(
                    severidade, codigo, no.linha, no.coluna, descricao, dica
                )
            )

    def erro(self, codigo, no, descricao, dica):
        self._diagnostico("erro", codigo, no, descricao, dica)

    def aviso(self, codigo, no, descricao, dica):
        self._diagnostico("aviso", codigo, no, descricao, dica)

    def analisar(self, programa):
        self.bloco(programa.comandos, novo_escopo=False)
        return ResultadoSemantico(programa, self.tabela, list(self.diagnosticos))

    def bloco(self, comandos, novo_escopo=True, rotulo="bloco"):
        if novo_escopo:
            self.tabela.abrir(rotulo)
        for comando in comandos:
            self.comando(comando)
        if novo_escopo:
            self.tabela.fechar()

    def _estado_externo(self):
        return {
            id(s): (s, s.inicializada, s.valor_conhecido)
            for s in self.tabela.simbolos
        }

    def _restaurar_fluxo(self, estado, alterados):
        for identificador, (simbolo, inicializada, valor) in estado.items():
            simbolo.inicializada = inicializada
            simbolo.valor_conhecido = (
                DESCONHECIDO if identificador in alterados else valor
            )

    def _restaurar_exato(self, estado):
        for simbolo, inicializada, valor in estado.values():
            simbolo.inicializada = inicializada
            simbolo.valor_conhecido = valor

    def _analisar_caminho(self, comandos, rotulo, estado):
        alterados = set()
        self._rastros_atribuicao.append(alterados)
        self.bloco(comandos, novo_escopo=True, rotulo=rotulo)
        self._rastros_atribuicao.pop()
        # O próximo caminho parte do mesmo estado anterior à decisão.
        self._restaurar_exato(estado)
        return alterados

    def comando(self, comando):
        if isinstance(comando, Decl):
            self.declaracao(comando)
        elif isinstance(comando, Atrib):
            self.atribuicao(comando)
        elif isinstance(comando, Mostrar):
            comando.expr = self.expr(comando.expr)
        elif isinstance(comando, Se):
            comando.condicao = self.condicao(comando.condicao, comando, "se")
            estado = self._estado_externo()
            alterados = self._analisar_caminho(comando.entao, "se.entao", estado)
            if comando.senao is not None:
                alterados |= self._analisar_caminho(comando.senao, "se.senao", estado)
            self._restaurar_fluxo(estado, alterados)
        elif isinstance(comando, Enquanto):
            comando.condicao = self.condicao(comando.condicao, comando, "enquanto")
            estado = self._estado_externo()
            alterados = self._analisar_caminho(comando.corpo, "enquanto", estado)
            self._restaurar_fluxo(estado, alterados)
        elif isinstance(comando, Cotar):
            self.cotar(comando)
        elif isinstance(comando, Despachar):
            comando.codigo = self.exigir_tipo(
                self.expr(comando.codigo), "texto", comando,
                "ARG_DESPACHAR", "O código de despacho precisa ser texto."
            )
        elif isinstance(comando, Recusar):
            comando.motivo = self.exigir_tipo(
                self.expr(comando.motivo), "texto", comando,
                "ARG_RECUSAR", "O motivo da recusa precisa ser texto."
            )
        elif isinstance(comando, Rastrear):
            comando.codigo = self.exigir_tipo(
                self.expr(comando.codigo), "texto", comando,
                "ARG_RASTREAR", "O código de rastreamento precisa ser texto."
            )

    def declaracao(self, comando):
        simbolo = self.tabela.declarar(
            comando.nome, comando.tipo, comando.linha, comando.coluna
        )
        if simbolo is None:
            self.erro(
                "DECLARACAO_DUPLICADA", comando,
                f"A variável '{comando.nome}' já foi declarada neste escopo.",
                "Use outro nome ou faça apenas uma atribuição, sem repetir o tipo."
            )
        if comando.expr is None:
            return
        comando.expr = self.expr(comando.expr)
        if simbolo is None:
            return
        comando.expr, compativel = self.compatibilizar(
            comando.expr, comando.tipo, comando,
            f"A declaração de '{comando.nome}'"
        )
        if compativel:
            simbolo.inicializada = True
            simbolo.valor_conhecido = self.valor_constante(comando.expr)

    def atribuicao(self, comando):
        simbolo = self.tabela.buscar(comando.nome)
        comando.expr = self.expr(comando.expr)
        if simbolo is None:
            self.erro(
                "VAR_NAO_DECLARADA", comando,
                f"A variável '{comando.nome}' não foi declarada.",
                f"Declare antes de atribuir, por exemplo: inteiro {comando.nome};"
            )
            return
        comando.expr, compativel = self.compatibilizar(
            comando.expr, simbolo.tipo, comando,
            f"A atribuição a '{comando.nome}'"
        )
        if compativel:
            simbolo.inicializada = True
            simbolo.valor_conhecido = self.valor_constante(comando.expr)
            for rastro in self._rastros_atribuicao:
                rastro.add(id(simbolo))

    def compatibilizar(self, expr, destino, no, contexto):
        if expr.tipo is None:
            return expr, False
        if expr.tipo == destino:
            return expr, True
        if expr.tipo == "inteiro" and destino == "decimal":
            coercao = Coercao(expr.linha, expr.coluna, expr)
            coercao.tipo = "decimal"
            return coercao, True
        dica = (
            "Use decimal no destino ou forneça um valor inteiro."
            if expr.tipo == "decimal" and destino == "inteiro"
            else f"Forneça uma expressão do tipo {destino}."
        )
        self.erro(
            "TIPO_INCOMPATIVEL", no,
            f"{contexto} espera {destino}, mas recebeu {expr.tipo}.", dica
        )
        return expr, False

    def exigir_tipo(self, expr, esperado, no, codigo, descricao):
        if expr.tipo is not None and expr.tipo != esperado:
            self.erro(codigo, no, descricao, f"Forneça uma expressão do tipo {esperado}.")
        return expr

    def condicao(self, expr, comando, nome):
        expr = self.expr(expr)
        if expr.tipo is not None and expr.tipo != "logico":
            self.erro(
                "CONDICAO_NAO_LOGICA", comando,
                f"A condição do comando '{nome}' precisa ser logico, mas recebeu {expr.tipo}.",
                "Use uma comparação ou uma expressão lógica."
            )
        return expr

    def expr(self, expr):
        if isinstance(expr, Lit):
            expr.tipo = {
                bool: "logico", int: "inteiro", float: "decimal", str: "texto"
            }[type(expr.valor)]
            return expr

        if isinstance(expr, Var):
            simbolo = self.tabela.buscar(expr.nome)
            if simbolo is None:
                self.erro(
                    "VAR_NAO_DECLARADA", expr,
                    f"A variável '{expr.nome}' não foi declarada.",
                    "Confira a grafia ou declare a variável antes do uso."
                )
                expr.tipo = None
            else:
                expr.tipo = simbolo.tipo
                if not simbolo.inicializada:
                    self.erro(
                        "VAR_NAO_INICIALIZADA", expr,
                        f"A variável '{expr.nome}' foi usada antes de receber um valor.",
                        f"Atribua um valor a '{expr.nome}' antes deste uso."
                    )
            return expr

        if isinstance(expr, Coercao):
            expr.expr = self.expr(expr.expr)
            expr.tipo = expr.destino
            return expr

        if isinstance(expr, UnOp):
            expr.expr = self.expr(expr.expr)
            tipo = expr.expr.tipo
            if expr.op == "nao":
                if tipo is not None and tipo != "logico":
                    self.erro(
                        "OPERADOR_LOGICO_INVALIDO", expr,
                        f"O operador 'nao' exige logico, mas recebeu {tipo}.",
                        "Use 'nao' somente com valores lógicos."
                    )
                expr.tipo = "logico" if tipo is not None else None
            else:
                if tipo is not None and tipo not in TIPOS_NUMERICOS:
                    self.erro(
                        "MENOS_UNARIO_INVALIDO", expr,
                        f"O sinal negativo exige número, mas recebeu {tipo}.",
                        "Use '-' somente com inteiro ou decimal."
                    )
                    expr.tipo = None
                else:
                    expr.tipo = tipo
            return expr

        if isinstance(expr, BinOp):
            expr.esq = self.expr(expr.esq)
            expr.dir = self.expr(expr.dir)
            esquerda, direita = expr.esq.tipo, expr.dir.tipo
            if esquerda is None or direita is None:
                expr.tipo = None
                return expr

            if expr.op in {"e", "ou"}:
                if esquerda != "logico" or direita != "logico":
                    self.erro(
                        "OPERADOR_LOGICO_INVALIDO", expr,
                        f"O operador '{expr.op}' exige dois valores logicos, "
                        f"mas recebeu {esquerda} e {direita}.",
                        "Use operandos do tipo logico."
                    )
                expr.tipo = "logico"
                return expr

            if expr.op in OPERADORES_ARITMETICOS:
                if esquerda not in TIPOS_NUMERICOS or direita not in TIPOS_NUMERICOS:
                    self.erro(
                        "OPERACAO_NUMERICA_INVALIDA", expr,
                        f"A operação '{expr.op}' exige números, mas recebeu "
                        f"{esquerda} e {direita}.",
                        "Use operandos inteiro ou decimal."
                    )
                    expr.tipo = None
                    return expr
                expr.esq, expr.dir = self.coagir_numericos(expr.esq, expr.dir, expr.op == "/")
                expr.tipo = "decimal" if expr.op == "/" or "decimal" in (esquerda, direita) else "inteiro"
                if expr.op == "/" and self.valor_constante(expr.dir) == 0:
                    self.erro(
                        "DIVISAO_POR_ZERO", expr, "A divisão por zero não é válida.",
                        "Use um divisor diferente de zero."
                    )
                return expr

            if expr.op in OPERADORES_ORDEM:
                if esquerda not in TIPOS_NUMERICOS or direita not in TIPOS_NUMERICOS:
                    self.erro(
                        "COMPARACAO_INVALIDA", expr,
                        f"A comparação '{expr.op}' exige números, mas recebeu "
                        f"{esquerda} e {direita}.",
                        "Compare valores inteiro ou decimal."
                    )
                else:
                    expr.esq, expr.dir = self.coagir_numericos(expr.esq, expr.dir)
                expr.tipo = "logico"
                return expr

            if expr.op in OPERADORES_IGUALDADE:
                compativeis = esquerda == direita or (
                    esquerda in TIPOS_NUMERICOS and direita in TIPOS_NUMERICOS
                )
                if not compativeis:
                    self.erro(
                        "COMPARACAO_INVALIDA", expr,
                        f"A comparação '{expr.op}' recebeu tipos incompatíveis: "
                        f"{esquerda} e {direita}.",
                        "Compare valores do mesmo tipo ou dois valores numéricos."
                    )
                elif esquerda in TIPOS_NUMERICOS and direita in TIPOS_NUMERICOS:
                    expr.esq, expr.dir = self.coagir_numericos(expr.esq, expr.dir)
                expr.tipo = "logico"
                return expr
        return expr

    def coagir_numericos(self, esquerda, direita, forcar_decimal=False):
        usar_decimal = forcar_decimal or "decimal" in (esquerda.tipo, direita.tipo)
        if usar_decimal and esquerda.tipo == "inteiro":
            nova = Coercao(esquerda.linha, esquerda.coluna, esquerda)
            nova.tipo = "decimal"
            esquerda = nova
        if usar_decimal and direita.tipo == "inteiro":
            nova = Coercao(direita.linha, direita.coluna, direita)
            nova.tipo = "decimal"
            direita = nova
        return esquerda, direita

    def valor_constante(self, expr):
        if isinstance(expr, Lit):
            return expr.valor
        if isinstance(expr, Var):
            simbolo = self.tabela.buscar(expr.nome)
            if simbolo is not None and simbolo.inicializada:
                return simbolo.valor_conhecido
            return DESCONHECIDO
        if isinstance(expr, Coercao):
            valor = self.valor_constante(expr.expr)
            return DESCONHECIDO if valor is DESCONHECIDO else float(valor)
        if isinstance(expr, UnOp):
            valor = self.valor_constante(expr.expr)
            if valor is DESCONHECIDO:
                return DESCONHECIDO
            if expr.op == "-":
                return -valor
            if expr.op == "nao":
                return not valor
        if isinstance(expr, BinOp):
            a = self.valor_constante(expr.esq)
            b = self.valor_constante(expr.dir)
            if a is DESCONHECIDO or b is DESCONHECIDO:
                return DESCONHECIDO
            if expr.op == "+": return a + b
            if expr.op == "-": return a - b
            if expr.op == "*": return a * b
            if expr.op == "/": return DESCONHECIDO if b == 0 else a / b
            if expr.op == ">": return a > b
            if expr.op == "<": return a < b
            if expr.op == ">=": return a >= b
            if expr.op == "<=": return a <= b
            if expr.op == "==": return a == b
            if expr.op == "!=": return a != b
            if expr.op == "e": return a and b
            if expr.op == "ou": return a or b
        return DESCONHECIDO

## Regras de negócio e comandos logísticos

`cotar` exige texto, dois números e um prazo inteiro. Quando peso, valor ou prazo são conhecidos com segurança, aplicam-se as regras da ficha. Os demais comandos exigem texto e não consultam qualquer estado de entrega nesta etapa.

In [24]:
def _cotar(self, comando):
    comando.modalidade = self.exigir_tipo(
        self.expr(comando.modalidade), "texto", comando,
        "ARG_COTAR_MODALIDADE", "A modalidade da cotação precisa ser texto."
    )
    comando.peso = self.expr(comando.peso)
    if comando.peso.tipo is not None and comando.peso.tipo not in TIPOS_NUMERICOS:
        self.erro(
            "ARG_COTAR_PESO", comando, "O peso da cotação precisa ser numérico.",
            "Informe o peso como inteiro ou decimal."
        )
    comando.valor = self.expr(comando.valor)
    if comando.valor.tipo is not None and comando.valor.tipo not in TIPOS_NUMERICOS:
        self.erro(
            "ARG_COTAR_VALOR", comando, "O valor da cotação precisa ser numérico.",
            "Informe o valor como inteiro ou decimal."
        )
    comando.prazo = self.exigir_tipo(
        self.expr(comando.prazo), "inteiro", comando,
        "ARG_COTAR_PRAZO", "O prazo da cotação precisa ser inteiro."
    )

    peso = self.valor_constante(comando.peso)
    valor = self.valor_constante(comando.valor)
    prazo = self.valor_constante(comando.prazo)
    if comando.peso.tipo in TIPOS_NUMERICOS and peso is not DESCONHECIDO and peso <= 0:
        self.erro(
            "PESO_INVALIDO", comando,
            "O peso da cotação precisa ser maior que zero.",
            "Informe um peso positivo em quilogramas."
        )
    if comando.prazo.tipo == "inteiro" and prazo is not DESCONHECIDO and prazo < 1:
        self.erro(
            "PRAZO_INVALIDO", comando,
            "O prazo da cotação precisa ser de pelo menos um dia.",
            "Informe um prazo inteiro maior ou igual a 1."
        )
    if comando.valor.tipo in TIPOS_NUMERICOS and valor is not DESCONHECIDO and valor > 1000:
        self.aviso(
            "FRETE_ELEVADO", comando,
            f"O valor do frete é superior a R$ 1.000 ({valor:.2f}).",
            "Revise o cálculo antes de confirmar a cotação."
        )


AnalisadorSemantico.cotar = _cotar
print("Regras logísticas carregadas: peso > 0, prazo >= 1 e aviso para valor > 1000.")

Regras logísticas carregadas: peso > 0, prazo >= 1 e aviso para valor > 1000.


## Função principal

Cada chamada cria uma nova AST, uma nova tabela e um novo analisador. Não há estado global compartilhado entre programas.

In [25]:
def analisar_semantica(codigo):
    ast = analisar_sintaxe(codigo)
    analisador = AnalisadorSemantico()
    return analisador.analisar(ast)


RESULTADOS_EXEMPLOS = [
    analisar_semantica(programa) for programa in MINHA_FICHA["programas_exemplo"]
]
for numero, resultado in enumerate(RESULTADOS_EXEMPLOS, 1):
    print(
        f"Programa {numero}: "
        f"{'aprovado' if resultado.aprovado else 'reprovado'} — "
        f"{len(resultado.erros)} erro(s), {len(resultado.avisos)} aviso(s)."
    )

Programa 1: aprovado — 0 erro(s), 0 aviso(s).
Programa 2: aprovado — 0 erro(s), 0 aviso(s).
Programa 3: aprovado — 0 erro(s), 0 aviso(s).


## Exemplo da tabela de símbolos

O exemplo mostra símbolos globais e internos. A variável global `peso` é consultada dentro do bloco, enquanto a segunda variável `mensagem` existe somente no escopo do `se`.

In [26]:
EXEMPLO_ESCOPOS = r'''
decimal peso = 5;
se peso > 0.0 {
    texto mensagem = "Peso valido";
    mostrar mensagem;
}
mostrar peso;
'''
RESULTADO_ESCOPOS = analisar_semantica(EXEMPLO_ESCOPOS)
mostrar_tabela_simbolos(RESULTADO_ESCOPOS.tabela)
print("\nAprovação:", RESULTADO_ESCOPOS.aprovado)

print("\nCoerção registrada na AST:")
mostrar_ast(RESULTADO_ESCOPOS.ast.comandos[0])

Nome           | Tipo     | Escopo           | Declaração | Inicializada | Valor
-------------------------------------------------------------------------------------------
peso           | decimal  | global           | 2:1        | sim         | 5.0
mensagem       | texto    | se.entao#1       | 4:5        | sim         | 'Peso valido'

Aprovação: True

Coerção registrada na AST:
└── Decl decimal peso [2:1]
    └── valor: Coercao para decimal [2:16]
        └── expr: Lit 5 [2:16] : inteiro


## Galeria de erros semânticos

Todos os casos abaixo são aceitos pelo léxico e pelo parser antes da análise semântica. A galeria cobre dez erros distintos e mostra diagnósticos produzidos a partir das posições reais da AST.

In [27]:
GALERIA_SEMANTICA = [
    ("Variável não declarada", "mostrar distancia;", "VAR_NAO_DECLARADA"),
    ("Declaração duplicada", "inteiro x = 1; inteiro x = 2;", "DECLARACAO_DUPLICADA"),
    ("Tipo incompatível", 'inteiro quantidade = "cinco";', "TIPO_INCOMPATIVEL"),
    ("Condição não lógica", 'se 10 { mostrar "Teste"; }', "CONDICAO_NAO_LOGICA"),
    ("Operação matemática inválida", 'decimal r = "Frete" * 3;', "OPERACAO_NUMERICA_INVALIDA"),
    ("Peso inválido", 'cotar "Expresso", 0.0, 45.0, 2;', "PESO_INVALIDO"),
    ("Prazo inválido", 'cotar "Economico", 2.0, 45.0, 0;', "PRAZO_INVALIDO"),
    ("Variável não inicializada", "inteiro distancia; mostrar distancia;", "VAR_NAO_INICIALIZADA"),
    ("Variável fora do escopo", 'se verdadeiro { inteiro local = 1; } mostrar local;', "VAR_NAO_DECLARADA"),
    ("Argumento logístico incorreto", "despachar 123;", "ARG_DESPACHAR"),
]

for titulo, fonte, codigo_esperado in GALERIA_SEMANTICA:
    tokens = analisar_lexico(fonte)
    ast = analisar_sintaxe(fonte)
    resultado = analisar_semantica(fonte)
    diagnostico = next(d for d in resultado.erros if d.codigo == codigo_esperado)
    print(f"\n--- {titulo} ---")
    print(f"R2: aceito ({len(tokens)} tokens) | R3: aceito ({len(ast.comandos)} comando(s))")
    print(diagnostico)


--- Variável não declarada ---
R2: aceito (3 tokens) | R3: aceito (1 comando(s))
Erro semântico — linha 1, coluna 9
Descrição: A variável 'distancia' não foi declarada.
Dica: Confira a grafia ou declare a variável antes do uso.

--- Declaração duplicada ---
R2: aceito (10 tokens) | R3: aceito (2 comando(s))
Erro semântico — linha 1, coluna 16
Descrição: A variável 'x' já foi declarada neste escopo.
Dica: Use outro nome ou faça apenas uma atribuição, sem repetir o tipo.

--- Tipo incompatível ---
R2: aceito (5 tokens) | R3: aceito (1 comando(s))
Erro semântico — linha 1, coluna 1
Descrição: A declaração de 'quantidade' espera inteiro, mas recebeu texto.
Dica: Forneça uma expressão do tipo inteiro.

--- Condição não lógica ---
R2: aceito (7 tokens) | R3: aceito (1 comando(s))
Erro semântico — linha 1, coluna 1
Descrição: A condição do comando 'se' precisa ser logico, mas recebeu inteiro.
Dica: Use uma comparação ou uma expressão lógica.

--- Operação matemática inválida ---
R2: aceito (

## Aviso semântico

Valor superior a R$ 1.000 gera aviso, sem reprovar. O limite é estrito: exatamente R$ 1.000 não gera aviso.

In [28]:
RESULTADO_AVISO = analisar_semantica(
    'cotar "Expresso", 10.0, 1250.0, 2;'
)
print("Aprovado:", RESULTADO_AVISO.aprovado)
for aviso in RESULTADO_AVISO.avisos:
    print(aviso)

RESULTADO_LIMITE = analisar_semantica(
    'cotar "Expresso", 10.0, 1000.0, 2;'
)
print("\nValor igual a 1000 — avisos:", len(RESULTADO_LIMITE.avisos))

Aprovado: True
Aviso semântico — linha 1, coluna 1
Descrição: O valor do frete é superior a R$ 1.000 (1250.00).
Dica: Revise o cálculo antes de confirmar a cotação.

Valor igual a 1000 — avisos: 0


## Testes automatizados do R4

Os testes verificam resultados, códigos, severidades, posições, tipos inferidos, coerções, escopos e valores constantes. As baterias R2 e R3 já foram executadas anteriormente no mesmo kernel e são conferidas novamente no resumo.

In [29]:
def tem_diagnostico(resultado, codigo, severidade="erro"):
    return any(
        d.codigo == codigo and d.severidade == severidade
        and d.linha > 0 and d.coluna > 0 and bool(d.dica)
        for d in resultado.diagnosticos
    )


def resultado_de(fonte):
    return analisar_semantica(fonte)


def teste_quatro_tipos():
    r = resultado_de('inteiro a = 1; decimal b = 2.0; texto c = "x"; logico d = verdadeiro;')
    return r.aprovado and [s.tipo for s in r.tabela.simbolos] == ["inteiro", "decimal", "texto", "logico"]


def teste_coercao():
    r = resultado_de("decimal peso = 5;")
    return r.aprovado and isinstance(r.ast.comandos[0].expr, Coercao) and r.ast.comandos[0].expr.tipo == "decimal"


def teste_valor_variavel_negativo():
    r = resultado_de('decimal peso = -2.0; inteiro prazo = 3; cotar "Expresso", peso, 50.0, prazo;')
    return tem_diagnostico(r, "PESO_INVALIDO")


def teste_valor_desconhecido():
    fonte = 'decimal peso = 2.0; se verdadeiro { peso = -1.0; } cotar "Expresso", peso, 50.0, 2;'
    r = resultado_de(fonte)
    simbolo = r.tabela.simbolos[0]
    return r.aprovado and simbolo.inicializada and simbolo.valor_conhecido is DESCONHECIDO


def teste_isolamento():
    primeiro = resultado_de("inteiro temporaria = 1;")
    segundo = resultado_de("mostrar temporaria;")
    return primeiro.aprovado and tem_diagnostico(segundo, "VAR_NAO_DECLARADA")


CASOS_TESTE_SEMANTICO = [
    ("Programa original 1", lambda: RESULTADOS_EXEMPLOS[0].aprovado),
    ("Programa original 2", lambda: RESULTADOS_EXEMPLOS[1].aprovado),
    ("Programa original 3", lambda: RESULTADOS_EXEMPLOS[2].aprovado),
    ("Quatro tipos de dados", teste_quatro_tipos),
    ("Declaração e atribuição válidas", lambda: resultado_de("inteiro x; x = 2; mostrar x;").aprovado),
    ("Coerção inteiro para decimal", teste_coercao),
    ("Rejeição decimal para inteiro", lambda: tem_diagnostico(resultado_de("inteiro x = 2.5;"), "TIPO_INCOMPATIVEL")),
    ("Operadores numéricos", lambda: resultado_de("decimal x = 2 + 3.0 * 4 / 2;").aprovado),
    ("Divisão resulta decimal", lambda: resultado_de("decimal x = 4 / 2;").ast.comandos[0].expr.tipo == "decimal"),
    ("Operadores lógicos", lambda: resultado_de("logico x = verdadeiro e nao falso ou falso;").aprovado),
    ("Comparações numéricas mistas", lambda: resultado_de("logico x = 2 < 3.5;").aprovado),
    ("Lógico não é inteiro", lambda: tem_diagnostico(resultado_de("inteiro x = verdadeiro;"), "TIPO_INCOMPATIVEL")),
    ("Identificador inexistente", lambda: tem_diagnostico(resultado_de("mostrar ausente;"), "VAR_NAO_DECLARADA")),
    ("Declaração duplicada", lambda: tem_diagnostico(resultado_de("inteiro x; inteiro x;"), "DECLARACAO_DUPLICADA")),
    ("Uso antes da inicialização", lambda: tem_diagnostico(resultado_de("inteiro x; mostrar x;"), "VAR_NAO_INICIALIZADA")),
    ("Escopo interno consulta externo", lambda: resultado_de("inteiro x = 1; se verdadeiro { mostrar x; }").aprovado),
    ("Variável interna não escapa", lambda: tem_diagnostico(resultado_de("se verdadeiro { inteiro x = 1; } mostrar x;"), "VAR_NAO_DECLARADA")),
    ("Sombreamento interno permitido", lambda: resultado_de("inteiro x = 1; se verdadeiro { texto x = \"interno\"; mostrar x; } mostrar x;").aprovado),
    ("Atribuição externa não cria local", lambda: len(resultado_de("inteiro x = 1; se verdadeiro { x = 2; }").tabela.simbolos) == 1),
    ("Inicialização condicional conservadora", lambda: tem_diagnostico(resultado_de("inteiro x; se verdadeiro { x = 1; } mostrar x;"), "VAR_NAO_INICIALIZADA")),
    ("Condição se deve ser lógica", lambda: tem_diagnostico(resultado_de("se 10 { mostrar 1; }"), "CONDICAO_NAO_LOGICA")),
    ("Condição enquanto deve ser lógica", lambda: tem_diagnostico(resultado_de("enquanto 1 { mostrar 1; }"), "CONDICAO_NAO_LOGICA")),
    ("Tipos válidos de cotar", lambda: resultado_de('cotar "Expresso", 2, 45.0, 1;').aprovado),
    ("Tipos inválidos de cotar", lambda: {"ARG_COTAR_MODALIDADE", "ARG_COTAR_PESO", "ARG_COTAR_VALOR", "ARG_COTAR_PRAZO"}.issubset(
        {d.codigo for d in resultado_de('cotar 1, "peso", falso, 1.5;').erros})),
    ("Tipos dos outros comandos", lambda: resultado_de('texto c = "FS"; despachar c; rastrear c; recusar "motivo";').aprovado),
    ("Tipos inválidos dos comandos", lambda: all(tem_diagnostico(resultado_de(f), c) for f, c in [("despachar 1;", "ARG_DESPACHAR"), ("recusar falso;", "ARG_RECUSAR"), ("rastrear 1.0;", "ARG_RASTREAR")])),
    ("Peso zero inválido", lambda: tem_diagnostico(resultado_de('cotar "E", 0.0, 45.0, 2;'), "PESO_INVALIDO")),
    ("Peso negativo por variável", teste_valor_variavel_negativo),
    ("Prazo zero inválido", lambda: tem_diagnostico(resultado_de('cotar "E", 2.0, 45.0, 0;'), "PRAZO_INVALIDO")),
    ("Frete acima de 1000 avisa", lambda: tem_diagnostico(resultado_de('cotar "E", 2.0, 1000.01, 1;'), "FRETE_ELEVADO", "aviso")),
    ("Frete igual a 1000 não avisa", lambda: not resultado_de('cotar "E", 2.0, 1000.0, 1;').avisos),
    ("Constante aritmética por variáveis", lambda: tem_diagnostico(resultado_de('decimal base = 600.0; decimal valor = base * 2; cotar "E", 2.0, valor, 1;'), "FRETE_ELEVADO", "aviso")),
    ("Valor incerto fica para execução", teste_valor_desconhecido),
    ("Ramificações partem do mesmo estado", lambda: tem_diagnostico(
        resultado_de('decimal peso = -2.0; se verdadeiro { peso = 2.0; } senao { cotar "E", peso, 50.0, 2; }'),
        "PESO_INVALIDO")),
    ("Análises consecutivas isoladas", teste_isolamento),
]

TESTES_R4_APROVADOS = 0
for nome, teste in CASOS_TESTE_SEMANTICO:
    try:
        passou = bool(teste())
    except Exception as erro:
        passou = False
        print(f"FALHOU — {nome}: {type(erro).__name__}: {erro}")
    else:
        print(f"{'OK' if passou else 'FALHOU'} — {nome}")
    TESTES_R4_APROVADOS += int(passou)

TESTES_R4_EXECUTADOS = len(CASOS_TESTE_SEMANTICO)
TESTES_R4_REPROVADOS = TESTES_R4_EXECUTADOS - TESTES_R4_APROVADOS
print(
    f"\nR4: {TESTES_R4_EXECUTADOS} executados, "
    f"{TESTES_R4_APROVADOS} aprovados e {TESTES_R4_REPROVADOS} reprovados."
)
assert TESTES_R4_REPROVADOS == 0, "Há testes semânticos reprovados."

OK — Programa original 1
OK — Programa original 2
OK — Programa original 3
OK — Quatro tipos de dados
OK — Declaração e atribuição válidas
OK — Coerção inteiro para decimal
OK — Rejeição decimal para inteiro
OK — Operadores numéricos
OK — Divisão resulta decimal
OK — Operadores lógicos
OK — Comparações numéricas mistas
OK — Lógico não é inteiro
OK — Identificador inexistente
OK — Declaração duplicada
OK — Uso antes da inicialização
OK — Escopo interno consulta externo
OK — Variável interna não escapa
OK — Sombreamento interno permitido
OK — Atribuição externa não cria local
OK — Inicialização condicional conservadora
OK — Condição se deve ser lógica
OK — Condição enquanto deve ser lógica
OK — Tipos válidos de cotar
OK — Tipos inválidos de cotar
OK — Tipos dos outros comandos
OK — Tipos inválidos dos comandos
OK — Peso zero inválido
OK — Peso negativo por variável
OK — Prazo zero inválido
OK — Frete acima de 1000 avisa
OK — Frete igual a 1000 não avisa
OK — Constante aritmética por vari

## Resumo da Etapa 04

Os testes desta seção verificam somente o R4 e complementam a bateria de integração apresentada adiante. O resumo também confirma que as baterias anteriores permaneceram aprovadas durante a mesma execução do notebook.

In [30]:
print(
    f"R4: {TESTES_R4_EXECUTADOS}/{TESTES_R4_EXECUTADOS} executados — "
    f"{TESTES_R4_APROVADOS} aprovados e {TESTES_R4_REPROVADOS} reprovados."
)
print(
    f"Regressão R3: {TESTES_R3_EXECUTADOS} executados — "
    f"{TESTES_R3_APROVADOS} aprovados e {TESTES_R3_REPROVADOS} reprovados."
)
print(
    f"Regressão R2: {TESTES_EXECUTADOS} executados — "
    f"{TESTES_APROVADOS} aprovados e {TESTES_REPROVADOS} reprovados."
)
assert TESTES_R3_REPROVADOS == 0
assert TESTES_REPROVADOS == 0

R4: 35/35 executados — 35 aprovados e 0 reprovados.
Regressão R3: 25 executados — 25 aprovados e 0 reprovados.
Regressão R2: 17 executados — 17 aprovados e 0 reprovados.


# Etapa 05 — Código de três endereços (TAC)

O TAC transforma a AST semanticamente aprovada em instruções pequenas, com no máximo uma operação principal por linha. Essa representação é a entrada do otimizador e da máquina virtual apresentados adiante.

## Estrutura das instruções

Literais, variáveis, temporários e rótulos possuem classes distintas. Assim, o texto literal `"_t1"` nunca é confundido com o temporário `_t1`, e duas variáveis homônimas podem possuir identidades internas diferentes.

In [31]:
@dataclass(frozen=True)
class LiteralTAC:
    valor: object
    tipo: str


@dataclass(frozen=True)
class VariavelTAC:
    identificador: int
    nome: str
    tipo: str
    escopo: str


@dataclass(frozen=True)
class TemporarioTAC:
    identificador: int
    tipo: str


@dataclass(frozen=True)
class RotuloTAC:
    identificador: int


@dataclass(frozen=True)
class InstrucaoTAC:
    operacao: str
    destino: object | None
    argumentos: tuple
    linha: int
    coluna: int


@dataclass
class ResultadoTAC:
    semantica: ResultadoSemantico
    instrucoes: list
    variaveis: list
    problemas_estruturais: list = field(default_factory=list)

    @property
    def avisos(self):
        return self.semantica.avisos


class ErroGeracaoTAC(Exception):
    def __init__(self, diagnosticos):
        self.diagnosticos = list(diagnosticos)
        super().__init__("A geração TAC foi impedida por erros semânticos.")


TIPOS_OPERANDO_TAC = (LiteralTAC, VariavelTAC, TemporarioTAC)
OPERACOES_TAC = {
    "DECLARAR", "ATRIBUIR", "BINARIO", "UNARIO", "CONVERTER_DECIMAL",
    "MOSTRAR", "COTAR", "DESPACHAR", "RECUSAR", "RASTREAR",
    "ROTULO", "SE_FALSO", "IR_PARA",
}

## Temporários, variáveis e rótulos

Cada execução do gerador começa os contadores novamente. As variáveis recebem identificadores na ordem das declarações e ficam em uma pilha de escopos própria do gerador. Declarações sem valor produzem somente `DECLARAR`, permitindo que a VM diferencie uma variável existente de uma variável inicializada.

In [32]:
class GeradorTAC:
    def __init__(self, resultado_semantico):
        self.semantica = resultado_semantico
        self.instrucoes = []
        self.variaveis = []
        self._escopos = [("global", {})]
        self._n_variavel = 0
        self._n_temporario = 0
        self._n_rotulo = 0
        self._n_escopo = 0

    def emitir(self, operacao, destino=None, argumentos=(), no=None):
        linha = no.linha if no is not None else 1
        coluna = no.coluna if no is not None else 1
        instrucao = InstrucaoTAC(
            operacao, destino, tuple(argumentos), linha, coluna
        )
        self.instrucoes.append(instrucao)
        return instrucao

    def novo_temporario(self, tipo):
        self._n_temporario += 1
        return TemporarioTAC(self._n_temporario, tipo)

    def novo_rotulo(self):
        self._n_rotulo += 1
        return RotuloTAC(self._n_rotulo)

    def abrir_escopo(self, rotulo):
        self._n_escopo += 1
        self._escopos.append((f"{rotulo}#{self._n_escopo}", {}))

    def fechar_escopo(self):
        self._escopos.pop()

    def declarar_variavel(self, nome, tipo):
        self._n_variavel += 1
        variavel = VariavelTAC(
            self._n_variavel, nome, tipo, self._escopos[-1][0]
        )
        self._escopos[-1][1][nome] = variavel
        self.variaveis.append(variavel)
        return variavel

    def buscar_variavel(self, nome):
        for _, escopo in reversed(self._escopos):
            if nome in escopo:
                return escopo[nome]
        raise RuntimeError(f"Vínculo semântico ausente para a variável {nome!r}.")

    def gerar(self):
        for comando in self.semantica.ast.comandos:
            self.comando(comando)
        return ResultadoTAC(
            self.semantica, list(self.instrucoes), list(self.variaveis)
        )

## Geração de expressões

Os filhos são gerados antes do operador pai, preservando precedência e associatividade da AST. Coerções viram `CONVERTER_DECIMAL`. Para `e` e `ou`, rótulos e `SE_FALSO` evitam avaliar o segundo operando quando o primeiro já determina o resultado.

In [33]:
def _expr_tac(self, expr):
    if isinstance(expr, Lit):
        return LiteralTAC(expr.valor, expr.tipo)
    if isinstance(expr, Var):
        return self.buscar_variavel(expr.nome)
    if isinstance(expr, Coercao):
        origem = self.expr(expr.expr)
        destino = self.novo_temporario("decimal")
        self.emitir("CONVERTER_DECIMAL", destino, (origem,), expr)
        return destino
    if isinstance(expr, UnOp):
        origem = self.expr(expr.expr)
        destino = self.novo_temporario(expr.tipo)
        self.emitir("UNARIO", destino, (expr.op, origem), expr)
        return destino
    if isinstance(expr, BinOp) and expr.op in {"e", "ou"}:
        return self.expr_logica(expr)
    if isinstance(expr, BinOp):
        esquerda = self.expr(expr.esq)
        direita = self.expr(expr.dir)
        destino = self.novo_temporario(expr.tipo)
        self.emitir(
            "BINARIO", destino, (esquerda, expr.op, direita), expr
        )
        return destino
    raise TypeError(f"Expressão sem geração TAC: {type(expr).__name__}")


def _expr_logica_tac(self, expr):
    resultado = self.novo_temporario("logico")
    esquerda = self.expr(expr.esq)

    if expr.op == "e":
        resultado_curto = self.novo_rotulo()
        fim = self.novo_rotulo()
        self.emitir("SE_FALSO", None, (esquerda, resultado_curto), expr)
        direita = self.expr(expr.dir)
        self.emitir("ATRIBUIR", resultado, (direita,), expr)
        self.emitir("IR_PARA", None, (fim,), expr)
        self.emitir("ROTULO", resultado_curto, (), expr)
        self.emitir("ATRIBUIR", resultado, (LiteralTAC(False, "logico"),), expr)
    else:
        avaliar_direita = self.novo_rotulo()
        fim = self.novo_rotulo()
        self.emitir("SE_FALSO", None, (esquerda, avaliar_direita), expr)
        self.emitir("ATRIBUIR", resultado, (LiteralTAC(True, "logico"),), expr)
        self.emitir("IR_PARA", None, (fim,), expr)
        self.emitir("ROTULO", avaliar_direita, (), expr)
        direita = self.expr(expr.dir)
        self.emitir("ATRIBUIR", resultado, (direita,), expr)

    self.emitir("ROTULO", fim, (), expr)
    return resultado


GeradorTAC.expr = _expr_tac
GeradorTAC.expr_logica = _expr_logica_tac

## Comandos e controle de fluxo

O `se` usa `SE_FALSO` para alcançar o bloco alternativo ou o fim. O `enquanto` coloca a geração da condição depois do rótulo inicial, garantindo que ela seja recalculada a cada iteração. Declarações dentro do laço permanecem dentro desse intervalo de controle.

In [34]:
def _bloco_tac(self, comandos, rotulo):
    self.abrir_escopo(rotulo)
    for comando in comandos:
        self.comando(comando)
    self.fechar_escopo()


def _comando_tac(self, comando):
    if isinstance(comando, Decl):
        variavel = self.declarar_variavel(comando.nome, comando.tipo)
        self.emitir("DECLARAR", variavel, (), comando)
        if comando.expr is not None:
            valor = self.expr(comando.expr)
            self.emitir("ATRIBUIR", variavel, (valor,), comando)
        return

    if isinstance(comando, Atrib):
        variavel = self.buscar_variavel(comando.nome)
        valor = self.expr(comando.expr)
        self.emitir("ATRIBUIR", variavel, (valor,), comando)
        return

    if isinstance(comando, Mostrar):
        self.emitir("MOSTRAR", None, (self.expr(comando.expr),), comando)
        return

    if isinstance(comando, Cotar):
        argumentos = (
            self.expr(comando.modalidade), self.expr(comando.peso),
            self.expr(comando.valor), self.expr(comando.prazo),
        )
        self.emitir("COTAR", None, argumentos, comando)
        return

    if isinstance(comando, Despachar):
        self.emitir("DESPACHAR", None, (self.expr(comando.codigo),), comando)
        return

    if isinstance(comando, Recusar):
        self.emitir("RECUSAR", None, (self.expr(comando.motivo),), comando)
        return

    if isinstance(comando, Rastrear):
        self.emitir("RASTREAR", None, (self.expr(comando.codigo),), comando)
        return

    if isinstance(comando, Se):
        if comando.senao is None:
            fim = self.novo_rotulo()
            condicao = self.expr(comando.condicao)
            self.emitir("SE_FALSO", None, (condicao, fim), comando)
            self.bloco(comando.entao, "se.entao")
            self.emitir("ROTULO", fim, (), comando)
        else:
            senao = self.novo_rotulo()
            fim = self.novo_rotulo()
            condicao = self.expr(comando.condicao)
            self.emitir("SE_FALSO", None, (condicao, senao), comando)
            self.bloco(comando.entao, "se.entao")
            self.emitir("IR_PARA", None, (fim,), comando)
            self.emitir("ROTULO", senao, (), comando)
            self.bloco(comando.senao, "se.senao")
            self.emitir("ROTULO", fim, (), comando)
        return

    if isinstance(comando, Enquanto):
        inicio = self.novo_rotulo()
        fim = self.novo_rotulo()
        self.emitir("ROTULO", inicio, (), comando)
        condicao = self.expr(comando.condicao)
        self.emitir("SE_FALSO", None, (condicao, fim), comando)
        self.bloco(comando.corpo, "enquanto")
        self.emitir("IR_PARA", None, (inicio,), comando)
        self.emitir("ROTULO", fim, (), comando)
        return

    raise TypeError(f"Comando sem geração TAC: {type(comando).__name__}")


GeradorTAC.bloco = _bloco_tac
GeradorTAC.comando = _comando_tac

## Validação estrutural

A validação confere operações, posições, contratos, identidades, temporários, rótulos e destinos de salto. Ela valida a estrutura produzida, mas não executa as instruções.

In [35]:
ARIDADE_TAC = {
    "DECLARAR": 0, "ATRIBUIR": 1, "BINARIO": 3, "UNARIO": 2,
    "CONVERTER_DECIMAL": 1, "MOSTRAR": 1, "COTAR": 4,
    "DESPACHAR": 1, "RECUSAR": 1, "RASTREAR": 1,
    "ROTULO": 0, "SE_FALSO": 2, "IR_PARA": 1,
}


def validar_tac(resultado):
    problemas = []
    instrucoes = resultado.instrucoes
    rotulos = [i.destino for i in instrucoes if i.operacao == "ROTULO"]
    conjunto_rotulos = set(rotulos)
    if len(rotulos) != len(conjunto_rotulos):
        problemas.append("Existem rótulos duplicados.")

    variaveis = set(resultado.variaveis)
    declaradas = {
        i.destino for i in instrucoes if i.operacao == "DECLARAR"
    }
    if declaradas != variaveis:
        problemas.append("As identidades declaradas não correspondem às variáveis do resultado.")

    temporarios_definidos = {
        i.destino for i in instrucoes
        if isinstance(i.destino, TemporarioTAC)
    }
    temporarios_usados = set()

    for numero, instrucao in enumerate(instrucoes, 1):
        if instrucao.operacao not in OPERACOES_TAC:
            problemas.append(f"Instrução {numero}: operação desconhecida {instrucao.operacao}.")
        if len(instrucao.argumentos) != ARIDADE_TAC.get(instrucao.operacao, -1):
            problemas.append(f"Instrução {numero}: aridade incorreta.")
        if instrucao.linha < 1 or instrucao.coluna < 1:
            problemas.append(f"Instrução {numero}: posição de origem inválida.")

        if instrucao.operacao == "DECLARAR" and not isinstance(instrucao.destino, VariavelTAC):
            problemas.append(f"Instrução {numero}: DECLARAR exige variável.")
        if instrucao.operacao in {"BINARIO", "UNARIO", "CONVERTER_DECIMAL"} and not isinstance(instrucao.destino, TemporarioTAC):
            problemas.append(f"Instrução {numero}: operação pura exige temporário de destino.")
        if instrucao.operacao == "ROTULO" and not isinstance(instrucao.destino, RotuloTAC):
            problemas.append(f"Instrução {numero}: ROTULO exige um rótulo.")
        if instrucao.operacao == "ATRIBUIR" and not isinstance(instrucao.destino, (VariavelTAC, TemporarioTAC)):
            problemas.append(f"Instrução {numero}: destino de ATRIBUIR inválido.")

        for argumento in instrucao.argumentos:
            if isinstance(argumento, VariavelTAC) and argumento not in variaveis:
                problemas.append(f"Instrução {numero}: variável sem identidade declarada.")
            if isinstance(argumento, TemporarioTAC):
                temporarios_usados.add(argumento)

        if instrucao.operacao == "SE_FALSO":
            condicao, alvo = instrucao.argumentos
            if not isinstance(condicao, TIPOS_OPERANDO_TAC):
                problemas.append(f"Instrução {numero}: condição inválida.")
            if alvo not in conjunto_rotulos:
                problemas.append(f"Instrução {numero}: salto para rótulo inexistente.")
        if instrucao.operacao == "IR_PARA":
            alvo = instrucao.argumentos[0]
            if alvo not in conjunto_rotulos:
                problemas.append(f"Instrução {numero}: salto para rótulo inexistente.")

    if not temporarios_usados.issubset(temporarios_definidos):
        problemas.append("Há temporário usado sem definição no TAC.")
    return problemas

## Função principal `gerar_tac`

O fluxo sempre passa por R2, R3 e R4. Erros semânticos impedem qualquer geração; avisos são preservados no resultado e não bloqueiam o TAC.

In [36]:
def gerar_tac(codigo):
    semantica = analisar_semantica(codigo)
    if not semantica.aprovado:
        raise ErroGeracaoTAC(semantica.erros)
    resultado = GeradorTAC(semantica).gerar()
    resultado.problemas_estruturais = validar_tac(resultado)
    if resultado.problemas_estruturais:
        raise RuntimeError(
            "TAC estruturalmente inválido: "
            + "; ".join(resultado.problemas_estruturais)
        )
    return resultado

## Exibição do TAC

A apresentação abaixo é apenas uma formatação das dataclasses. O resultado estruturado permanece disponível para otimização e execução nas seções seguintes.

In [37]:
def formatar_operando(operando):
    if isinstance(operando, LiteralTAC):
        if operando.tipo == "texto":
            return repr(operando.valor)
        if operando.tipo == "logico":
            return "verdadeiro" if operando.valor else "falso"
        return repr(operando.valor)
    if isinstance(operando, VariavelTAC):
        return f"{operando.nome}@v{operando.identificador}"
    if isinstance(operando, TemporarioTAC):
        return f"_t{operando.identificador}"
    if isinstance(operando, RotuloTAC):
        return f"L{operando.identificador}"
    return str(operando)


def formatar_instrucao(instrucao):
    op, d, a = instrucao.operacao, instrucao.destino, instrucao.argumentos
    if op == "ROTULO": return f"{formatar_operando(d)}:"
    if op == "DECLARAR": return f"    declarar {d.tipo} {formatar_operando(d)}"
    if op == "ATRIBUIR": return f"    {formatar_operando(d)} = {formatar_operando(a[0])}"
    if op == "BINARIO": return f"    {formatar_operando(d)} = {formatar_operando(a[0])} {a[1]} {formatar_operando(a[2])}"
    if op == "UNARIO": return f"    {formatar_operando(d)} = {a[0]} {formatar_operando(a[1])}"
    if op == "CONVERTER_DECIMAL": return f"    {formatar_operando(d)} = TO_DECIMAL {formatar_operando(a[0])}"
    if op == "SE_FALSO": return f"    SE_FALSO {formatar_operando(a[0])} IR_PARA {formatar_operando(a[1])}"
    if op == "IR_PARA": return f"    IR_PARA {formatar_operando(a[0])}"
    return f"    {op} " + ", ".join(formatar_operando(x) for x in a)


def mostrar_tac(resultado, titulo=None):
    if titulo:
        print(f"--- {titulo} ({len(resultado.instrucoes)} instruções) ---")
    for instrucao in resultado.instrucoes:
        print(formatar_instrucao(instrucao))

## TAC dos três programas originais

As listagens são geradas diretamente a partir dos programas da ficha. Elas incluem cálculo e decisão, comandos de despacho/rastreamento com laço, e a alternativa entre recusa e cotação.

In [38]:
RESULTADOS_TAC_EXEMPLOS = [
    gerar_tac(programa) for programa in MINHA_FICHA["programas_exemplo"]
]
for numero, resultado in enumerate(RESULTADOS_TAC_EXEMPLOS, 1):
    mostrar_tac(resultado, f"Programa original {numero}")
    print()

--- Programa original 1 (20 instruções) ---
    declarar decimal peso@v1
    peso@v1 = 4.5
    declarar inteiro distancia@v2
    distancia@v2 = 180
    declarar decimal valor@v3
    _t1 = peso@v1 * 2.0
    _t2 = 15.0 + _t1
    _t3 = TO_DECIMAL distancia@v2
    _t4 = _t3 * 0.08
    _t5 = _t2 + _t4
    valor@v3 = _t5
    declarar inteiro prazo@v4
    prazo@v4 = 3
    _t6 = valor@v3 <= 80.0
    SE_FALSO _t6 IR_PARA L1
    COTAR 'Economico', peso@v1, valor@v3, prazo@v4
    IR_PARA L2
L1:
    MOSTRAR 'Cotacao acima do orcamento'
L2:

--- Programa original 2 (21 instruções) ---
    declarar texto codigo@v1
    codigo@v1 = 'FS-2026-001'
    declarar logico autorizado@v2
    autorizado@v2 = verdadeiro
    declarar inteiro consultas@v3
    consultas@v3 = 0
    SE_FALSO autorizado@v2 IR_PARA L1
    COTAR 'Expresso', 2.0, 45.0, 1
    DESPACHAR codigo@v1
L3:
    _t1 = consultas@v3 < 2
    SE_FALSO _t1 IR_PARA L4
    RASTREAR codigo@v1
    _t2 = consultas@v3 + 1
    consultas@v3 = _t2
    IR_PARA L

## Sombreamento, coerção e lógica

O exemplo menor evidencia duas identidades para `quantidade`, uma conversão explícita para decimal e a estrutura de curto-circuito do operador `e`.

In [39]:
EXEMPLO_TAC_RECURSOS = r'''
inteiro quantidade = 10;
se verdadeiro {
    inteiro quantidade = 2;
    mostrar quantidade;
}
mostrar quantidade;
decimal peso = 5;
logico permitido = verdadeiro e nao falso;
'''
RESULTADO_TAC_RECURSOS = gerar_tac(EXEMPLO_TAC_RECURSOS)
mostrar_tac(RESULTADO_TAC_RECURSOS, "Sombreamento, coerção e lógica")

--- Sombreamento, coerção e lógica (20 instruções) ---
    declarar inteiro quantidade@v1
    quantidade@v1 = 10
    SE_FALSO verdadeiro IR_PARA L1
    declarar inteiro quantidade@v2
    quantidade@v2 = 2
    MOSTRAR quantidade@v2
L1:
    MOSTRAR quantidade@v1
    declarar decimal peso@v3
    _t1 = TO_DECIMAL 5
    peso@v3 = _t1
    declarar logico permitido@v4
    SE_FALSO verdadeiro IR_PARA L2
    _t3 = nao falso
    _t2 = _t3
    IR_PARA L3
L2:
    _t2 = falso
L3:
    permitido@v4 = _t2


## Testes automatizados do R5

Os testes inspecionam operações, operandos, ordem, identidades, rótulos e desvios. O TAC não é executado e nenhuma otimização é aplicada nesta etapa.

In [40]:
def operacoes_de(resultado):
    return [i.operacao for i in resultado.instrucoes]


def instrucoes_de(resultado, operacao):
    return [i for i in resultado.instrucoes if i.operacao == operacao]


def rejeita_tac(fonte):
    try:
        gerar_tac(fonte)
    except ErroGeracaoTAC as erro:
        return bool(erro.diagnosticos)
    return False


def teste_precedencia_tac():
    r = gerar_tac("decimal x = 10.0 + 2.0 * 5.0;")
    bins = instrucoes_de(r, "BINARIO")
    return [i.argumentos[1] for i in bins] == ["*", "+"] and bins[1].argumentos[2] == bins[0].destino


def teste_curto_circuito():
    r = gerar_tac("logico x = verdadeiro e (1 < 2);")
    ops = operacoes_de(r)
    indice_desvio = ops.index("SE_FALSO")
    indice_comparacao = ops.index("BINARIO")
    return indice_desvio < indice_comparacao and "IR_PARA" in ops and len(instrucoes_de(r, "ROTULO")) >= 2


def teste_enquanto():
    r = gerar_tac("inteiro x = 0; enquanto x < 2 { x = x + 1; }")
    inicio = instrucoes_de(r, "ROTULO")[0].destino
    indice_inicio = next(i for i, x in enumerate(r.instrucoes) if x.operacao == "ROTULO" and x.destino == inicio)
    indice_condicao = next(i for i, x in enumerate(r.instrucoes) if x.operacao == "BINARIO" and x.argumentos[1] == "<")
    retornos = [x for x in r.instrucoes if x.operacao == "IR_PARA" and x.argumentos[0] == inicio]
    return indice_inicio < indice_condicao and len(retornos) == 1


def teste_sombreamento():
    r = gerar_tac("inteiro x = 1; se verdadeiro { inteiro x = 2; mostrar x; } mostrar x;")
    xs = [v for v in r.variaveis if v.nome == "x"]
    mostrados = [i.argumentos[0] for i in instrucoes_de(r, "MOSTRAR")]
    return len(xs) == 2 and xs[0] != xs[1] and mostrados == [xs[1], xs[0]]


def teste_atribuicao_externa():
    r = gerar_tac("inteiro x = 1; se verdadeiro { x = 2; }")
    xs = [v for v in r.variaveis if v.nome == "x"]
    destinos = [i.destino for i in instrucoes_de(r, "ATRIBUIR")]
    return len(xs) == 1 and all(d == xs[0] for d in destinos)


def teste_literal_temporario():
    r = gerar_tac('texto nome = "_t1"; inteiro x = 1 + 2; mostrar nome;')
    literal = next(a for i in r.instrucoes for a in i.argumentos if isinstance(a, LiteralTAC) and a.valor == "_t1")
    temporarios = [i.destino for i in r.instrucoes if isinstance(i.destino, TemporarioTAC)]
    return isinstance(literal, LiteralTAC) and temporarios and all(literal != t for t in temporarios)


def teste_rotulos_e_saltos(resultado):
    rotulos = [i.destino for i in instrucoes_de(resultado, "ROTULO")]
    alvos = []
    for i in resultado.instrucoes:
        if i.operacao == "IR_PARA": alvos.append(i.argumentos[0])
        if i.operacao == "SE_FALSO": alvos.append(i.argumentos[1])
    return len(rotulos) == len(set(rotulos)) and set(alvos).issubset(set(rotulos))


def teste_determinismo():
    a = gerar_tac(PROGRAMA_2)
    b = gerar_tac(PROGRAMA_2)
    return a.instrucoes == b.instrucoes and a.variaveis == b.variaveis


def teste_reinicio_contadores():
    a = gerar_tac("inteiro x = 1 + 2;")
    b = gerar_tac("inteiro y = 3 + 4;")
    temp_a = instrucoes_de(a, "BINARIO")[0].destino
    temp_b = instrucoes_de(b, "BINARIO")[0].destino
    return (
        a.variaveis[0].identificador == b.variaveis[0].identificador == 1
        and temp_a.identificador == temp_b.identificador == 1
    )


CASOS_TESTE_TAC = [
    ("Programa original 1 gera TAC", lambda: len(RESULTADOS_TAC_EXEMPLOS[0].instrucoes) > 0 and not RESULTADOS_TAC_EXEMPLOS[0].problemas_estruturais),
    ("Programa original 2 gera TAC", lambda: {"DESPACHAR", "RASTREAR"}.issubset(set(operacoes_de(RESULTADOS_TAC_EXEMPLOS[1])))),
    ("Programa original 3 gera TAC", lambda: {"RECUSAR", "COTAR"}.issubset(set(operacoes_de(RESULTADOS_TAC_EXEMPLOS[2])))),
    ("Declaração com valor", lambda: operacoes_de(gerar_tac("inteiro x = 1;")) == ["DECLARAR", "ATRIBUIR"]),
    ("Declaração sem valor", lambda: operacoes_de(gerar_tac("inteiro x;")) == ["DECLARAR"]),
    ("Atribuição", lambda: len(instrucoes_de(gerar_tac("inteiro x; x = 1;"), "ATRIBUIR")) == 1),
    ("Operações aritméticas não otimizadas", lambda: len(instrucoes_de(gerar_tac("inteiro x = 2 + 3 * 4;"), "BINARIO")) == 2),
    ("Precedência das operações", teste_precedencia_tac),
    ("Comparação relacional", lambda: any(i.argumentos[1] == ">=" for i in instrucoes_de(gerar_tac("logico x = 2 >= 1;"), "BINARIO"))),
    ("Operador e com curto-circuito", teste_curto_circuito),
    ("Operador ou com curto-circuito", lambda: (lambda r: operacoes_de(r).index("SE_FALSO") < operacoes_de(r).index("BINARIO") and len(instrucoes_de(r, "ROTULO")) == 2)(gerar_tac("logico x = verdadeiro ou (1 < 2);"))),
    ("Operador nao", lambda: any(i.argumentos[0] == "nao" for i in instrucoes_de(gerar_tac("logico x = nao falso;"), "UNARIO"))),
    ("Menos unário", lambda: len(instrucoes_de(gerar_tac("inteiro x = -2;"), "UNARIO")) == 1),
    ("Coerção inteiro para decimal", lambda: len(instrucoes_de(gerar_tac("decimal x = 2;"), "CONVERTER_DECIMAL")) == 1),
    ("Comando mostrar", lambda: len(instrucoes_de(gerar_tac("mostrar 1;"), "MOSTRAR")) == 1),
    ("Cotar preserva quatro argumentos", lambda: len(instrucoes_de(gerar_tac('cotar "E", 2.0, 3.0, 1;'), "COTAR")[0].argumentos) == 4),
    ("Comando despachar", lambda: "DESPACHAR" in operacoes_de(gerar_tac('despachar "FS";'))),
    ("Comando recusar", lambda: "RECUSAR" in operacoes_de(gerar_tac('recusar "motivo";'))),
    ("Comando rastrear", lambda: "RASTREAR" in operacoes_de(gerar_tac('rastrear "FS";'))),
    ("Condicional com senao", lambda: "IR_PARA" in operacoes_de(gerar_tac('se verdadeiro { mostrar 1; } senao { mostrar 2; }')) and len(instrucoes_de(gerar_tac('se verdadeiro { mostrar 1; } senao { mostrar 2; }'), "ROTULO")) == 2),
    ("Condicional sem senao", lambda: len(instrucoes_de(gerar_tac('se verdadeiro { mostrar 1; }'), "ROTULO")) == 1),
    ("Enquanto recalcula condição", teste_enquanto),
    ("Controle aninhado", lambda: len(instrucoes_de(gerar_tac("inteiro x = 0; enquanto x < 2 { se x == 1 { mostrar x; } x = x + 1; }"), "SE_FALSO")) == 2),
    ("Rótulos únicos", lambda: teste_rotulos_e_saltos(RESULTADOS_TAC_EXEMPLOS[1])),
    ("Desvios têm destinos", lambda: all(teste_rotulos_e_saltos(r) for r in RESULTADOS_TAC_EXEMPLOS)),
    ("Sombreamento preservado", teste_sombreamento),
    ("Atribuição externa preserva identidade", teste_atribuicao_externa),
    ("Literal textual difere de temporário", teste_literal_temporario),
    ("Erro semântico impede TAC", lambda: rejeita_tac('inteiro x = "texto";')),
    ("Aviso semântico permite TAC", lambda: len(gerar_tac('cotar "E", 2.0, 1200.0, 1;').avisos) == 1),
    ("Geração determinística", teste_determinismo),
    ("Geradores reiniciam contadores", teste_reinicio_contadores),
    ("Posições de origem preservadas", lambda: all(i.linha > 0 and i.coluna > 0 for i in gerar_tac("\n  inteiro x = 1;").instrucoes)),
    ("Declaração no corpo do laço", lambda: (lambda r: operacoes_de(r).index("ROTULO") < operacoes_de(r).index("DECLARAR") < max(i for i, op in enumerate(operacoes_de(r)) if op == "IR_PARA"))(gerar_tac("enquanto falso { inteiro local = 1; }"))),
    ("Somente operações conhecidas", lambda: all(i.operacao in OPERACOES_TAC for r in RESULTADOS_TAC_EXEMPLOS for i in r.instrucoes)),
    ("Validação estrutural limpa", lambda: all(not validar_tac(r) for r in RESULTADOS_TAC_EXEMPLOS)),
]

TESTES_R5_APROVADOS = 0
for nome, teste in CASOS_TESTE_TAC:
    try:
        passou = bool(teste())
    except Exception as erro:
        passou = False
        print(f"FALHOU — {nome}: {type(erro).__name__}: {erro}")
    else:
        print(f"{'OK' if passou else 'FALHOU'} — {nome}")
    TESTES_R5_APROVADOS += int(passou)

TESTES_R5_EXECUTADOS = len(CASOS_TESTE_TAC)
TESTES_R5_REPROVADOS = TESTES_R5_EXECUTADOS - TESTES_R5_APROVADOS
print(
    f"\nR5: {TESTES_R5_EXECUTADOS} executados, "
    f"{TESTES_R5_APROVADOS} aprovados e {TESTES_R5_REPROVADOS} reprovados."
)
assert TESTES_R5_REPROVADOS == 0, "Há testes TAC reprovados."

OK — Programa original 1 gera TAC
OK — Programa original 2 gera TAC
OK — Programa original 3 gera TAC
OK — Declaração com valor
OK — Declaração sem valor
OK — Atribuição
OK — Operações aritméticas não otimizadas
OK — Precedência das operações
OK — Comparação relacional
OK — Operador e com curto-circuito
OK — Operador ou com curto-circuito
OK — Operador nao
OK — Menos unário
OK — Coerção inteiro para decimal
OK — Comando mostrar
OK — Cotar preserva quatro argumentos
OK — Comando despachar
OK — Comando recusar
OK — Comando rastrear
OK — Condicional com senao
OK — Condicional sem senao
OK — Enquanto recalcula condição
OK — Controle aninhado
OK — Rótulos únicos
OK — Desvios têm destinos
OK — Sombreamento preservado
OK — Atribuição externa preserva identidade
OK — Literal textual difere de temporário
OK — Erro semântico impede TAC
OK — Aviso semântico permite TAC
OK — Geração determinística
OK — Geradores reiniciam contadores
OK — Posições de origem preservadas
OK — Declaração no corpo do l

## Resumo da Etapa 05

O TAC original permanece estruturado e sem otimizações, e pode ser comparado com o TAC otimizado e interpretado pela VM. As baterias anteriores são confirmadas no mesmo kernel.

In [41]:
print(
    f"R5: {TESTES_R5_EXECUTADOS} executados — "
    f"{TESTES_R5_APROVADOS} aprovados e {TESTES_R5_REPROVADOS} reprovados."
)
print(
    f"Regressão R4: {TESTES_R4_EXECUTADOS} executados — "
    f"{TESTES_R4_APROVADOS} aprovados e {TESTES_R4_REPROVADOS} reprovados."
)
print(
    f"Regressão R3: {TESTES_R3_EXECUTADOS} executados — "
    f"{TESTES_R3_APROVADOS} aprovados e {TESTES_R3_REPROVADOS} reprovados."
)
print(
    f"Regressão R2: {TESTES_EXECUTADOS} executados — "
    f"{TESTES_APROVADOS} aprovados e {TESTES_REPROVADOS} reprovados."
)
assert TESTES_R4_REPROVADOS == 0
assert TESTES_R3_REPROVADOS == 0
assert TESTES_REPROVADOS == 0

R5: 36 executados — 36 aprovados e 0 reprovados.
Regressão R4: 35 executados — 35 aprovados e 0 reprovados.
Regressão R3: 25 executados — 25 aprovados e 0 reprovados.
Regressão R2: 17 executados — 17 aprovados e 0 reprovados.


# Etapa 06 — Otimização do TAC

O otimizador recebe o TAC estruturado do R5 e produz uma nova lista de instruções. O resultado original é preservado. As transformações são conservadoras e não executam a linguagem: a equivalência observável é conferida pela máquina virtual.

## Estrutura do otimizador

Cada técnica recebe e devolve listas novas. `ResultadoOtimizacao` mantém as duas versões, as técnicas efetivamente aplicadas e a quantidade de alterações. A validação estrutural do R5 é executada depois de cada passe.

In [42]:
@dataclass
class ResultadoOtimizacao:
    original: ResultadoTAC
    otimizado: ResultadoTAC
    tecnicas_aplicadas: tuple
    alteracoes: dict


@dataclass(frozen=True)
class ComparacaoTAC:
    instrucoes_originais: int
    instrucoes_otimizadas: int
    reducao: int
    percentual: float
    tecnicas: tuple


TECNICAS_OTIMIZACAO = (
    "simplificação de constantes",
    "simplificação algébrica",
    "eliminação de código inalcançável",
)


def clonar_instrucoes(instrucoes):
    return [
        InstrucaoTAC(i.operacao, i.destino, tuple(i.argumentos), i.linha, i.coluna)
        for i in instrucoes
    ]


def _variaveis_declaradas(instrucoes, candidatas):
    declaradas = {
        i.destino for i in instrucoes if i.operacao == "DECLARAR"
    }
    return [v for v in candidatas if v in declaradas]


def _resultado_intermediario(base, instrucoes, variaveis):
    return ResultadoTAC(
        base.semantica,
        clonar_instrucoes(instrucoes),
        list(variaveis),
        [],
    )


def _validar_passe(base, instrucoes, variaveis, tecnica):
    parcial = _resultado_intermediario(base, instrucoes, variaveis)
    problemas = validar_tac(parcial)
    if problemas:
        raise RuntimeError(
            f"Falha estrutural após {tecnica}: " + "; ".join(problemas)
        )
    return parcial

## Simplificação de constantes

Somente temporários locais a um bloco básico são propagados e removidos. Operações com literais são calculadas sem `eval`. Divisão por zero, valores desconhecidos, coerções e temporários definidos em caminhos diferentes permanecem intactos.

In [43]:
def _blocos_basicos(instrucoes):
    blocos, bloco = [], 0
    for indice, instrucao in enumerate(instrucoes):
        if instrucao.operacao == "ROTULO" and indice > 0:
            bloco += 1
        blocos.append(bloco)
        if instrucao.operacao in {"SE_FALSO", "IR_PARA"}:
            bloco += 1
    return blocos


def _dados_temporarios(instrucoes):
    definicoes, usos = {}, {}
    blocos = _blocos_basicos(instrucoes)
    for indice, instrucao in enumerate(instrucoes):
        if isinstance(instrucao.destino, TemporarioTAC):
            definicoes.setdefault(instrucao.destino, []).append(indice)
        for argumento in instrucao.argumentos:
            if isinstance(argumento, TemporarioTAC):
                usos.setdefault(argumento, []).append(indice)
    return definicoes, usos, blocos


def _temporario_local(temporario, indice, definicoes, usos, blocos):
    return (
        len(definicoes.get(temporario, [])) == 1
        and all(blocos[uso] == blocos[indice] for uso in usos.get(temporario, []))
    )


def _calcular_binario(operador, a, b, tipo_resultado):
    try:
        if operador == "/" and b == 0:
            return None
        if operador == "+": valor = a + b
        elif operador == "-": valor = a - b
        elif operador == "*": valor = a * b
        elif operador == "/": valor = a / b
        elif operador == ">": valor = a > b
        elif operador == "<": valor = a < b
        elif operador == ">=": valor = a >= b
        elif operador == "<=": valor = a <= b
        elif operador == "==": valor = a == b
        elif operador == "!=": valor = a != b
        else: return None
    except (ArithmeticError, TypeError, ValueError, OverflowError):
        return None
    if tipo_resultado == "inteiro" and type(valor) is not int:
        return None
    if tipo_resultado == "decimal" and type(valor) not in (int, float):
        return None
    if tipo_resultado == "decimal":
        valor = float(valor)
    if tipo_resultado == "logico" and type(valor) is not bool:
        return None
    return LiteralTAC(valor, tipo_resultado)


def _calcular_unario(operador, valor, tipo_resultado):
    try:
        resultado = -valor if operador == "-" else (not valor if operador == "nao" else None)
    except (ArithmeticError, TypeError, ValueError):
        return None
    if resultado is None:
        return None
    return LiteralTAC(resultado, tipo_resultado)


def simplificar_constantes(instrucoes):
    definicoes, usos, blocos = _dados_temporarios(instrucoes)
    constantes, novas, alteracoes = {}, [], 0

    def substituir(argumento):
        return constantes.get(argumento, argumento) if isinstance(argumento, TemporarioTAC) else argumento

    for indice, original in enumerate(instrucoes):
        argumentos = tuple(substituir(a) for a in original.argumentos)
        atual = InstrucaoTAC(
            original.operacao, original.destino, argumentos,
            original.linha, original.coluna,
        )
        literal = None
        if atual.operacao == "BINARIO":
            esquerda, operador, direita = atual.argumentos
            if isinstance(esquerda, LiteralTAC) and isinstance(direita, LiteralTAC):
                literal = _calcular_binario(
                    operador, esquerda.valor, direita.valor, atual.destino.tipo
                )
        elif atual.operacao == "UNARIO":
            operador, origem = atual.argumentos
            if isinstance(origem, LiteralTAC):
                literal = _calcular_unario(
                    operador, origem.valor, atual.destino.tipo
                )

        if literal is not None and isinstance(atual.destino, TemporarioTAC):
            alteracoes += 1
            constantes[atual.destino] = literal
            if _temporario_local(
                atual.destino, indice, definicoes, usos, blocos
            ):
                continue
            atual = InstrucaoTAC(
                "ATRIBUIR", atual.destino, (literal,),
                atual.linha, atual.coluna,
            )

        novas.append(atual)
        if atual.operacao in {"ROTULO", "SE_FALSO", "IR_PARA"}:
            constantes.clear()
    return novas, alteracoes

## Simplificação algébrica

As identidades `x + 0`, `x - 0` e `x * 1` são aplicadas somente a inteiros. `x / 1.0` é aplicado somente quando operando e resultado já são decimais. Nenhuma coerção é removida e não há transformações inseguras como `x * 0`.

In [44]:
def _eh_literal(operando, valor, tipo):
    return (
        isinstance(operando, LiteralTAC)
        and operando.tipo == tipo
        and type(operando.valor) is (int if tipo == "inteiro" else float)
        and operando.valor == valor
    )


def _identidade_segura(instrucao):
    if instrucao.operacao != "BINARIO":
        return None
    esquerda, operador, direita = instrucao.argumentos
    destino = instrucao.destino
    if destino.tipo == "inteiro":
        if operador in {"+", "-"} and _eh_literal(direita, 0, "inteiro"):
            return esquerda
        if operador == "+" and _eh_literal(esquerda, 0, "inteiro"):
            return direita
        if operador == "*" and _eh_literal(direita, 1, "inteiro"):
            return esquerda
        if operador == "*" and _eh_literal(esquerda, 1, "inteiro"):
            return direita
    if (
        destino.tipo == "decimal" and operador == "/"
        and getattr(esquerda, "tipo", None) == "decimal"
        and _eh_literal(direita, 1.0, "decimal")
    ):
        return esquerda
    return None


def simplificar_algebra(instrucoes):
    definicoes, usos, blocos = _dados_temporarios(instrucoes)
    aliases, novas, alteracoes = {}, [], 0

    def substituir(argumento):
        visitados = set()
        while isinstance(argumento, TemporarioTAC) and argumento in aliases and argumento not in visitados:
            visitados.add(argumento)
            argumento = aliases[argumento]
        return argumento

    for indice, original in enumerate(instrucoes):
        argumentos = tuple(substituir(a) for a in original.argumentos)
        atual = InstrucaoTAC(
            original.operacao, original.destino, argumentos,
            original.linha, original.coluna,
        )
        identidade = _identidade_segura(atual)
        if identidade is not None and isinstance(atual.destino, TemporarioTAC):
            alteracoes += 1
            aliases[atual.destino] = identidade
            if _temporario_local(
                atual.destino, indice, definicoes, usos, blocos
            ):
                continue
            atual = InstrucaoTAC(
                "ATRIBUIR", atual.destino, (identidade,),
                atual.linha, atual.coluna,
            )
        novas.append(atual)
        if atual.operacao in {"ROTULO", "SE_FALSO", "IR_PARA"}:
            aliases.clear()
    return novas, alteracoes

## Eliminação de código inalcançável

O passe transforma desvios com condição literal em fluxo conhecido e calcula os índices alcançáveis por busca no grafo do TAC. Apenas blocos comprovadamente inalcançáveis são removidos. Em seguida, saltos para o rótulo imediatamente seguinte e rótulos sem referências são descartados.

In [45]:
def _mapa_rotulos(instrucoes):
    return {
        i.destino: indice
        for indice, i in enumerate(instrucoes)
        if i.operacao == "ROTULO"
    }


def _indices_alcancaveis(instrucoes):
    if not instrucoes:
        return set()
    rotulos = _mapa_rotulos(instrucoes)
    visitar, vistos = [0], set()
    while visitar:
        indice = visitar.pop()
        if indice in vistos or not 0 <= indice < len(instrucoes):
            continue
        vistos.add(indice)
        instrucao = instrucoes[indice]
        if instrucao.operacao == "IR_PARA":
            visitar.append(rotulos[instrucao.argumentos[0]])
        elif instrucao.operacao == "SE_FALSO":
            visitar.append(rotulos[instrucao.argumentos[1]])
            visitar.append(indice + 1)
        else:
            visitar.append(indice + 1)
    return vistos


def eliminar_inalcancavel(instrucoes):
    preparadas, alteracoes = [], 0
    for instrucao in instrucoes:
        if instrucao.operacao == "SE_FALSO" and isinstance(instrucao.argumentos[0], LiteralTAC):
            condicao, alvo = instrucao.argumentos
            alteracoes += 1
            if condicao.valor is False:
                preparadas.append(InstrucaoTAC(
                    "IR_PARA", None, (alvo,), instrucao.linha, instrucao.coluna
                ))
            continue  # verdadeiro: segue; falso: IR_PARA já foi incluído
        preparadas.append(instrucao)

    alcancaveis = _indices_alcancaveis(preparadas)
    filtradas = [i for n, i in enumerate(preparadas) if n in alcancaveis]
    alteracoes += len(preparadas) - len(filtradas)

    sem_saltos_redundantes = []
    for indice, instrucao in enumerate(filtradas):
        proxima = filtradas[indice + 1] if indice + 1 < len(filtradas) else None
        if (
            instrucao.operacao == "IR_PARA" and proxima is not None
            and proxima.operacao == "ROTULO"
            and instrucao.argumentos[0] == proxima.destino
        ):
            alteracoes += 1
            continue
        sem_saltos_redundantes.append(instrucao)

    alvos = set()
    for instrucao in sem_saltos_redundantes:
        if instrucao.operacao == "IR_PARA":
            alvos.add(instrucao.argumentos[0])
        elif instrucao.operacao == "SE_FALSO":
            alvos.add(instrucao.argumentos[1])
    finais = []
    for instrucao in sem_saltos_redundantes:
        if instrucao.operacao == "ROTULO" and instrucao.destino not in alvos:
            alteracoes += 1
            continue
        finais.append(instrucao)
    return finais, alteracoes

## Função principal de otimização

Os três passes são aplicados em rodadas até a estabilização, com limite defensivo. As variáveis são filtradas somente quando sua declaração foi removida junto com um bloco inalcançável. O objeto original não é alterado.

In [46]:
PASSES_OTIMIZACAO = (
    ("simplificação de constantes", simplificar_constantes),
    ("simplificação algébrica", simplificar_algebra),
    ("eliminação de código inalcançável", eliminar_inalcancavel),
)


class OtimizadorTAC:
    def __init__(self, max_rodadas=10):
        self.max_rodadas = max_rodadas

    def otimizar(self, resultado):
        problemas = validar_tac(resultado)
        if problemas:
            raise ValueError("O TAC de entrada é inválido: " + "; ".join(problemas))

        instrucoes = clonar_instrucoes(resultado.instrucoes)
        variaveis = list(resultado.variaveis)
        alteracoes = {nome: 0 for nome, _ in PASSES_OTIMIZACAO}

        for _ in range(self.max_rodadas):
            antes = list(instrucoes)
            for nome, tecnica in PASSES_OTIMIZACAO:
                instrucoes, quantidade = tecnica(instrucoes)
                alteracoes[nome] += quantidade
                variaveis = _variaveis_declaradas(instrucoes, variaveis)
                _validar_passe(resultado, instrucoes, variaveis, nome)
            if instrucoes == antes:
                break

        otimizado = _resultado_intermediario(resultado, instrucoes, variaveis)
        otimizado.problemas_estruturais = validar_tac(otimizado)
        if otimizado.problemas_estruturais:
            raise RuntimeError(
                "TAC otimizado inválido: "
                + "; ".join(otimizado.problemas_estruturais)
            )
        aplicadas = tuple(nome for nome, total in alteracoes.items() if total > 0)
        return ResultadoOtimizacao(resultado, otimizado, aplicadas, alteracoes)


def otimizar_tac(resultado_tac):
    return OtimizadorTAC().otimizar(resultado_tac)


def compilar_otimizado(codigo):
    original = gerar_tac(codigo)
    return otimizar_tac(original)

## Comparação entre TACs

As quantidades e o percentual são calculados diretamente das listas. Programas vazios produzem percentual zero, sem divisão por zero.

In [47]:
def comparar_tac(original, otimizado, tecnicas=()):
    antes = len(original.instrucoes)
    depois = len(otimizado.instrucoes)
    reducao = antes - depois
    percentual = (reducao / antes * 100) if antes else 0.0
    return ComparacaoTAC(antes, depois, reducao, percentual, tuple(tecnicas))


def mostrar_comparacao(resultado, titulo=None):
    comparacao = comparar_tac(
        resultado.original, resultado.otimizado,
        resultado.tecnicas_aplicadas,
    )
    if titulo:
        print(f"--- {titulo} ---")
    print("Instruções originais:", comparacao.instrucoes_originais)
    print("Instruções otimizadas:", comparacao.instrucoes_otimizadas)
    print("Redução:", comparacao.reducao)
    print(f"Percentual: {comparacao.percentual:.2f}%")
    print("Técnicas:", ", ".join(comparacao.tecnicas) or "nenhuma aplicável")
    return comparacao

## Programas originais

Os três programas da ficha são otimizados sem alterações em suas fontes. A ausência de redução também é um resultado válido quando não há oportunidade comprovadamente segura.

In [48]:
RESULTADOS_OTIMIZADOS_ORIGINAIS = [
    compilar_otimizado(programa)
    for programa in MINHA_FICHA["programas_exemplo"]
]
COMPARACOES_ORIGINAIS = []
for numero, resultado in enumerate(RESULTADOS_OTIMIZADOS_ORIGINAIS, 1):
    COMPARACOES_ORIGINAIS.append(
        mostrar_comparacao(resultado, f"Programa original {numero}")
    )
    print()

--- Programa original 1 ---
Instruções originais: 20
Instruções otimizadas: 20
Redução: 0
Percentual: 0.00%
Técnicas: nenhuma aplicável

--- Programa original 2 ---
Instruções originais: 21
Instruções otimizadas: 21
Redução: 0
Percentual: 0.00%
Técnicas: nenhuma aplicável

--- Programa original 3 ---
Instruções originais: 11
Instruções otimizadas: 11
Redução: 0
Percentual: 0.00%
Técnicas: nenhuma aplicável



## Programas demonstrativos

O primeiro demonstra dobra de constantes e identidade algébrica. O segundo acrescenta a remoção segura do `senao` quando a condição é literalmente verdadeira. Ambos passam por R2–R5 antes da otimização.

In [49]:
PROGRAMA_OTIMIZACAO_A = r'''
inteiro base = 2 + 3 * 4;
inteiro resultado = base + 0;
mostrar resultado;
'''

PROGRAMA_OTIMIZACAO_B = r'''
se verdadeiro {
    decimal total = 10.0 + 2.0 * 5.0;
    mostrar total;
} senao {
    mostrar "Nao deveria executar";
}
'''

OTIMIZACAO_A = compilar_otimizado(PROGRAMA_OTIMIZACAO_A)
OTIMIZACAO_B = compilar_otimizado(PROGRAMA_OTIMIZACAO_B)
COMPARACAO_A = mostrar_comparacao(OTIMIZACAO_A, "Demonstrativo A")
print()
COMPARACAO_B = mostrar_comparacao(OTIMIZACAO_B, "Demonstrativo B")

--- Demonstrativo A ---
Instruções originais: 8
Instruções otimizadas: 5
Redução: 3
Percentual: 37.50%
Técnicas: simplificação de constantes, simplificação algébrica

--- Demonstrativo B ---
Instruções originais: 10
Instruções otimizadas: 3
Redução: 7
Percentual: 70.00%
Técnicas: simplificação de constantes, eliminação de código inalcançável


## TAC antes e depois

As listagens abaixo vêm dos objetos reais. Não há execução ou equivalência de saída nesta etapa.

In [50]:
print("=== Demonstrativo A — original ===")
mostrar_tac(OTIMIZACAO_A.original)
print("\n=== Demonstrativo A — otimizado ===")
mostrar_tac(OTIMIZACAO_A.otimizado)
print("\n=== Demonstrativo B — original ===")
mostrar_tac(OTIMIZACAO_B.original)
print("\n=== Demonstrativo B — otimizado ===")
mostrar_tac(OTIMIZACAO_B.otimizado)

=== Demonstrativo A — original ===
    declarar inteiro base@v1
    _t1 = 3 * 4
    _t2 = 2 + _t1
    base@v1 = _t2
    declarar inteiro resultado@v2
    _t3 = base@v1 + 0
    resultado@v2 = _t3
    MOSTRAR resultado@v2

=== Demonstrativo A — otimizado ===
    declarar inteiro base@v1
    base@v1 = 14
    declarar inteiro resultado@v2
    resultado@v2 = base@v1
    MOSTRAR resultado@v2

=== Demonstrativo B — original ===
    SE_FALSO verdadeiro IR_PARA L1
    declarar decimal total@v1
    _t1 = 2.0 * 5.0
    _t2 = 10.0 + _t1
    total@v1 = _t2
    MOSTRAR total@v1
    IR_PARA L2
L1:
    MOSTRAR 'Nao deveria executar'
L2:

=== Demonstrativo B — otimizado ===
    declarar decimal total@v1
    total@v1 = 20.0
    MOSTRAR total@v1


## Testes automatizados do R6

Os testes verificam condições de segurança, efeitos observáveis, cópia do resultado original, determinismo, estrutura de controle e reduções reais. Nenhum teste executa o TAC.

In [51]:
def instrucoes_operacao(resultado, operacao):
    return [i for i in resultado.instrucoes if i.operacao == operacao]


def tem_binario(resultado, operador):
    return any(
        i.argumentos[1] == operador
        for i in instrucoes_operacao(resultado, "BINARIO")
    )


def teste_folding(fonte, valor, tipo):
    r = compilar_otimizado(fonte)
    atribuicoes = instrucoes_operacao(r.otimizado, "ATRIBUIR")
    return any(
        isinstance(i.argumentos[0], LiteralTAC)
        and i.argumentos[0].valor == valor
        and i.argumentos[0].tipo == tipo
        for i in atribuicoes
    )


def teste_original_intacto():
    original = gerar_tac(PROGRAMA_OTIMIZACAO_A)
    instrucoes = list(original.instrucoes)
    variaveis = list(original.variaveis)
    problemas = list(original.problemas_estruturais)
    otimizar_tac(original)
    return (
        original.instrucoes == instrucoes
        and original.variaveis == variaveis
        and original.problemas_estruturais == problemas
    )


def teste_divisao_zero_preservada():
    base = gerar_tac("decimal x = 1.0;")
    temp = TemporarioTAC(1, "decimal")
    div = InstrucaoTAC(
        "BINARIO", temp,
        (LiteralTAC(10.0, "decimal"), "/", LiteralTAC(0.0, "decimal")),
        1, 1,
    )
    atribuicao = InstrucaoTAC("ATRIBUIR", base.variaveis[0], (temp,), 1, 1)
    manual = ResultadoTAC(
        base.semantica,
        [base.instrucoes[0], div, atribuicao],
        list(base.variaveis), [],
    )
    assert not validar_tac(manual)
    otimizado = otimizar_tac(manual).otimizado
    return tem_binario(otimizado, "/")


def teste_condicao_variavel():
    fonte = 'logico ok = verdadeiro; se ok { mostrar "sim"; } senao { mostrar "nao"; }'
    r = compilar_otimizado(fonte)
    return len(instrucoes_operacao(r.otimizado, "MOSTRAR")) == 2


def sequencia_efeitos(resultado):
    efeitos = {"MOSTRAR", "COTAR", "DESPACHAR", "RECUSAR", "RASTREAR"}
    return [i.operacao for i in resultado.instrucoes if i.operacao in efeitos]


def teste_determinismo_otimizacao():
    original = gerar_tac(PROGRAMA_OTIMIZACAO_B)
    a = otimizar_tac(original)
    b = otimizar_tac(original)
    return (
        a.otimizado.instrucoes == b.otimizado.instrucoes
        and a.tecnicas_aplicadas == b.tecnicas_aplicadas
    )


def teste_divisao_por_um_decimal():
    r = compilar_otimizado("decimal x = 4.0 / 1.0;")
    return not tem_binario(r.otimizado, "/") and r.otimizado.instrucoes[-1].argumentos[0] == LiteralTAC(4.0, "decimal")


CASOS_TESTE_OTIMIZACAO = [
    ("Folding de inteiros", lambda: teste_folding("inteiro x = 2 + 3;", 5, "inteiro")),
    ("Folding de decimais", lambda: teste_folding("decimal x = 2.5 * 4.0;", 10.0, "decimal")),
    ("Precedência com folding sucessivo", lambda: teste_folding("inteiro x = 2 + 3 * 4;", 14, "inteiro")),
    ("Identidade x + 0", lambda: not tem_binario(compilar_otimizado("inteiro x = 2; inteiro y = x + 0;").otimizado, "+")),
    ("Identidade x - 0", lambda: not tem_binario(compilar_otimizado("inteiro x = 2; inteiro y = x - 0;").otimizado, "-")),
    ("Identidade x * 1", lambda: not tem_binario(compilar_otimizado("inteiro x = 2; inteiro y = x * 1;").otimizado, "*")),
    ("Identidade decimal x / 1", teste_divisao_por_um_decimal),
    ("Coerção preservada", lambda: len(instrucoes_operacao(compilar_otimizado("decimal x = 2;").otimizado, "CONVERTER_DECIMAL")) == 1),
    ("Comparação constante", lambda: teste_folding("logico x = 2 < 3;", True, "logico")),
    ("Condicional constante remove caminho", lambda: len(instrucoes_operacao(OTIMIZACAO_B.otimizado, "MOSTRAR")) == 1),
    ("Condicional variável preserva caminhos", teste_condicao_variavel),
    ("Código inalcançável removido", lambda: all(not (isinstance(a, LiteralTAC) and a.valor == "Nao deveria executar") for i in OTIMIZACAO_B.otimizado.instrucoes for a in i.argumentos)),
    ("Enquanto variável preservado", lambda: "SE_FALSO" in operacoes_de(compilar_otimizado("inteiro x = 0; enquanto x < 2 { x = x + 1; }").otimizado)),
    ("Enquanto falso remove corpo", lambda: not instrucoes_operacao(compilar_otimizado('enquanto falso { mostrar "nunca"; }').otimizado, "MOSTRAR")),
    ("Laços aninhados válidos", lambda: not validar_tac(compilar_otimizado("inteiro x = 0; enquanto x < 2 { inteiro y = 0; enquanto y < 2 { y = y + 1; } x = x + 1; }").otimizado)),
    ("Curto-circuito e preservado", lambda: not validar_tac(compilar_otimizado("logico x = verdadeiro e falso;").otimizado)),
    ("Curto-circuito ou preservado", lambda: not validar_tac(compilar_otimizado("logico x = falso ou verdadeiro;").otimizado)),
    ("Sombreamento preserva identidades", lambda: len({v.identificador for v in compilar_otimizado("inteiro x = 1; se verdadeiro { inteiro x = 2; mostrar x; } mostrar x;").otimizado.variaveis if v.nome == "x"}) == 2),
    ("Temporário com múltiplas definições válido", lambda: not validar_tac(compilar_otimizado("logico x = verdadeiro e falso;").otimizado)),
    ("COTAR alcançável preservado", lambda: "COTAR" in operacoes_de(compilar_otimizado('cotar "E", 2.0, 10.0, 1;').otimizado)),
    ("DESPACHAR preservado", lambda: "DESPACHAR" in operacoes_de(compilar_otimizado('despachar "FS";').otimizado)),
    ("RECUSAR preservado", lambda: "RECUSAR" in operacoes_de(compilar_otimizado('recusar "motivo";').otimizado)),
    ("RASTREAR preservado", lambda: "RASTREAR" in operacoes_de(compilar_otimizado('rastrear "FS";').otimizado)),
    ("MOSTRAR alcançável preservado", lambda: "MOSTRAR" in operacoes_de(compilar_otimizado("mostrar 1;").otimizado)),
    ("Divisão por zero não dobrada", teste_divisao_zero_preservada),
    ("TAC original não modificado", teste_original_intacto),
    ("Otimização determinística", teste_determinismo_otimizacao),
    ("Execuções consecutivas independentes", lambda: compilar_otimizado("inteiro a = 1 + 2;").otimizado.variaveis[0].identificador == compilar_otimizado("inteiro b = 3 + 4;").otimizado.variaveis[0].identificador == 1),
    ("Rótulos e saltos válidos", lambda: all(not validar_tac(r.otimizado) for r in RESULTADOS_OTIMIZADOS_ORIGINAIS)),
    ("Demonstrativo A reduz", lambda: COMPARACAO_A.reducao > 0),
    ("Demonstrativo B reduz", lambda: COMPARACAO_B.reducao > 0),
    ("Três técnicas aplicadas", lambda: set(TECNICAS_OTIMIZACAO).issubset(set(OTIMIZACAO_A.tecnicas_aplicadas) | set(OTIMIZACAO_B.tecnicas_aplicadas))),
    ("Caso sem oportunidade permanece", lambda: comparar_tac(compilar_otimizado('mostrar "ok";').original, compilar_otimizado('mostrar "ok";').otimizado).reducao == 0),
    ("Contagem calculada corretamente", lambda: COMPARACAO_A.reducao == COMPARACAO_A.instrucoes_originais - COMPARACAO_A.instrucoes_otimizadas),
    ("Variável mutável não vira constante", lambda: tem_binario(compilar_otimizado("inteiro x = 1; enquanto x < 3 { x = x + 1; }").otimizado, "+")),
    ("Efeitos dos originais preservados", lambda: all(sequencia_efeitos(r.original) == sequencia_efeitos(r.otimizado) for r in RESULTADOS_OTIMIZADOS_ORIGINAIS)),
]

TESTES_R6_APROVADOS = 0
for nome, teste in CASOS_TESTE_OTIMIZACAO:
    try:
        passou = bool(teste())
    except Exception as erro:
        passou = False
        print(f"FALHOU — {nome}: {type(erro).__name__}: {erro}")
    else:
        print(f"{'OK' if passou else 'FALHOU'} — {nome}")
    TESTES_R6_APROVADOS += int(passou)

TESTES_R6_EXECUTADOS = len(CASOS_TESTE_OTIMIZACAO)
TESTES_R6_REPROVADOS = TESTES_R6_EXECUTADOS - TESTES_R6_APROVADOS
print(
    f"\nR6: {TESTES_R6_EXECUTADOS} executados, "
    f"{TESTES_R6_APROVADOS} aprovados e {TESTES_R6_REPROVADOS} reprovados."
)
assert TESTES_R6_REPROVADOS == 0, "Há testes de otimização reprovados."

OK — Folding de inteiros
OK — Folding de decimais
OK — Precedência com folding sucessivo
OK — Identidade x + 0
OK — Identidade x - 0
OK — Identidade x * 1
OK — Identidade decimal x / 1
OK — Coerção preservada
OK — Comparação constante
OK — Condicional constante remove caminho
OK — Condicional variável preserva caminhos
OK — Código inalcançável removido
OK — Enquanto variável preservado
OK — Enquanto falso remove corpo
OK — Laços aninhados válidos
OK — Curto-circuito e preservado
OK — Curto-circuito ou preservado
OK — Sombreamento preserva identidades
OK — Temporário com múltiplas definições válido
OK — COTAR alcançável preservado
OK — DESPACHAR preservado
OK — RECUSAR preservado
OK — RASTREAR preservado
OK — MOSTRAR alcançável preservado
OK — Divisão por zero não dobrada
OK — TAC original não modificado
OK — Otimização determinística
OK — Execuções consecutivas independentes
OK — Rótulos e saltos válidos
OK — Demonstrativo A reduz
OK — Demonstrativo B reduz
OK — Três técnicas aplicadas

## Verificação da otimização

A validação estrutural e as condições locais de segurança são verificadas nesta seção. A comparação executável de saídas, valores finais e efeitos é apresentada na seção da máquina virtual.

In [52]:
print(
    f"R6: {TESTES_R6_EXECUTADOS} executados — "
    f"{TESTES_R6_APROVADOS} aprovados e {TESTES_R6_REPROVADOS} reprovados."
)
print(
    f"Regressão R5: {TESTES_R5_EXECUTADOS} executados — "
    f"{TESTES_R5_APROVADOS} aprovados e {TESTES_R5_REPROVADOS} reprovados."
)
print(
    f"Regressão R4: {TESTES_R4_EXECUTADOS} executados — "
    f"{TESTES_R4_APROVADOS} aprovados e {TESTES_R4_REPROVADOS} reprovados."
)
print(
    f"Regressão R3: {TESTES_R3_EXECUTADOS} executados — "
    f"{TESTES_R3_APROVADOS} aprovados e {TESTES_R3_REPROVADOS} reprovados."
)
print(
    f"Regressão R2: {TESTES_EXECUTADOS} executados — "
    f"{TESTES_APROVADOS} aprovados e {TESTES_REPROVADOS} reprovados."
)
print("Equivalência executável original × otimizado: pendente para R7.")
assert TESTES_R5_REPROVADOS == 0
assert TESTES_R4_REPROVADOS == 0
assert TESTES_R3_REPROVADOS == 0
assert TESTES_REPROVADOS == 0

R6: 36 executados — 36 aprovados e 0 reprovados.
Regressão R5: 36 executados — 36 aprovados e 0 reprovados.
Regressão R4: 35 executados — 35 aprovados e 0 reprovados.
Regressão R3: 25 executados — 25 aprovados e 0 reprovados.
Regressão R2: 17 executados — 17 aprovados e 0 reprovados.
Equivalência executável original × otimizado: pendente para R7.


# Etapa 07 — Máquina virtual e execução

A máquina virtual interpreta diretamente as dataclasses do TAC. Cada execução começa com memória, entregas, eventos, saídas e estatísticas vazias. Não há acesso a rede, arquivos, relógio ou serviços de transporte.

## Estrutura da memória e do estado

Variáveis e temporários ficam em dicionários separados e usam os próprios objetos TAC como chaves. O marcador interno `NAO_INICIALIZADA` não se confunde com nenhum valor válido. Cotações e eventos são registros imutáveis, adequados à comparação original × otimizado.

In [53]:
NAO_INICIALIZADA = object()


@dataclass(frozen=True)
class CotacaoSimulada:
    modalidade: str
    peso_kg: float
    valor_reais: float
    prazo_dias: int


@dataclass(frozen=True)
class EventoLogistico:
    operacao: str
    detalhes: tuple
    linha: int
    coluna: int


@dataclass
class EstatisticasExecucao:
    instrucoes_estaticas: int
    instrucoes_executadas: int = 0
    por_operacao: dict = field(default_factory=dict)
    cotacoes: int = 0
    despachos: int = 0
    rastreamentos: int = 0
    recusas: int = 0
    desvios_realizados: int = 0
    iteracoes_laco: int = 0

## Diagnósticos e resultado

Falhas de execução carregam categoria, posição real do TAC, descrição e dica. A VM captura a falha e devolve um resultado parcial, preservando saídas e efeitos anteriores ao erro.

In [54]:
@dataclass(frozen=True)
class DiagnosticoExecucao:
    severidade: str
    categoria: str
    linha: int
    coluna: int
    descricao: str
    dica: str
    fase: str = field(default="execução", init=False)

    def __str__(self):
        titulo = "Erro" if self.severidade == "erro" else "Aviso"
        return (
            f"{titulo} de execução — {self.categoria} — "
            f"linha {self.linha}, coluna {self.coluna}\n"
            f"Descrição: {self.descricao}\n"
            f"Dica: {self.dica}"
        )


@dataclass
class ResultadoExecucao:
    sucesso: bool
    saidas: list
    eventos: list
    cotacoes: list
    entregas: dict
    memoria_variaveis: dict
    memoria_temporarios: dict
    variaveis_finais: dict
    erro: DiagnosticoExecucao | None
    avisos_execucao: list
    avisos_compilacao: list
    estatisticas: EstatisticasExecucao


class _FalhaExecucao(Exception):
    def __init__(self, diagnostico):
        self.diagnostico = diagnostico
        super().__init__(str(diagnostico))

## Operandos, operações e tipos

Literais são lidos diretamente; variáveis e temporários são consultados em suas memórias. As operações são implementadas explicitamente, sem `eval` ou `exec`. A divisão verifica zero no momento em que é executada.

In [55]:
def formatar_valor_execucao(valor):
    if type(valor) is bool:
        return "verdadeiro" if valor else "falso"
    if type(valor) is float:
        if valor == 0:
            valor = 0.0
        return str(valor)
    return str(valor)


class MaquinaVirtual:
    def __init__(self, resultado_tac, max_passos=10_000):
        problemas = validar_tac(resultado_tac)
        if problemas:
            raise ValueError("TAC inválido para execução: " + "; ".join(problemas))
        if not isinstance(max_passos, int) or max_passos < 1:
            raise ValueError("max_passos precisa ser um inteiro positivo.")
        self.tac = resultado_tac
        self.max_passos = max_passos
        self.variaveis = {}
        self.temporarios = {}
        self.saidas = []
        self.eventos = []
        self.cotacoes = []
        self.entregas = {}
        self.avisos_execucao = []
        self.estatisticas = EstatisticasExecucao(len(resultado_tac.instrucoes))
        self.rotulos = {
            i.destino: indice
            for indice, i in enumerate(resultado_tac.instrucoes)
            if i.operacao == "ROTULO"
        }

    def falhar(self, instrucao, categoria, descricao, dica):
        raise _FalhaExecucao(DiagnosticoExecucao(
            "erro", categoria, instrucao.linha, instrucao.coluna,
            descricao, dica,
        ))

    def avisar(self, instrucao, categoria, descricao, dica):
        aviso = DiagnosticoExecucao(
            "aviso", categoria, instrucao.linha, instrucao.coluna,
            descricao, dica,
        )
        if aviso not in self.avisos_execucao:
            self.avisos_execucao.append(aviso)

    def resolver(self, operando, instrucao):
        if isinstance(operando, LiteralTAC):
            return operando.valor
        if isinstance(operando, VariavelTAC):
            if operando not in self.variaveis:
                self.falhar(
                    instrucao, "VARIAVEL_NAO_DECLARADA",
                    f"A variável interna '{operando.nome}' não foi declarada na execução.",
                    "Confira a instrução DECLARAR no TAC."
                )
            valor = self.variaveis[operando]
            if valor is NAO_INICIALIZADA:
                self.falhar(
                    instrucao, "VALOR_NAO_INICIALIZADO",
                    f"A variável '{operando.nome}' foi lida sem valor.",
                    "Atribua um valor antes da leitura."
                )
            return valor
        if isinstance(operando, TemporarioTAC):
            if operando not in self.temporarios:
                self.falhar(
                    instrucao, "TEMPORARIO_SEM_VALOR",
                    f"O temporário _t{operando.identificador} foi lido sem valor.",
                    "Confira as definições de temporários no TAC."
                )
            return self.temporarios[operando]
        self.falhar(
            instrucao, "OPERANDO_INVALIDO", "O TAC contém um operando inválido.",
            "Gere novamente o código intermediário."
        )

    def atribuir(self, destino, valor, instrucao):
        if isinstance(destino, VariavelTAC):
            if destino not in self.variaveis:
                self.falhar(
                    instrucao, "VARIAVEL_NAO_DECLARADA",
                    f"A variável interna '{destino.nome}' não foi declarada.",
                    "Confira a instrução DECLARAR."
                )
            self.variaveis[destino] = valor
        elif isinstance(destino, TemporarioTAC):
            self.temporarios[destino] = valor
        else:
            self.falhar(
                instrucao, "DESTINO_INVALIDO", "A instrução possui destino inválido.",
                "Gere novamente o TAC."
            )

    def binario(self, instrucao):
        esquerda, operador, direita = instrucao.argumentos
        a = self.resolver(esquerda, instrucao)
        b = self.resolver(direita, instrucao)
        if operador == "/" and b == 0:
            self.falhar(
                instrucao, "DIVISAO_POR_ZERO", "Tentativa de divisão por zero.",
                "Garanta que o divisor seja diferente de zero durante a execução."
            )
        try:
            if operador == "+": valor = a + b
            elif operador == "-": valor = a - b
            elif operador == "*": valor = a * b
            elif operador == "/": valor = a / b
            elif operador == ">": valor = a > b
            elif operador == "<": valor = a < b
            elif operador == ">=": valor = a >= b
            elif operador == "<=": valor = a <= b
            elif operador == "==": valor = a == b
            elif operador == "!=": valor = a != b
            else:
                self.falhar(
                    instrucao, "OPERADOR_DESCONHECIDO",
                    f"Operador binário desconhecido: {operador!r}.",
                    "Utilize um operador definido pela linguagem."
                )
        except (ArithmeticError, TypeError, ValueError) as erro:
            self.falhar(
                instrucao, "OPERACAO_INVALIDA", str(erro),
                "Confira os operandos da operação."
            )
        self.atribuir(instrucao.destino, valor, instrucao)

    def unario(self, instrucao):
        operador, origem = instrucao.argumentos
        valor = self.resolver(origem, instrucao)
        if operador == "nao":
            if type(valor) is not bool:
                self.falhar(
                    instrucao, "TIPO_LOGICO_INVALIDO",
                    "O operador 'nao' recebeu valor não lógico.",
                    "Use 'nao' somente com valores logicos."
                )
            resultado = not valor
        elif operador == "-":
            if type(valor) not in (int, float):
                self.falhar(
                    instrucao, "TIPO_NUMERICO_INVALIDO",
                    "O sinal negativo recebeu valor não numérico.",
                    "Use '-' somente com inteiro ou decimal."
                )
            resultado = -valor
        else:
            self.falhar(
                instrucao, "OPERADOR_DESCONHECIDO",
                f"Operador unário desconhecido: {operador!r}.",
                "Utilize '-' ou 'nao'."
            )
        self.atribuir(instrucao.destino, resultado, instrucao)

## Contador de programa e desvios

O PC avança uma instrução por vez. `IR_PARA` e `SE_FALSO` consultam o mapa de rótulos. Um salto para trás conta uma iteração concluída. O limite de passos é independente da velocidade da máquina.

In [56]:
def _executar_vm(self):
    pc, erro = 0, None
    instrucoes = self.tac.instrucoes
    try:
        while pc < len(instrucoes):
            instrucao = instrucoes[pc]
            if self.estatisticas.instrucoes_executadas >= self.max_passos:
                self.falhar(
                    instrucao, "LIMITE_PASSOS",
                    f"O limite de {self.max_passos} instruções executadas foi atingido.",
                    "Revise a condição do laço ou aumente max_passos para um programa finito."
                )
            indice_atual = pc
            pc += 1
            self.estatisticas.instrucoes_executadas += 1
            self.estatisticas.por_operacao[instrucao.operacao] = (
                self.estatisticas.por_operacao.get(instrucao.operacao, 0) + 1
            )

            op = instrucao.operacao
            if op == "DECLARAR":
                self.variaveis[instrucao.destino] = NAO_INICIALIZADA
            elif op == "ATRIBUIR":
                self.atribuir(
                    instrucao.destino,
                    self.resolver(instrucao.argumentos[0], instrucao),
                    instrucao,
                )
            elif op == "BINARIO":
                self.binario(instrucao)
            elif op == "UNARIO":
                self.unario(instrucao)
            elif op == "CONVERTER_DECIMAL":
                valor = self.resolver(instrucao.argumentos[0], instrucao)
                if type(valor) is not int:
                    self.falhar(
                        instrucao, "COERCAO_INVALIDA",
                        "TO_DECIMAL recebeu um valor que não é inteiro.",
                        "Converta somente valores inteiro para decimal."
                    )
                self.atribuir(instrucao.destino, float(valor), instrucao)
            elif op == "ROTULO":
                pass
            elif op == "IR_PARA":
                alvo = instrucao.argumentos[0]
                destino = self.rotulos[alvo]
                self.estatisticas.desvios_realizados += 1
                if destino <= indice_atual:
                    self.estatisticas.iteracoes_laco += 1
                pc = destino
            elif op == "SE_FALSO":
                condicao = self.resolver(instrucao.argumentos[0], instrucao)
                if type(condicao) is not bool:
                    self.falhar(
                        instrucao, "CONDICAO_NAO_LOGICA",
                        "SE_FALSO recebeu uma condição que não é logica.",
                        "Gere uma expressão logica antes do desvio."
                    )
                if not condicao:
                    self.estatisticas.desvios_realizados += 1
                    pc = self.rotulos[instrucao.argumentos[1]]
            elif op == "MOSTRAR":
                valor = self.resolver(instrucao.argumentos[0], instrucao)
                self.saidas.append(formatar_valor_execucao(valor))
            elif op in {"COTAR", "DESPACHAR", "RECUSAR", "RASTREAR"}:
                self.comando_logistico(instrucao)
            else:
                self.falhar(
                    instrucao, "INSTRUCAO_DESCONHECIDA",
                    f"Instrução TAC desconhecida: {op!r}.",
                    "Gere novamente o TAC."
                )
    except _FalhaExecucao as falha:
        erro = falha.diagnostico

    valores_finais = {}
    for variavel, valor in self.variaveis.items():
        if variavel.escopo == "global":
            chave = f"{variavel.nome}@v{variavel.identificador}"
            valores_finais[chave] = (
                "<não inicializada>" if valor is NAO_INICIALIZADA else valor
            )
    return ResultadoExecucao(
        erro is None,
        list(self.saidas), list(self.eventos), list(self.cotacoes),
        dict(self.entregas), dict(self.variaveis), dict(self.temporarios),
        valores_finais, erro, list(self.avisos_execucao),
        list(self.tac.semantica.avisos), self.estatisticas,
    )


MaquinaVirtual.executar = _executar_vm

## Comandos logísticos

As operações registram eventos e saídas determinísticas. `COTAR` usa o valor calculado pelo programa; `DESPACHAR` cria status local; `RASTREAR` apenas consulta; e `RECUSAR` registra o motivo sem encerrar a execução.

In [57]:
def _comando_logistico_vm(self, instrucao):
    valores = [self.resolver(a, instrucao) for a in instrucao.argumentos]
    op = instrucao.operacao

    if op == "COTAR":
        modalidade, peso, valor, prazo = valores
        if type(modalidade) is not str:
            self.falhar(instrucao, "TIPO_MODALIDADE", "A modalidade não é texto.", "Informe uma modalidade textual.")
        if type(peso) not in (int, float) or peso <= 0:
            self.falhar(instrucao, "PESO_INVALIDO", "O peso da cotação precisa ser maior que zero.", "Informe um peso positivo em quilogramas.")
        if type(valor) not in (int, float):
            self.falhar(instrucao, "TIPO_VALOR", "O valor da cotação não é numérico.", "Informe o valor em reais.")
        if type(prazo) is not int or prazo < 1:
            self.falhar(instrucao, "PRAZO_INVALIDO", "O prazo da cotação precisa ser inteiro e pelo menos 1.", "Informe um prazo inteiro maior ou igual a 1.")
        cotacao = CotacaoSimulada(modalidade, float(peso), float(valor), prazo)
        self.cotacoes.append(cotacao)
        self.eventos.append(EventoLogistico(
            "COTAR",
            (("modalidade", modalidade), ("peso_kg", float(peso)),
             ("valor_reais", float(valor)), ("prazo_dias", prazo)),
            instrucao.linha, instrucao.coluna,
        ))
        self.estatisticas.cotacoes += 1
        self.saidas.append(
            f"Cotação {modalidade}: {formatar_valor_execucao(float(peso))} kg, "
            f"R$ {float(valor):.2f}, prazo {prazo} dia(s)"
        )
        ja_avisado = any(
            getattr(a, "codigo", "") == "FRETE_ELEVADO"
            and a.linha == instrucao.linha
            for a in self.tac.semantica.avisos
        )
        if valor > 1000 and not ja_avisado:
            self.avisar(
                instrucao, "FRETE_ELEVADO",
                f"O valor executado do frete é superior a R$ 1.000 ({float(valor):.2f}).",
                "Revise o cálculo antes de confirmar a cotação."
            )
        return

    if op == "DESPACHAR":
        codigo = valores[0]
        if type(codigo) is not str:
            self.falhar(instrucao, "TIPO_CODIGO", "O código de despacho não é texto.", "Informe um código textual.")
        self.entregas[codigo] = "Despachado"
        self.eventos.append(EventoLogistico(
            "DESPACHAR", (("codigo", codigo), ("status", "Despachado")),
            instrucao.linha, instrucao.coluna,
        ))
        self.estatisticas.despachos += 1
        self.saidas.append(f"Despacho {codigo}: Despachado")
        return

    if op == "RASTREAR":
        codigo = valores[0]
        if codigo not in self.entregas:
            self.falhar(
                instrucao, "RASTREAMENTO_INEXISTENTE",
                f"O código '{codigo}' não foi encontrado.",
                "Despache o código antes de rastreá-lo ou confira a grafia."
            )
        status = self.entregas[codigo]
        self.eventos.append(EventoLogistico(
            "RASTREAR", (("codigo", codigo), ("status", status)),
            instrucao.linha, instrucao.coluna,
        ))
        self.estatisticas.rastreamentos += 1
        self.saidas.append(f"Rastreamento {codigo}: {status}")
        return

    motivo = valores[0]
    if type(motivo) is not str:
        self.falhar(instrucao, "TIPO_MOTIVO", "O motivo da recusa não é texto.", "Informe um motivo textual.")
    self.eventos.append(EventoLogistico(
        "RECUSAR", (("motivo", motivo),), instrucao.linha, instrucao.coluna
    ))
    self.estatisticas.recusas += 1
    self.saidas.append(f"Recusa: {motivo}")


MaquinaVirtual.comando_logistico = _comando_logistico_vm

## Funções principais e apresentação

`executar_tac` recebe código intermediário pronto. `executar_fretescript` usa o pipeline e permite selecionar TAC original ou otimizado. A VM nunca imprime durante a execução; a apresentação é responsabilidade de `mostrar_execucao`.

In [58]:
def executar_tac(resultado_tac, max_passos=10_000):
    return MaquinaVirtual(resultado_tac, max_passos).executar()


def executar_fretescript(codigo, otimizar=False, max_passos=10_000):
    if otimizar:
        resultado_tac = compilar_otimizado(codigo).otimizado
    else:
        resultado_tac = gerar_tac(codigo)
    return executar_tac(resultado_tac, max_passos)


def mostrar_execucao(resultado, titulo=None):
    if titulo:
        print(f"--- {titulo} ---")
    print("Situação:", "sucesso" if resultado.sucesso else "erro")
    print("Saídas:")
    for saida in resultado.saidas:
        print(" ", saida)
    if resultado.erro:
        print(resultado.erro)
    if resultado.avisos_compilacao:
        print("Avisos da compilação:")
        for aviso in resultado.avisos_compilacao:
            print(" ", aviso.descricao)
    if resultado.avisos_execucao:
        print("Avisos da execução:")
        for aviso in resultado.avisos_execucao:
            print(" ", aviso.descricao)
    e = resultado.estatisticas
    print(
        f"Estatísticas: {e.instrucoes_executadas} instruções executadas; "
        f"{e.iteracoes_laco} iteração(ões); {e.desvios_realizados} desvio(s); "
        f"{e.cotacoes} cotação(ões); {e.despachos} despacho(s); "
        f"{e.rastreamentos} rastreamento(s); {e.recusas} recusa(s)."
    )

## Execução dos três programas originais

As execuções abaixo usam os textos preservados na ficha e o TAC original do R5.

In [59]:
EXECUCOES_ORIGINAIS = [
    executar_fretescript(programa)
    for programa in MINHA_FICHA["programas_exemplo"]
]
for numero, resultado in enumerate(EXECUCOES_ORIGINAIS, 1):
    mostrar_execucao(resultado, f"Programa original {numero}")
    print()

--- Programa original 1 ---
Situação: sucesso
Saídas:
  Cotação Economico: 4.5 kg, R$ 38.40, prazo 3 dia(s)
Estatísticas: 18 instruções executadas; 0 iteração(ões); 1 desvio(s); 1 cotação(ões); 0 despacho(s); 0 rastreamento(s); 0 recusa(s).

--- Programa original 2 ---
Situação: sucesso
Saídas:
  Cotação Expresso: 2.0 kg, R$ 45.00, prazo 1 dia(s)
  Despacho FS-2026-001: Despachado
  Rastreamento FS-2026-001: Despachado
  Rastreamento FS-2026-001: Despachado
Estatísticas: 29 instruções executadas; 2 iteração(ões); 4 desvio(s); 1 cotação(ões); 1 despacho(s); 2 rastreamento(s); 0 recusa(s).

--- Programa original 3 ---
Situação: sucesso
Saídas:
  Recusa: Carga acima da capacidade do veiculo
Estatísticas: 9 instruções executadas; 0 iteração(ões); 1 desvio(s); 0 cotação(ões); 0 despacho(s); 0 rastreamento(s); 1 recusa(s).



## Execução dos demonstrativos de otimização

Os dois programas do R6 são executados nas versões original e otimizada. As estatísticas podem mudar; as observações devem permanecer iguais.

In [60]:
EXECUCAO_DEMO_A_ORIGINAL = executar_fretescript(PROGRAMA_OTIMIZACAO_A)
EXECUCAO_DEMO_A_OTIMIZADA = executar_fretescript(PROGRAMA_OTIMIZACAO_A, otimizar=True)
EXECUCAO_DEMO_B_ORIGINAL = executar_fretescript(PROGRAMA_OTIMIZACAO_B)
EXECUCAO_DEMO_B_OTIMIZADA = executar_fretescript(PROGRAMA_OTIMIZACAO_B, otimizar=True)

mostrar_execucao(EXECUCAO_DEMO_A_ORIGINAL, "Demonstrativo A — original")
mostrar_execucao(EXECUCAO_DEMO_A_OTIMIZADA, "Demonstrativo A — otimizado")
mostrar_execucao(EXECUCAO_DEMO_B_ORIGINAL, "Demonstrativo B — original")
mostrar_execucao(EXECUCAO_DEMO_B_OTIMIZADA, "Demonstrativo B — otimizado")

--- Demonstrativo A — original ---
Situação: sucesso
Saídas:
  14
Estatísticas: 8 instruções executadas; 0 iteração(ões); 0 desvio(s); 0 cotação(ões); 0 despacho(s); 0 rastreamento(s); 0 recusa(s).
--- Demonstrativo A — otimizado ---
Situação: sucesso
Saídas:
  14
Estatísticas: 5 instruções executadas; 0 iteração(ões); 0 desvio(s); 0 cotação(ões); 0 despacho(s); 0 rastreamento(s); 0 recusa(s).
--- Demonstrativo B — original ---
Situação: sucesso
Saídas:
  20.0
Estatísticas: 8 instruções executadas; 0 iteração(ões); 1 desvio(s); 0 cotação(ões); 0 despacho(s); 0 rastreamento(s); 0 recusa(s).
--- Demonstrativo B — otimizado ---
Situação: sucesso
Saídas:
  20.0
Estatísticas: 3 instruções executadas; 0 iteração(ões); 0 desvio(s); 0 cotação(ões); 0 despacho(s); 0 rastreamento(s); 0 recusa(s).


## Comparação original × otimizado

A comparação considera saídas, eventos, cotações, entregas, variáveis globais, avisos e erro. Estatísticas e temporários não são observáveis. Comparações interrompidas pelo limite de passos são marcadas como inconclusivas.

In [61]:
@dataclass
class ComparacaoExecucoes:
    original: ResultadoExecucao
    otimizada: ResultadoExecucao
    equivalentes: bool
    inconclusiva: bool
    diferencas: list


def _resumo_diagnostico(diagnostico):
    if diagnostico is None:
        return None
    return (
        diagnostico.categoria, diagnostico.linha, diagnostico.coluna,
        diagnostico.descricao,
    )


def _observavel(resultado):
    return {
        "sucesso": resultado.sucesso,
        "saidas": resultado.saidas,
        "eventos": resultado.eventos,
        "cotacoes": resultado.cotacoes,
        "entregas": resultado.entregas,
        "variaveis_finais": resultado.variaveis_finais,
        "avisos_execucao": [_resumo_diagnostico(a) for a in resultado.avisos_execucao],
        "avisos_compilacao": [
            (a.codigo, a.linha, a.coluna, a.descricao)
            for a in resultado.avisos_compilacao
        ],
        "erro": _resumo_diagnostico(resultado.erro),
    }


def comparar_execucoes(codigo, max_passos=10_000):
    compilacao = compilar_otimizado(codigo)
    original = executar_tac(compilacao.original, max_passos)
    otimizada = executar_tac(compilacao.otimizado, max_passos)
    inconclusiva = any(
        r.erro is not None and r.erro.categoria == "LIMITE_PASSOS"
        for r in (original, otimizada)
    )
    diferencas = []
    if inconclusiva:
        diferencas.append("O limite de passos impediu concluir a equivalência.")
    else:
        a, b = _observavel(original), _observavel(otimizada)
        for campo in a:
            if a[campo] != b[campo]:
                diferencas.append(
                    f"{campo}: original={a[campo]!r}; otimizado={b[campo]!r}"
                )
    return ComparacaoExecucoes(
        original, otimizada, not inconclusiva and not diferencas,
        inconclusiva, diferencas,
    )


def mostrar_comparacao_execucoes(comparacao, titulo=None):
    if titulo:
        print(f"--- {titulo} ---")
    if comparacao.inconclusiva:
        estado = "INCONCLUSIVA"
    else:
        estado = "EQUIVALENTES" if comparacao.equivalentes else "DIVERGENTES"
    print("Original × Otimizado:", estado)
    for diferenca in comparacao.diferencas:
        print(" ", diferenca)

## Comparações obrigatórias

São comparados os três programas, os dois demonstrativos, sombreamento, decisão, laço, curto-circuito, erro dinâmico e aviso logístico.

In [62]:
PROGRAMA_SOMBREAMENTO_VM = 'inteiro x = 10; se verdadeiro { inteiro x = 2; mostrar x; } mostrar x;'
PROGRAMA_LACO_VM = 'inteiro x = 0; enquanto x < 3 { mostrar x; x = x + 1; }'
PROGRAMA_CURTO_VM = 'inteiro d = 1; enquanto d > 0 { d = d - 1; } logico x = falso e (10 / d > 0); mostrar x;'
PROGRAMA_AVISO_VM = 'decimal valor = 999.0; enquanto valor < 1000.0 { valor = valor + 2.0; } cotar "E", 2.0, valor, 1;'

COMPARACOES_EXECUCAO = {
    "Original 1": comparar_execucoes(PROGRAMA_1),
    "Original 2": comparar_execucoes(PROGRAMA_2),
    "Original 3": comparar_execucoes(PROGRAMA_3),
    "Demonstrativo A": comparar_execucoes(PROGRAMA_OTIMIZACAO_A),
    "Demonstrativo B": comparar_execucoes(PROGRAMA_OTIMIZACAO_B),
    "Sombreamento": comparar_execucoes(PROGRAMA_SOMBREAMENTO_VM),
    "Condicional": comparar_execucoes('se falso { mostrar "a"; } senao { mostrar "b"; }'),
    "Laço": comparar_execucoes(PROGRAMA_LACO_VM),
    "Curto-circuito": comparar_execucoes(PROGRAMA_CURTO_VM),
    "Erro dinâmico": comparar_execucoes('rastrear "FS-INEXISTENTE";'),
    "Aviso logístico": comparar_execucoes(PROGRAMA_AVISO_VM),
}

for nome, comparacao in COMPARACOES_EXECUCAO.items():
    mostrar_comparacao_execucoes(comparacao, nome)

--- Original 1 ---
Original × Otimizado: EQUIVALENTES
--- Original 2 ---
Original × Otimizado: EQUIVALENTES
--- Original 3 ---
Original × Otimizado: EQUIVALENTES
--- Demonstrativo A ---
Original × Otimizado: EQUIVALENTES
--- Demonstrativo B ---
Original × Otimizado: EQUIVALENTES
--- Sombreamento ---
Original × Otimizado: EQUIVALENTES
--- Condicional ---
Original × Otimizado: EQUIVALENTES
--- Laço ---
Original × Otimizado: EQUIVALENTES
--- Curto-circuito ---
Original × Otimizado: EQUIVALENTES
--- Erro dinâmico ---
Original × Otimizado: EQUIVALENTES
--- Aviso logístico ---
Original × Otimizado: EQUIVALENTES


## Galeria de erros dinâmicos

Os cinco programas passam por R2, R3 e R4. Os erros aparecem somente com os valores reais da execução ou com o limite configurado. A proteção contra valor não inicializado é demonstrada separadamente com um TAC defensivo, pois o R4 impede que esse caso seja gerado a partir de um programa aprovado.

In [63]:
ERROS_DINAMICOS = {
    "Divisão por zero": r'''
inteiro divisor = 1;
enquanto divisor > 0 { divisor = divisor - 1; }
decimal resultado = 10.0 / divisor;
mostrar resultado;
''',
    "Rastreamento inexistente": 'rastrear "FS-INEXISTENTE";',
    "Peso inválido": r'''
decimal peso = 1.0;
enquanto peso > 0.0 { peso = peso - 1.0; }
cotar "Expresso", peso, 45.0, 2;
''',
    "Prazo inválido": r'''
inteiro prazo = 2;
enquanto prazo > 0 { prazo = prazo - 1; }
cotar "Economico", 2.0, 50.0, prazo;
''',
    "Limite de passos": 'enquanto verdadeiro { mostrar "Repetindo"; }',
}

RESULTADOS_ERROS_DINAMICOS = {}
for nome, fonte in ERROS_DINAMICOS.items():
    analisar_lexico(fonte)
    analisar_sintaxe(fonte)
    semantica = analisar_semantica(fonte)
    assert semantica.aprovado
    max_passos = 20 if nome == "Limite de passos" else 10_000
    resultado = executar_fretescript(fonte, max_passos=max_passos)
    RESULTADOS_ERROS_DINAMICOS[nome] = resultado
    print(f"\n--- {nome} ---")
    print(resultado.erro)

# Caso defensivo: um TAC externo tenta ler uma variável apenas declarada.
_tac_declaracao = gerar_tac("inteiro pendente;")
_tac_nao_inicializado = ResultadoTAC(
    _tac_declaracao.semantica,
    list(_tac_declaracao.instrucoes) + [
        InstrucaoTAC(
            "MOSTRAR", None, (_tac_declaracao.variaveis[0],), 1, 1
        )
    ],
    list(_tac_declaracao.variaveis),
    [],
)
RESULTADO_NAO_INICIALIZADO = executar_tac(_tac_nao_inicializado)
print("\n--- Valor não inicializado (proteção defensiva) ---")
print(RESULTADO_NAO_INICIALIZADO.erro)


--- Divisão por zero ---
Erro de execução — DIVISAO_POR_ZERO — linha 4, coluna 26
Descrição: Tentativa de divisão por zero.
Dica: Garanta que o divisor seja diferente de zero durante a execução.

--- Rastreamento inexistente ---
Erro de execução — RASTREAMENTO_INEXISTENTE — linha 1, coluna 1
Descrição: O código 'FS-INEXISTENTE' não foi encontrado.
Dica: Despache o código antes de rastreá-lo ou confira a grafia.

--- Peso inválido ---
Erro de execução — PESO_INVALIDO — linha 4, coluna 1
Descrição: O peso da cotação precisa ser maior que zero.
Dica: Informe um peso positivo em quilogramas.

--- Prazo inválido ---
Erro de execução — PRAZO_INVALIDO — linha 4, coluna 1
Descrição: O prazo da cotação precisa ser inteiro e pelo menos 1.
Dica: Informe um prazo inteiro maior ou igual a 1.

--- Limite de passos ---
Erro de execução — LIMITE_PASSOS — linha 1, coluna 1
Descrição: O limite de 20 instruções executadas foi atingido.
Dica: Revise a condição do laço ou aumente max_passos para um progra

## Testes automatizados do R7

Os testes verificam resultados reais, efeitos, erros, estatísticas, isolamento e equivalência. A bateria de integração aparece na seção seguinte.

In [64]:
def executar_codigo(fonte, otimizar=False, max_passos=10_000):
    return executar_fretescript(fonte, otimizar, max_passos)


def erro_categoria(resultado, categoria):
    return (
        not resultado.sucesso and resultado.erro is not None
        and resultado.erro.categoria == categoria
        and resultado.erro.linha > 0 and resultado.erro.coluna > 0
        and bool(resultado.erro.dica)
    )


def teste_curto_e():
    r = executar_codigo(PROGRAMA_CURTO_VM)
    return r.sucesso and r.saidas == ["falso"]


def teste_curto_ou():
    fonte = 'inteiro d = 1; enquanto d > 0 { d = d - 1; } logico x = verdadeiro ou (10 / d > 0); mostrar x;'
    r = executar_codigo(fonte)
    return r.sucesso and r.saidas == ["verdadeiro"]


def teste_reinicializacao_laco():
    fonte = 'inteiro i = 0; enquanto i < 2 { inteiro local = i; mostrar local; i = i + 1; }'
    r = executar_codigo(fonte)
    return r.sucesso and r.saidas == ["0", "1"]


def teste_memorias_independentes():
    tac = gerar_tac("inteiro x = 1; mostrar x;")
    a, b = executar_tac(tac), executar_tac(tac)
    chave = next(iter(a.memoria_variaveis))
    a.memoria_variaveis[chave] = 999
    return b.memoria_variaveis[chave] == 1 and a.saidas == b.saidas


def teste_tac_nao_alterado():
    tac = gerar_tac(PROGRAMA_2)
    antes = list(tac.instrucoes)
    executar_tac(tac)
    return tac.instrucoes == antes


CASOS_TESTE_VM = [
    ("Programa original 1", lambda: EXECUCOES_ORIGINAIS[0].sucesso and len(EXECUCOES_ORIGINAIS[0].cotacoes) == 1 and abs(EXECUCOES_ORIGINAIS[0].cotacoes[0].valor_reais - 38.4) < 1e-9 and EXECUCOES_ORIGINAIS[0].cotacoes[0].modalidade == "Economico"),
    ("Programa original 2", lambda: EXECUCOES_ORIGINAIS[1].sucesso and EXECUCOES_ORIGINAIS[1].entregas == {"FS-2026-001": "Despachado"} and EXECUCOES_ORIGINAIS[1].estatisticas.rastreamentos == 2 and EXECUCOES_ORIGINAIS[1].estatisticas.iteracoes_laco == 2),
    ("Programa original 3", lambda: EXECUCOES_ORIGINAIS[2].sucesso and EXECUCOES_ORIGINAIS[2].estatisticas.recusas == 1 and not EXECUCOES_ORIGINAIS[2].cotacoes),
    ("Declaração e atribuição", lambda: executar_codigo("inteiro x; x = 2; mostrar x;").saidas == ["2"]),
    ("Declaração sem inicialização", lambda: executar_codigo("inteiro x;").variaveis_finais["x@v1"] == "<não inicializada>"),
    ("Coerção inteiro para decimal", lambda: executar_codigo("decimal x = 5; mostrar x;").saidas == ["5.0"]),
    ("Operadores aritméticos", lambda: executar_codigo("mostrar 2 + 3 * 4 - 1;").saidas == ["13"]),
    ("Divisão produz decimal", lambda: executar_codigo("mostrar 5 / 2;").saidas == ["2.5"]),
    ("Comparações", lambda: executar_codigo("mostrar 2 < 3; mostrar 3 == 3;").saidas == ["verdadeiro", "verdadeiro"]),
    ("Operador nao", lambda: executar_codigo("mostrar nao falso;").saidas == ["verdadeiro"]),
    ("Curto-circuito e", teste_curto_e),
    ("Curto-circuito ou", teste_curto_ou),
    ("Se verdadeiro", lambda: executar_codigo('se verdadeiro { mostrar "sim"; }').saidas == ["sim"]),
    ("Se falso", lambda: executar_codigo('se falso { mostrar "nao"; }').saidas == []),
    ("Se com senao", lambda: executar_codigo('se falso { mostrar "a"; } senao { mostrar "b"; }').saidas == ["b"]),
    ("Enquanto com iterações", lambda: executar_codigo(PROGRAMA_LACO_VM).saidas == ["0", "1", "2"] and executar_codigo(PROGRAMA_LACO_VM).estatisticas.iteracoes_laco == 3),
    ("Enquanto zero iterações", lambda: executar_codigo('enquanto falso { mostrar "nunca"; }').saidas == []),
    ("Laços aninhados", lambda: executar_codigo('inteiro i = 0; enquanto i < 2 { inteiro j = 0; enquanto j < 2 { mostrar i + j; j = j + 1; } i = i + 1; }').saidas == ["0", "1", "1", "2"]),
    ("Sombreamento", lambda: executar_codigo(PROGRAMA_SOMBREAMENTO_VM).saidas == ["2", "10"]),
    ("Atribuição externa", lambda: executar_codigo('inteiro x = 1; se verdadeiro { x = 3; } mostrar x;').saidas == ["3"]),
    ("Declaração reinicializada no laço", teste_reinicializacao_laco),
    ("Saídas em ordem", lambda: executar_codigo('mostrar "a"; mostrar 2; mostrar falso;').saidas == ["a", "2", "falso"]),
    ("Cotação válida", lambda: executar_codigo('cotar "E", 2.0, 50.0, 1;').cotacoes == [CotacaoSimulada("E", 2.0, 50.0, 1)]),
    ("Despacho", lambda: executar_codigo('despachar "FS";').entregas == {"FS": "Despachado"}),
    ("Rastreamento válido", lambda: executar_codigo('despachar "FS"; rastrear "FS";').estatisticas.rastreamentos == 1),
    ("Recusa", lambda: executar_codigo('recusar "motivo"; mostrar "continua";').saidas == ["Recusa: motivo", "continua"]),
    ("Divisão por zero dinâmica", lambda: erro_categoria(RESULTADOS_ERROS_DINAMICOS["Divisão por zero"], "DIVISAO_POR_ZERO")),
    ("Rastreamento inexistente", lambda: erro_categoria(RESULTADOS_ERROS_DINAMICOS["Rastreamento inexistente"], "RASTREAMENTO_INEXISTENTE")),
    ("Peso inválido dinâmico", lambda: erro_categoria(RESULTADOS_ERROS_DINAMICOS["Peso inválido"], "PESO_INVALIDO")),
    ("Prazo inválido dinâmico", lambda: erro_categoria(RESULTADOS_ERROS_DINAMICOS["Prazo inválido"], "PRAZO_INVALIDO")),
    ("Aviso acima de 1000 sem duplicação", lambda: len(executar_codigo('cotar "E", 2.0, 1200.0, 1;').avisos_compilacao) == 1 and not executar_codigo('cotar "E", 2.0, 1200.0, 1;').avisos_execucao),
    ("Aviso dinâmico", lambda: len(executar_codigo(PROGRAMA_AVISO_VM).avisos_execucao) == 1),
    ("Limite de passos", lambda: erro_categoria(RESULTADOS_ERROS_DINAMICOS["Limite de passos"], "LIMITE_PASSOS")),
    ("Proteção contra valor não inicializado", lambda: erro_categoria(RESULTADO_NAO_INICIALIZADO, "VALOR_NAO_INICIALIZADO")),
    ("Memórias independentes", teste_memorias_independentes),
    ("Estatísticas por operação", lambda: EXECUCOES_ORIGINAIS[1].estatisticas.por_operacao["RASTREAR"] == 2 and EXECUCOES_ORIGINAIS[1].estatisticas.despachos == 1),
    ("Equivalência original 1", lambda: COMPARACOES_EXECUCAO["Original 1"].equivalentes),
    ("Equivalência original 2", lambda: COMPARACOES_EXECUCAO["Original 2"].equivalentes),
    ("Equivalência original 3", lambda: COMPARACOES_EXECUCAO["Original 3"].equivalentes),
    ("Equivalência demonstrativo A", lambda: COMPARACOES_EXECUCAO["Demonstrativo A"].equivalentes),
    ("Equivalência demonstrativo B", lambda: COMPARACOES_EXECUCAO["Demonstrativo B"].equivalentes),
    ("Equivalência com laço", lambda: COMPARACOES_EXECUCAO["Laço"].equivalentes),
    ("Equivalência curto-circuito", lambda: COMPARACOES_EXECUCAO["Curto-circuito"].equivalentes),
    ("Equivalência com erro", lambda: COMPARACOES_EXECUCAO["Erro dinâmico"].equivalentes),
    ("Equivalência com aviso", lambda: COMPARACOES_EXECUCAO["Aviso logístico"].equivalentes),
    ("Efeitos anteriores ao erro", lambda: (lambda c: c.equivalentes and c.original.entregas == {"FS": "Despachado"})(comparar_execucoes('despachar "FS"; rastrear "INEXISTENTE";'))),
    ("TAC não alterado pela execução", teste_tac_nao_alterado),
    ("Valores lógicos formatados", lambda: executar_codigo("mostrar verdadeiro; mostrar falso;").saidas == ["verdadeiro", "falso"]),
    ("Estatística dinâmica difere da estática", lambda: executar_codigo(PROGRAMA_LACO_VM).estatisticas.instrucoes_executadas > executar_codigo(PROGRAMA_LACO_VM).estatisticas.instrucoes_estaticas),
]

TESTES_R7_APROVADOS = 0
for nome, teste in CASOS_TESTE_VM:
    try:
        passou = bool(teste())
    except Exception as erro:
        passou = False
        print(f"FALHOU — {nome}: {type(erro).__name__}: {erro}")
    else:
        print(f"{'OK' if passou else 'FALHOU'} — {nome}")
    TESTES_R7_APROVADOS += int(passou)

TESTES_R7_EXECUTADOS = len(CASOS_TESTE_VM)
TESTES_R7_REPROVADOS = TESTES_R7_EXECUTADOS - TESTES_R7_APROVADOS
print(
    f"\nR7: {TESTES_R7_EXECUTADOS} executados, "
    f"{TESTES_R7_APROVADOS} aprovados e {TESTES_R7_REPROVADOS} reprovados."
)
assert TESTES_R7_REPROVADOS == 0, "Há testes da VM reprovados."

OK — Programa original 1
OK — Programa original 2
OK — Programa original 3
OK — Declaração e atribuição
OK — Declaração sem inicialização
OK — Coerção inteiro para decimal
OK — Operadores aritméticos
OK — Divisão produz decimal
OK — Comparações
OK — Operador nao
OK — Curto-circuito e
OK — Curto-circuito ou
OK — Se verdadeiro
OK — Se falso
OK — Se com senao
OK — Enquanto com iterações
OK — Enquanto zero iterações
OK — Laços aninhados
OK — Sombreamento
OK — Atribuição externa
OK — Declaração reinicializada no laço
OK — Saídas em ordem
OK — Cotação válida
OK — Despacho
OK — Rastreamento válido
OK — Recusa
OK — Divisão por zero dinâmica
OK — Rastreamento inexistente
OK — Peso inválido dinâmico
OK — Prazo inválido dinâmico
OK — Aviso acima de 1000 sem duplicação
OK — Aviso dinâmico
OK — Limite de passos
OK — Proteção contra valor não inicializado
OK — Memórias independentes
OK — Estatísticas por operação
OK — Equivalência original 1
OK — Equivalência original 2
OK — Equivalência original 3


## Resumo da máquina virtual

A VM executa localmente o TAC original e o otimizado. Os testes desta seção verificam resultados, diagnósticos, estatísticas e equivalência; a bateria de integração e o painel Jupyter são apresentados a seguir.

In [65]:
print(
    f"R7: {TESTES_R7_EXECUTADOS} executados — "
    f"{TESTES_R7_APROVADOS} aprovados e {TESTES_R7_REPROVADOS} reprovados."
)
print(
    f"Regressão R6: {TESTES_R6_EXECUTADOS} executados — "
    f"{TESTES_R6_APROVADOS} aprovados e {TESTES_R6_REPROVADOS} reprovados."
)
print(
    f"Regressão R5: {TESTES_R5_EXECUTADOS} executados — "
    f"{TESTES_R5_APROVADOS} aprovados e {TESTES_R5_REPROVADOS} reprovados."
)
print(
    f"Regressão R4: {TESTES_R4_EXECUTADOS} executados — "
    f"{TESTES_R4_APROVADOS} aprovados e {TESTES_R4_REPROVADOS} reprovados."
)
print(
    f"Regressão R3: {TESTES_R3_EXECUTADOS} executados — "
    f"{TESTES_R3_APROVADOS} aprovados e {TESTES_R3_REPROVADOS} reprovados."
)
print(
    f"Regressão R2: {TESTES_EXECUTADOS} executados — "
    f"{TESTES_APROVADOS} aprovados e {TESTES_REPROVADOS} reprovados."
)
assert TESTES_R6_REPROVADOS == 0
assert TESTES_R5_REPROVADOS == 0
assert TESTES_R4_REPROVADOS == 0
assert TESTES_R3_REPROVADOS == 0
assert TESTES_REPROVADOS == 0

R7: 49 executados — 49 aprovados e 0 reprovados.
Regressão R6: 36 executados — 36 aprovados e 0 reprovados.
Regressão R5: 36 executados — 36 aprovados e 0 reprovados.
Regressão R4: 35 executados — 35 aprovados e 0 reprovados.
Regressão R3: 25 executados — 25 aprovados e 0 reprovados.
Regressão R2: 17 executados — 17 aprovados e 0 reprovados.


# Etapa 08 — Testes finais de integração (R8)

Esta bateria verifica o percurso completo da FreteScript, desde os caracteres do código-fonte até os efeitos observáveis da máquina virtual. Os casos são novos e independentes das baterias R2–R7: cada execução começa novamente na fase adequada e não reutiliza resultados gravados anteriormente.

O material da TreinoScript pede pelo menos doze casos, quatro programas válidos e um erro de cada fase. A bateria abaixo amplia essa cobertura e inclui as regras de negócio, avisos e equivalência entre TAC original e otimizado.

## Estrutura dos casos

Cada cenário informa identificação, código, fase esperada e resultado verificável. O relatório guarda o resultado efetivamente observado e detalhes de qualquer falha. Exceções inesperadas sempre reprovam o cenário.

In [66]:
from dataclasses import dataclass, field


@dataclass(frozen=True)
class CasoIntegracao:
    identificador: str
    nome: str
    fase: str
    codigo: str
    esperado: dict
    max_passos: int = 10_000


@dataclass
class ResultadoCasoIntegracao:
    identificador: str
    nome: str
    fase: str
    esperado: str
    obtido: str
    aprovado: bool
    detalhes: str = ""
    equivalencia: str = "não aplicável"


def exigir_integracao(condicao, mensagem):
    if not condicao:
        raise AssertionError(mensagem)

## Programas válidos

Os seis programas percorrem R2, R3, R4, TAC original, otimização e duas VMs independentes. Além da aprovação das fases, são conferidos valores e efeitos específicos de cada cenário.

In [67]:
PROGRAMA_R8_MATEMATICA = r'''
inteiro base = 4;
decimal fator = 2.5;
decimal total = base * fator + 1;
mostrar total;
'''

PROGRAMA_R8_CONTROLE = r'''
inteiro i = 0;
enquanto i < 3 {
    se i == 1 {
        mostrar "meio";
    } senao {
        mostrar i;
    }
    i = i + 1;
}
'''

PROGRAMA_R8_ESCOPO = r'''
inteiro quantidade = 10;
se verdadeiro {
    inteiro quantidade = 2;
    mostrar quantidade;
}
mostrar quantidade;
'''

CASOS_VALIDOS_R8 = [
    CasoIntegracao("V1", "Cotação normal", "válido", PROGRAMA_1,
                   {"resumo": "cotação Economico, 4.5 kg, R$ 38,40 e prazo 3"}),
    CasoIntegracao("V2", "Despacho e rastreamento", "válido", PROGRAMA_2,
                   {"resumo": "Expresso, despacho, dois rastreamentos e duas iterações"}),
    CasoIntegracao("V3", "Recusa logística", "válido", PROGRAMA_3,
                   {"resumo": "uma recusa, sem cotação ou despacho"}),
    CasoIntegracao("V4", "Cálculo matemático e coerção", "válido", PROGRAMA_R8_MATEMATICA,
                   {"resumo": "saída numérica 11.0"}),
    CasoIntegracao("V5", "Condicional dentro de laço", "válido", PROGRAMA_R8_CONTROLE,
                   {"resumo": "saídas 0, meio, 2 e três iterações"}),
    CasoIntegracao("V6", "Escopos e sombreamento", "válido", PROGRAMA_R8_ESCOPO,
                   {"resumo": "saídas 2 e 10, sem mistura de escopos"}),
]

## Erros léxicos e sintáticos

Os erros léxicos devem impedir o parser. Nos sintáticos, o léxico precisa aceitar o código antes que o parser informe a posição, o elemento encontrado, os esperados e uma dica.

In [68]:
CASOS_LEXICOS_R8 = [
    CasoIntegracao("L1", "Caractere inválido", "léxica", "inteiro peso = 10 @ 2;",
                   {"resumo": "ErroLexico para @", "trecho": "@", "dica": "operador"}),
    CasoIntegracao("L2", "Aspas simples", "léxica", "texto modalidade = 'Expresso';",
                   {"resumo": "ErroLexico orientando aspas duplas", "trecho": "'Expresso'", "dica": "aspas duplas"}),
    CasoIntegracao("L3", "Texto não finalizado", "léxica", 'mostrar "Frete calculado;',
                   {"resumo": "ErroLexico indicando fechamento", "trecho": '"Frete calculado;', "dica": "aspa dupla"}),
]

CASOS_SINTATICOS_R8 = [
    CasoIntegracao("S1", "Ponto e vírgula ausente", "sintática",
                   "inteiro distancia = 450\nmostrar distancia;",
                   {"resumo": "ErroSintatico sugerindo ';'", "dica": ";"}),
    CasoIntegracao("S2", "Chave não fechada", "sintática",
                   'se verdadeiro {\n    mostrar "Entrega aprovada";',
                   {"resumo": "ErroSintatico sugerindo '}'", "dica": "}"}),
    CasoIntegracao("S3", "Expressão incompleta", "sintática",
                   "decimal valor = 10.0 + ;",
                   {"resumo": "ErroSintatico após operador", "dica": "expressão"}),
]

## Erros semânticos

Estes programas são válidos no léxico e na sintaxe. A análise semântica deve produzir o código exato do diagnóstico e impedir a geração de TAC.

In [69]:
CASOS_SEMANTICOS_R8 = [
    CasoIntegracao("M1", "Variável não declarada", "semântica", "mostrar distancia;",
                   {"resumo": "VAR_NAO_DECLARADA", "codigo": "VAR_NAO_DECLARADA"}),
    CasoIntegracao("M2", "Declaração duplicada", "semântica",
                   "inteiro peso = 5;\ninteiro peso = 10;",
                   {"resumo": "DECLARACAO_DUPLICADA", "codigo": "DECLARACAO_DUPLICADA"}),
    CasoIntegracao("M3", "Tipo incompatível", "semântica", 'inteiro prazo = "dois";',
                   {"resumo": "TIPO_INCOMPATIVEL", "codigo": "TIPO_INCOMPATIVEL"}),
    CasoIntegracao("M4", "Condição não lógica", "semântica",
                   'se 10 { mostrar "Invalido"; }',
                   {"resumo": "CONDICAO_NAO_LOGICA", "codigo": "CONDICAO_NAO_LOGICA"}),
    CasoIntegracao("M5", "Peso inválido", "semântica",
                   'cotar "Expresso", 0.0, 45.0, 2;',
                   {"resumo": "PESO_INVALIDO", "codigo": "PESO_INVALIDO"}),
    CasoIntegracao("M6", "Prazo inválido", "semântica",
                   'cotar "Economico", 2.0, 45.0, 0;',
                   {"resumo": "PRAZO_INVALIDO", "codigo": "PRAZO_INVALIDO"}),
]

## Erros de execução

Os cinco casos são aprovados até o TAC. Quatro falhas finitas também são comparadas entre original e otimizado. O laço infinito usa limite pequeno e sua comparação é registrada como inconclusiva, sem ser apresentada como equivalência funcional.

In [70]:
PROGRAMA_R8_DIVISAO_ZERO = r'''
inteiro divisor = 1;
enquanto divisor > 0 { divisor = divisor - 1; }
decimal resultado = 10.0 / divisor;
mostrar resultado;
'''

PROGRAMA_R8_PESO_DINAMICO = r'''
decimal peso = 1.0;
enquanto peso > 0.0 { peso = peso - 1.0; }
cotar "Expresso", peso, 45.0, 2;
'''

PROGRAMA_R8_PRAZO_DINAMICO = r'''
inteiro prazo = 2;
enquanto prazo > 0 { prazo = prazo - 1; }
cotar "Economico", 2.0, 50.0, prazo;
'''

CASOS_EXECUCAO_R8 = [
    CasoIntegracao("E1", "Divisão por zero dinâmica", "execução", PROGRAMA_R8_DIVISAO_ZERO,
                   {"resumo": "DIVISAO_POR_ZERO", "categoria": "DIVISAO_POR_ZERO"}),
    CasoIntegracao("E2", "Rastreamento inexistente", "execução", 'rastrear "FS-INEXISTENTE";',
                   {"resumo": "RASTREAMENTO_INEXISTENTE", "categoria": "RASTREAMENTO_INEXISTENTE"}),
    CasoIntegracao("E3", "Peso inválido dinâmico", "execução", PROGRAMA_R8_PESO_DINAMICO,
                   {"resumo": "PESO_INVALIDO", "categoria": "PESO_INVALIDO"}),
    CasoIntegracao("E4", "Prazo inválido dinâmico", "execução", PROGRAMA_R8_PRAZO_DINAMICO,
                   {"resumo": "PRAZO_INVALIDO", "categoria": "PRAZO_INVALIDO"}),
    CasoIntegracao("E5", "Limite de passos", "execução",
                   'enquanto verdadeiro { mostrar "Repetindo"; }',
                   {"resumo": "LIMITE_PASSOS e comparação inconclusiva", "categoria": "LIMITE_PASSOS"},
                   max_passos=20),
]

## Avisos, equivalência e reduções

O limite de R$ 1.000 é verificado dos dois lados. Valores conhecidos na compilação não podem duplicar o aviso na VM; valores descobertos durante a execução devem produzir um único aviso dinâmico. As reduções dos dois demonstrativos R6 também são recalculadas.

In [71]:
PROGRAMA_R8_AVISO_DINAMICO = r'''
decimal valor = 900.0;
enquanto valor < 1100.0 { valor = valor + 200.0; }
cotar "Expresso", 2.0, valor, 1;
'''

CASOS_AVISOS_R8 = [
    CasoIntegracao("A1", "Limite estático do aviso", "aviso",
                   'cotar "E", 2.0, 1000.01, 1;',
                   {"resumo": "1000 sem aviso; 1000.01 com um aviso de compilação",
                    "codigo_sem_aviso": 'cotar "E", 2.0, 1000.0, 1;'}),
    CasoIntegracao("A2", "Aviso conhecido somente em execução", "aviso",
                   PROGRAMA_R8_AVISO_DINAMICO,
                   {"resumo": "um FRETE_ELEVADO dinâmico, sem duplicação"}),
]

CASOS_INTEGRACAO_R8 = (
    CASOS_VALIDOS_R8 + CASOS_LEXICOS_R8 + CASOS_SINTATICOS_R8
    + CASOS_SEMANTICOS_R8 + CASOS_EXECUCAO_R8 + CASOS_AVISOS_R8
)

exigir_integracao(len(CASOS_INTEGRACAO_R8) == 25, "A bateria R8 deve possuir 25 cenários.")
exigir_integracao(len({c.identificador for c in CASOS_INTEGRACAO_R8}) == 25,
                  "Os identificadores dos cenários precisam ser únicos.")

## Execução independente da bateria

As funções seguintes não usam resultados das demonstrações anteriores. Para cada caso, elas invocam novamente as fases necessárias e verificam conteúdo, posição, dicas, estrutura do TAC e efeitos observáveis.

In [72]:
def _pipeline_integrado_valido(caso):
    tokens = analisar_lexico(caso.codigo)
    ast = analisar_sintaxe(caso.codigo)
    semantica = analisar_semantica(caso.codigo)
    exigir_integracao(bool(tokens), "O programa válido não produziu tokens.")
    exigir_integracao(isinstance(ast, Programa), "O parser não devolveu Programa.")
    exigir_integracao(semantica.aprovado, "A análise semântica reprovou o programa válido.")

    original = gerar_tac(caso.codigo)
    otimizado = otimizar_tac(original).otimizado
    exigir_integracao(not validar_tac(original), "O TAC original é inválido.")
    exigir_integracao(not validar_tac(otimizado), "O TAC otimizado é inválido.")
    execucao_original = executar_tac(original, caso.max_passos)
    execucao_otimizada = executar_tac(otimizado, caso.max_passos)
    comparacao = comparar_execucoes(caso.codigo, caso.max_passos)
    return semantica, original, otimizado, execucao_original, execucao_otimizada, comparacao


def _verificar_programa_valido(caso, execucao):
    exigir_integracao(execucao.sucesso, "A VM reprovou um programa válido.")
    if caso.identificador == "V1":
        exigir_integracao(len(execucao.cotacoes) == 1, "Era esperada uma cotação.")
        cotacao = execucao.cotacoes[0]
        exigir_integracao(
            cotacao.modalidade == "Economico" and abs(cotacao.peso_kg - 4.5) < 1e-9
            and abs(cotacao.valor_reais - 38.4) < 1e-9 and cotacao.prazo_dias == 3,
            "A cotação normal possui dados incorretos."
        )
        exigir_integracao(execucao.estatisticas.recusas == 0, "Houve recusa indevida.")
    elif caso.identificador == "V2":
        exigir_integracao(len(execucao.cotacoes) == 1 and execucao.cotacoes[0].modalidade == "Expresso",
                          "A cotação Expresso não foi registrada.")
        exigir_integracao(execucao.entregas == {"FS-2026-001": "Despachado"},
                          "O estado final da entrega está incorreto.")
        exigir_integracao(execucao.estatisticas.rastreamentos == 2, "Eram esperados dois rastreamentos.")
        exigir_integracao(execucao.estatisticas.iteracoes_laco == 2, "Eram esperadas duas iterações.")
    elif caso.identificador == "V3":
        exigir_integracao(execucao.estatisticas.recusas == 1, "Era esperada uma recusa.")
        exigir_integracao(not execucao.cotacoes and not execucao.entregas,
                          "Cotação ou despacho indevido no caminho de recusa.")
    elif caso.identificador == "V4":
        exigir_integracao(execucao.saidas == ["11.0"], "O cálculo deveria produzir 11.0.")
    elif caso.identificador == "V5":
        exigir_integracao(execucao.saidas == ["0", "meio", "2"], "A ordem das saídas está incorreta.")
        exigir_integracao(execucao.estatisticas.iteracoes_laco == 3, "O laço deveria iterar três vezes.")
    elif caso.identificador == "V6":
        exigir_integracao(execucao.saidas == ["2", "10"], "O sombreamento misturou as variáveis.")


def _executar_valido_r8(caso):
    _, tac_original, tac_otimizado, original, otimizada, comparacao = _pipeline_integrado_valido(caso)
    _verificar_programa_valido(caso, original)
    _verificar_programa_valido(caso, otimizada)
    if caso.identificador == "V4":
        exigir_integracao(
            any(i.operacao == "CONVERTER_DECIMAL" for i in tac_original.instrucoes)
            and any(i.operacao == "CONVERTER_DECIMAL" for i in tac_otimizado.instrucoes),
            "A coerção inteiro para decimal não foi preservada nos dois TACs."
        )
    exigir_integracao(comparacao.equivalentes and not comparacao.inconclusiva,
                      "TAC original e otimizado não foram equivalentes.")
    return "válido até a VM; resultado e equivalência confirmados", "equivalente"


def _executar_lexico_r8(caso):
    try:
        analisar_lexico(caso.codigo)
    except ErroLexico as erro:
        exigir_integracao(erro.linha > 0 and erro.coluna > 0, "Posição léxica inválida.")
        exigir_integracao(bool(erro.dica), "O erro léxico não possui dica.")
        exigir_integracao(caso.esperado["trecho"] in erro.trecho, "Trecho léxico incorreto.")
        exigir_integracao(caso.esperado["dica"].lower() in erro.dica.lower(), "Dica léxica incorreta.")
        try:
            analisar_sintaxe(caso.codigo)
        except ErroLexico:
            return f"ErroLexico em {erro.linha}:{erro.coluna}; pipeline interrompido", "não aplicável"
        raise AssertionError("O parser não preservou a rejeição léxica.")
    raise AssertionError("O código inválido foi aceito pelo léxico.")


def _executar_sintatico_r8(caso):
    tokens = analisar_lexico(caso.codigo)
    exigir_integracao(bool(tokens), "O léxico não aceitou o caso sintático.")
    try:
        analisar_sintaxe(caso.codigo)
    except ErroSintatico as erro:
        exigir_integracao(erro.linha > 0 and erro.coluna > 0, "Posição sintática inválida.")
        exigir_integracao(bool(erro.dica) and bool(erro.descricao), "Diagnóstico sintático incompleto.")
        exigir_integracao(caso.esperado["dica"].lower() in erro.dica.lower(), "Dica sintática incorreta.")
        try:
            gerar_tac(caso.codigo)
        except ErroSintatico:
            return f"ErroSintatico em {erro.linha}:{erro.coluna}; sem AST/TAC aprovado", "não aplicável"
        raise AssertionError("O pipeline gerou TAC para código sintaticamente inválido.")
    raise AssertionError("O parser aceitou o código sintaticamente inválido.")


def _executar_semantico_r8(caso):
    exigir_integracao(bool(analisar_lexico(caso.codigo)), "O léxico rejeitou o caso semântico.")
    exigir_integracao(isinstance(analisar_sintaxe(caso.codigo), Programa), "O parser rejeitou o caso semântico.")
    resultado = analisar_semantica(caso.codigo)
    diagnosticos = [d for d in resultado.erros if d.codigo == caso.esperado["codigo"]]
    exigir_integracao(not resultado.aprovado and diagnosticos, "O diagnóstico semântico esperado não apareceu.")
    diagnostico = diagnosticos[0]
    exigir_integracao(diagnostico.linha > 0 and diagnostico.coluna > 0,
                      "Posição semântica inválida.")
    exigir_integracao(bool(diagnostico.dica) and diagnostico.severidade == "erro",
                      "Diagnóstico semântico sem dica ou severidade incorreta.")
    try:
        gerar_tac(caso.codigo)
    except ErroGeracaoTAC:
        return f"{diagnostico.codigo} em {diagnostico.linha}:{diagnostico.coluna}; TAC bloqueado", "não aplicável"
    raise AssertionError("Foi gerado TAC após reprovação semântica.")

In [73]:
def _executar_erro_execucao_r8(caso):
    exigir_integracao(bool(analisar_lexico(caso.codigo)), "O léxico rejeitou o erro dinâmico.")
    exigir_integracao(isinstance(analisar_sintaxe(caso.codigo), Programa), "O parser rejeitou o erro dinâmico.")
    semantica = analisar_semantica(caso.codigo)
    exigir_integracao(semantica.aprovado, "O R4 antecipou indevidamente o erro dinâmico.")
    original = gerar_tac(caso.codigo)
    otimizado = otimizar_tac(original).otimizado
    exigir_integracao(not validar_tac(original) and not validar_tac(otimizado), "TAC inválido no erro dinâmico.")
    execucao_original = executar_tac(original, caso.max_passos)
    execucao_otimizada = executar_tac(otimizado, caso.max_passos)
    categoria = caso.esperado["categoria"]
    for resultado in (execucao_original, execucao_otimizada):
        exigir_integracao(not resultado.sucesso and resultado.erro is not None,
                          "A VM não produziu o erro esperado.")
        exigir_integracao(resultado.erro.categoria == categoria, "Categoria de execução incorreta.")
        exigir_integracao(resultado.erro.linha > 0 and resultado.erro.coluna > 0,
                          "Posição do erro de execução inválida.")
        exigir_integracao(bool(resultado.erro.dica), "Erro de execução sem dica.")

    comparacao = comparar_execucoes(caso.codigo, caso.max_passos)
    if categoria == "LIMITE_PASSOS":
        exigir_integracao(comparacao.inconclusiva and not comparacao.equivalentes,
                          "O limite de passos deveria tornar a comparação inconclusiva.")
        equivalencia = "inconclusiva por limite de passos"
    else:
        exigir_integracao(comparacao.equivalentes and not comparacao.inconclusiva,
                          "A falha dinâmica divergiu após otimização.")
        equivalencia = "equivalente até o erro"
    return f"{categoria} em {execucao_original.erro.linha}:{execucao_original.erro.coluna}", equivalencia


def _executar_aviso_r8(caso):
    if caso.identificador == "A1":
        codigo_limite = caso.esperado["codigo_sem_aviso"]
        sem_limite = analisar_semantica(codigo_limite)
        acima = analisar_semantica(caso.codigo)
        exigir_integracao(sem_limite.aprovado and not sem_limite.avisos,
                          "R$ 1.000 não deveria gerar aviso.")
        exigir_integracao(acima.aprovado and len(acima.avisos) == 1,
                          "R$ 1.000,01 deveria gerar um aviso.")
        exigir_integracao(acima.avisos[0].codigo == "FRETE_ELEVADO"
                          and acima.avisos[0].severidade == "aviso",
                          "Código ou severidade do aviso está incorreto.")
        execucao = executar_fretescript(caso.codigo)
        exigir_integracao(execucao.sucesso and len(execucao.avisos_compilacao) == 1
                          and not execucao.avisos_execucao,
                          "O aviso conhecido foi perdido ou duplicado na VM.")
        comparacao_limite = comparar_execucoes(codigo_limite)
        comparacao_acima = comparar_execucoes(caso.codigo)
        exigir_integracao(comparacao_limite.equivalentes and comparacao_acima.equivalentes,
                          "A otimização alterou o comportamento dos limites do aviso.")
        return "1000 sem aviso; 1000.01 com um aviso não duplicado", "equivalente"

    semantica = analisar_semantica(caso.codigo)
    exigir_integracao(semantica.aprovado and not semantica.avisos,
                      "O aviso dinâmico foi antecipado sem valor seguro.")
    original = executar_fretescript(caso.codigo)
    otimizada = executar_fretescript(caso.codigo, otimizar=True)
    for resultado in (original, otimizada):
        exigir_integracao(resultado.sucesso and not resultado.avisos_compilacao,
                          "O programa com aviso dinâmico deveria ser válido.")
        exigir_integracao(len(resultado.avisos_execucao) == 1
                          and resultado.avisos_execucao[0].categoria == "FRETE_ELEVADO",
                          "Era esperado um único aviso dinâmico FRETE_ELEVADO.")
    comparacao = comparar_execucoes(caso.codigo)
    exigir_integracao(comparacao.equivalentes, "O aviso dinâmico divergiu após otimização.")
    return "um FRETE_ELEVADO em execução, sem duplicação", "equivalente"


def executar_caso_integracao(caso):
    despachantes = {
        "válido": _executar_valido_r8,
        "léxica": _executar_lexico_r8,
        "sintática": _executar_sintatico_r8,
        "semântica": _executar_semantico_r8,
        "execução": _executar_erro_execucao_r8,
        "aviso": _executar_aviso_r8,
    }
    try:
        obtido, equivalencia = despachantes[caso.fase](caso)
    except Exception as erro:
        return ResultadoCasoIntegracao(
            caso.identificador, caso.nome, caso.fase, caso.esperado["resumo"],
            f"{type(erro).__name__}: {erro}", False,
            "Exceção inesperada ou expectativa não atendida.", "não comprovada"
        )
    return ResultadoCasoIntegracao(
        caso.identificador, caso.nome, caso.fase, caso.esperado["resumo"],
        obtido, True, "", equivalencia
    )


def executar_testes_integracao():
    return [executar_caso_integracao(caso) for caso in CASOS_INTEGRACAO_R8]

## Tabela de resultados e resumo final

A tabela é produzida pelos resultados reais desta execução. Uma reprovação deixa a última validação em estado de falha; não há conclusão positiva automática quando algum cenário obrigatório falha.

In [74]:
RESULTADOS_R8 = executar_testes_integracao()


def mostrar_tabela_integracao(resultados):
    print(f"{'ID':<3} | {'Cenário':<34} | {'Fase':<10} | {'Esperado':<48} | {'Obtido':<62} | Situação")
    print("-" * 182)
    for resultado in resultados:
        situacao = "APROVADO" if resultado.aprovado else "REPROVADO"
        print(
            f"{resultado.identificador:<3} | {resultado.nome[:34]:<34} | "
            f"{resultado.fase:<10} | {resultado.esperado[:48]:<48} | "
            f"{resultado.obtido[:62]:<62} | {situacao}"
        )
        if resultado.detalhes:
            print("    Detalhes:", resultado.detalhes)


mostrar_tabela_integracao(RESULTADOS_R8)

TESTES_R8_EXECUTADOS = len(RESULTADOS_R8)
TESTES_R8_APROVADOS = sum(r.aprovado for r in RESULTADOS_R8)
TESTES_R8_REPROVADOS = TESTES_R8_EXECUTADOS - TESTES_R8_APROVADOS

CONTAGENS_R8 = {
    "programas válidos": len(CASOS_VALIDOS_R8),
    "erros léxicos": len(CASOS_LEXICOS_R8),
    "erros sintáticos": len(CASOS_SINTATICOS_R8),
    "erros semânticos": len(CASOS_SEMANTICOS_R8),
    "erros de execução": len(CASOS_EXECUCAO_R8),
    "testes de avisos": len(CASOS_AVISOS_R8),
}
EQUIVALENCIAS_R8 = sum(r.equivalencia.startswith("equivalente") for r in RESULTADOS_R8)
INCONCLUSIVAS_R8 = sum(r.equivalencia.startswith("inconclusiva") for r in RESULTADOS_R8)

DEMO_A_R8 = compilar_otimizado(PROGRAMA_OTIMIZACAO_A)
DEMO_B_R8 = compilar_otimizado(PROGRAMA_OTIMIZACAO_B)
REDUCOES_R6_R8 = {
    "Demonstrativo A": (len(DEMO_A_R8.original.instrucoes), len(DEMO_A_R8.otimizado.instrucoes)),
    "Demonstrativo B": (len(DEMO_B_R8.original.instrucoes), len(DEMO_B_R8.otimizado.instrucoes)),
}
exigir_integracao(all(depois < antes for antes, depois in REDUCOES_R6_R8.values()),
                  "Os demonstrativos R6 deixaram de apresentar redução real.")

print("\nR8 — Testes finais de integração")
print(f"Executados: {TESTES_R8_EXECUTADOS}")
print(f"Aprovados: {TESTES_R8_APROVADOS}")
print(f"Reprovados: {TESTES_R8_REPROVADOS}")
for categoria, quantidade in CONTAGENS_R8.items():
    print(f"{categoria.capitalize()}: {quantidade}")
print(f"Comparações conclusivas equivalentes: {EQUIVALENCIAS_R8}")
print(f"Comparações inconclusivas esperadas: {INCONCLUSIVAS_R8}")
for nome, (antes, depois) in REDUCOES_R6_R8.items():
    print(f"{nome}: {antes} → {depois} instruções")

exigir_integracao(TESTES_R8_REPROVADOS == 0,
                  f"Há {TESTES_R8_REPROVADOS} cenário(s) R8 reprovado(s).")

ID  | Cenário                            | Fase       | Esperado                                         | Obtido                                                         | Situação
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
V1  | Cotação normal                     | válido     | cotação Economico, 4.5 kg, R$ 38,40 e prazo 3    | válido até a VM; resultado e equivalência confirmados          | APROVADO
V2  | Despacho e rastreamento            | válido     | Expresso, despacho, dois rastreamentos e duas it | válido até a VM; resultado e equivalência confirmados          | APROVADO
V3  | Recusa logística                   | válido     | uma recusa, sem cotação ou despacho              | válido até a VM; resultado e equivalência confirmados          | APROVADO
V4  | Cálculo matemático e coerção       | válido     | saída numérica 11.0                  

In [75]:
REGRESSOES_R8 = {
    "R7": (TESTES_R7_EXECUTADOS, TESTES_R7_APROVADOS, TESTES_R7_REPROVADOS),
    "R6": (TESTES_R6_EXECUTADOS, TESTES_R6_APROVADOS, TESTES_R6_REPROVADOS),
    "R5": (TESTES_R5_EXECUTADOS, TESTES_R5_APROVADOS, TESTES_R5_REPROVADOS),
    "R4": (TESTES_R4_EXECUTADOS, TESTES_R4_APROVADOS, TESTES_R4_REPROVADOS),
    "R3": (TESTES_R3_EXECUTADOS, TESTES_R3_APROVADOS, TESTES_R3_REPROVADOS),
    "R2": (TESTES_EXECUTADOS, TESTES_APROVADOS, TESTES_REPROVADOS),
}

exigir_integracao(conferir_ficha(MINHA_FICHA), "A ficha R1 deixou de ser válida.")
for etapa, (executados, aprovados, reprovados) in REGRESSOES_R8.items():
    exigir_integracao(reprovados == 0 and aprovados == executados,
                      f"A regressão {etapa} possui falhas.")

TOTAL_ANTERIOR_R8 = sum(executados for executados, _, _ in REGRESSOES_R8.values())
TOTAL_GERAL_R8 = TOTAL_ANTERIOR_R8 + TESTES_R8_EXECUTADOS

print(f"R8: {TESTES_R8_APROVADOS}/{TESTES_R8_EXECUTADOS} aprovados.")
for etapa, (executados, aprovados, reprovados) in REGRESSOES_R8.items():
    print(f"Regressão {etapa}: {aprovados}/{executados} aprovados; {reprovados} reprovados.")
print(f"Testes anteriores confirmados: {TOTAL_ANTERIOR_R8}/198.")
print(f"Total acumulado R2–R8: {TOTAL_GERAL_R8} testes aprovados.")

Ficha conferida: mínimos documentais atendidos.
R8: 25/25 aprovados.
Regressão R7: 49/49 aprovados; 0 reprovados.
Regressão R6: 36/36 aprovados; 0 reprovados.
Regressão R5: 36/36 aprovados; 0 reprovados.
Regressão R4: 35/35 aprovados; 0 reprovados.
Regressão R3: 25/25 aprovados; 0 reprovados.
Regressão R2: 17/17 aprovados; 0 reprovados.
Testes anteriores confirmados: 198/198.
Total acumulado R2–R8: 223 testes aprovados.


## Resultado da Etapa 08

A bateria final de integração confirma o encadeamento das fases, os diagnósticos específicos, as regras logísticas e a equivalência observável entre TAC original e otimizado. O limite de passos permanece corretamente classificado como comparação inconclusiva quando impede o término do programa.

A interface Jupyter para explorar esses resultados aparece na seção seguinte.

# Etapa 09 — Interface Jupyter

O painel reúne as fases do compilador em cinco abas. Selecione um exemplo e clique em **Carregar exemplo**, ou escreva seu próprio programa. Os botões analisam, executam e comparam as duas versões do TAC.

In [76]:
import ipywidgets as widgets
from IPython.display import display

EXEMPLOS_PAINEL = {
    "Cotação normal": PROGRAMA_1,
    "Despacho com rastreamento": PROGRAMA_2,
    "Recusa logística": PROGRAMA_3,
    "Cálculo matemático": PROGRAMA_R8_MATEMATICA,
    "Condicional e repetição": PROGRAMA_R8_CONTROLE,
    "Erro semântico": 'inteiro prazo = "dois";',
    "Erro de execução": 'rastrear "FS-INEXISTENTE";',
}

seletor_exemplo = widgets.Dropdown(options=list(EXEMPLOS_PAINEL), description="Exemplo:", layout=widgets.Layout(width="420px"))
botao_carregar = widgets.Button(description="Carregar exemplo")
editor_codigo = widgets.Textarea(value=PROGRAMA_1, description="Código:", layout=widgets.Layout(width="100%", height="230px"))
selecionar_otimizado = widgets.Checkbox(value=False, description="Executar TAC otimizado")
botao_analisar = widgets.Button(description="Analisar", button_style="info")
botao_executar = widgets.Button(description="Executar", button_style="success")
botao_comparar = widgets.Button(description="Comparar TAC")
mensagens_painel = widgets.Output()
saidas_painel = {nome: widgets.Output() for nome in ("Tokens", "AST", "Semântica", "TAC", "Execução")}
estado_painel = {"fase": None, "execucao": None, "comparacao": None, "tac": None, "semantica": None}
abas_painel = widgets.Tab(children=list(saidas_painel.values()))
for indice, nome in enumerate(saidas_painel):
    abas_painel.set_title(indice, nome)

def limpar_painel():
    estado_painel.update({chave: None for chave in estado_painel})
    mensagens_painel.outputs = ()
    for saida in saidas_painel.values():
        saida.outputs = ()

def carregar_exemplo(_=None):
    editor_codigo.value = EXEMPLOS_PAINEL[seletor_exemplo.value]
    limpar_painel()

botao_carregar.on_click(carregar_exemplo)
editor_codigo.observe(lambda _: limpar_painel(), names="value")
selecionar_otimizado.observe(lambda _: limpar_painel(), names="value")

## Apresentação das fases

Os resultados são obtidos do código presente no editor. Um erro interrompe apenas as fases seguintes; os resultados da nova tentativa substituem os anteriores.

In [77]:
import contextlib
import io

def _enviar_texto_painel(widget, funcao):
    captura = io.StringIO()
    with contextlib.redirect_stdout(captura):
        funcao()
    texto = captura.getvalue()
    if texto:
        widget.append_stdout(texto)

def _mensagem_painel(texto):
    mensagens_painel.append_stdout(texto + "\n")

def _tokens_painel(fonte):
    _enviar_texto_painel(saidas_painel["Tokens"], lambda: mostrar_tabela_tokens(fonte))

def _ast_painel(arvore):
    _enviar_texto_painel(saidas_painel["AST"], lambda: mostrar_ast(arvore))

def _mostrar_semantica(resultado):
    print("Situação:", "aprovada" if resultado.aprovado else "reprovada")
    mostrar_tabela_simbolos(resultado.tabela)
    for diagnostico in resultado.erros:
        print("ERRO:", diagnostico)
    for diagnostico in resultado.avisos:
        print("AVISO:", diagnostico)

def _semantica_painel(resultado):
    _enviar_texto_painel(saidas_painel["Semântica"], lambda: _mostrar_semantica(resultado))

def _mostrar_tac_painel(original, otimizado):
    mostrar_tac(original, "TAC original")
    if otimizado is not None:
        mostrar_tac(otimizado, "TAC otimizado")
        antes, depois = len(original.instrucoes), len(otimizado.instrucoes)
        percentual = (antes - depois) / antes * 100 if antes else 0
        print(f"Instruções: {antes} → {depois}; redução: {antes - depois} ({percentual:.1f}%).")

def _tac_painel(original, otimizado=None):
    _enviar_texto_painel(saidas_painel["TAC"], lambda: _mostrar_tac_painel(original, otimizado))

def _mostrar_execucao_painel(resultado):
    mostrar_execucao(resultado)
    print("Cotações:", resultado.cotacoes)
    print("Eventos:", resultado.eventos)
    print("Entregas:", resultado.entregas)

def _execucao_painel(resultado):
    _enviar_texto_painel(saidas_painel["Execução"], lambda: _mostrar_execucao_painel(resultado))

def _analisar_fonte_painel(fonte):
    _tokens_painel(fonte)
    estado_painel["fase"] = "léxico"
    arvore = analisar_sintaxe(fonte)
    _ast_painel(arvore)
    estado_painel["fase"] = "sintático"
    semantica = analisar_semantica(fonte)
    estado_painel["semantica"] = semantica
    _semantica_painel(semantica)
    if semantica.aprovado:
        original = gerar_tac(fonte)
        estado_painel["tac"] = original
        _tac_painel(original)
        return original
    return None

def acionar_painel(acao):
    limpar_painel()
    fonte = editor_codigo.value
    try:
        original = _analisar_fonte_painel(fonte)
        if original is None:
            _mensagem_painel("Erro semântico: confira a aba Semântica e corrija o código.")
            return "semântica"
        if acao == "analisar":
            _mensagem_painel("Análise concluída. O programa não foi executado.")
            return "analisado"
        if acao == "executar":
            tac = otimizar_tac(original).otimizado if selecionar_otimizado.value else original
            resultado = executar_tac(tac)
            estado_painel["execucao"] = resultado
            _execucao_painel(resultado)
            _mensagem_painel("Execução concluída." if resultado.sucesso else f"Erro de execução: {resultado.erro}")
            return "executado" if resultado.sucesso else "execução"
        if acao == "comparar":
            otimizacao = otimizar_tac(original)
            _tac_painel(original, otimizacao.otimizado)
            comparacao = comparar_execucoes(fonte)
            estado_painel["comparacao"] = comparacao
            situacao = "INCONCLUSIVO" if comparacao.inconclusiva else "EQUIVALENTE" if comparacao.equivalentes else "DIVERGENTE"
            _mensagem_painel("Original x otimizado: " + situacao)
            _enviar_texto_painel(saidas_painel["Execução"], lambda: mostrar_comparacao_execucoes(comparacao))
            return situacao
        raise ValueError(f"Ação desconhecida: {acao}")
    except ErroLexico as erro:
        estado_painel["fase"] = "léxico"
        _mensagem_painel("Erro léxico: " + str(erro))
        return "léxico"
    except ErroSintatico as erro:
        estado_painel["fase"] = "sintático"
        _mensagem_painel("Erro sintático: " + str(erro))
        return "sintático"

botao_analisar.on_click(lambda _: acionar_painel("analisar"))
botao_executar.on_click(lambda _: acionar_painel("executar"))
botao_comparar.on_click(lambda _: acionar_painel("comparar"))

## Painel FreteScript

O editor pode ser alterado livremente. A seleção no menu só muda o exemplo escolhido; o botão **Carregar exemplo** substitui o texto do editor.

In [78]:
painel_fretescript = widgets.VBox([
    widgets.HTML("<h2>FreteScript — Compilador de Frete e Logística</h2><p>Uma linguagem criada para simular cotações, despachos, recusas e rastreamento de entregas.</p>"),
    widgets.HBox([seletor_exemplo, botao_carregar]),
    editor_codigo,
    widgets.HBox([botao_analisar, botao_executar, botao_comparar, selecionar_otimizado]),
    widgets.HTML("<h4>Mensagens e diagnósticos</h4>"),
    mensagens_painel,
    abas_painel,
])
display(painel_fretescript)

## Verificações dos controles

Os testes acionam as mesmas funções usadas pelos botões. A parte visual no navegador ainda precisa de conferência manual.

In [79]:
def texto_widget(widget):
    return "".join(output.get("text", "") for output in widget.outputs if output.get("output_type") == "stream")

def testar_painel():
    resultados = []
    original = editor_codigo.value
    escolha = seletor_exemplo.value
    otimizado = selecionar_otimizado.value
    def verificar(nome, condicao):
        resultados.append((nome, bool(condicao)))
    try:
        verificar("controles", all(isinstance(x, widgets.Widget) for x in (seletor_exemplo, editor_codigo, botao_analisar, botao_executar, botao_comparar, abas_painel)))
        verificar("cinco abas", [abas_painel.get_title(i) for i in range(5)] == list(saidas_painel))
        editor_codigo.value = "mostrar 7;"
        seletor_exemplo.value = "Recusa logística"
        verificar("seleção preserva texto", editor_codigo.value == "mostrar 7;")
        verificar("seleção não executa", not texto_widget(mensagens_painel) and all(not texto_widget(x) for x in saidas_painel.values()))
        botao_carregar.click()
        verificar("carregamento explícito sem execução", editor_codigo.value == PROGRAMA_3 and not texto_widget(saidas_painel["Execução"]))
        editor_codigo.value = PROGRAMA_1
        botao_analisar.click()
        verificar("mensagem após Analisar", "Análise concluída. O programa não foi executado." in texto_widget(mensagens_painel))
        verificar("tokens visíveis", "Linha" in texto_widget(saidas_painel["Tokens"]) and "TIPO_DECIMAL" in texto_widget(saidas_painel["Tokens"]))
        verificar("AST visível", "Programa" in texto_widget(saidas_painel["AST"]))
        verificar("semântica visível", "Situação: aprovada" in texto_widget(saidas_painel["Semântica"]))
        verificar("TAC visível", "TAC original" in texto_widget(saidas_painel["TAC"]) and "COTAR" in texto_widget(saidas_painel["TAC"]))
        verificar("Analisar não executa", not texto_widget(saidas_painel["Execução"]) and estado_painel["execucao"] is None)
        botao_executar.click()
        execucao = texto_widget(saidas_painel["Execução"])
        verificar("execução visível", "Economico" in execucao and "4.5 kg" in execucao and "38.40" in execucao and "3 dia(s)" in execucao and "sucesso" in execucao)
        verificar("mensagem de execução", "Execução concluída." in texto_widget(mensagens_painel))
        botao_comparar.click()
        tac = texto_widget(saidas_painel["TAC"])
        verificar("comparação visível", "TAC original" in tac and "TAC otimizado" in tac and "Instruções:" in tac and "EQUIVALENTES" in texto_widget(saidas_painel["Execução"]))
        verificar("mensagem de equivalência", "EQUIVALENTE" in texto_widget(mensagens_painel))
        selecionar_otimizado.value = True
        botao_executar.click()
        verificar("TAC otimizado selecionável", "Economico" in texto_widget(saidas_painel["Execução"]) and estado_painel["execucao"].sucesso)
        for fonte, categoria, trecho in [
            ("inteiro x = @;", "léxico", "Erro léxico"),
            ("inteiro x = ;", "sintático", "Erro sintático"),
            ('inteiro x = "texto";', "semântica", "Erro semântico"),
            ('rastrear "INEXISTENTE";', "execução", "RASTREAMENTO_INEXISTENTE"),
        ]:
            editor_codigo.value = fonte
            botao_executar.click()
            verificar("erro visível " + categoria, trecho in texto_widget(mensagens_painel) and "Dica:" in (texto_widget(mensagens_painel) + texto_widget(saidas_painel["Semântica"])))
        editor_codigo.value = 'cotar "E", 2.0, 1200.0, 1;'
        botao_analisar.click()
        verificar("aviso visível", "AVISO:" in texto_widget(saidas_painel["Semântica"]) and "1.000" in texto_widget(saidas_painel["Semântica"]))
        editor_codigo.value = "mostrar 9;"
        verificar("limpeza ao editar", not texto_widget(mensagens_painel) and all(not texto_widget(x) for x in saidas_painel.values()))
        botao_executar.click()
        primeira = texto_widget(saidas_painel["Execução"])
        botao_executar.click()
        segunda = texto_widget(saidas_painel["Execução"])
        verificar("execuções independentes", primeira == segunda and "9" in segunda and "1200" not in segunda)
    finally:
        seletor_exemplo.value = escolha
        selecionar_otimizado.value = otimizado
        editor_codigo.value = original
        limpar_painel()
    for nome, passou in resultados:
        print(("OK" if passou else "FALHOU") + " — " + nome)
    print(f"R9.1: {sum(ok for _, ok in resultados)}/{len(resultados)} testes aprovados.")
    assert all(ok for _, ok in resultados), "Há falhas na exibição do painel."
    return resultados

RESULTADOS_TESTES_R91 = testar_painel()


OK — controles
OK — cinco abas
OK — seleção preserva texto
OK — seleção não executa
OK — carregamento explícito sem execução
OK — mensagem após Analisar
OK — tokens visíveis
OK — AST visível
OK — semântica visível
OK — TAC visível
OK — Analisar não executa
OK — execução visível
OK — mensagem de execução
OK — comparação visível
OK — mensagem de equivalência
OK — TAC otimizado selecionável
OK — erro visível léxico
OK — erro visível sintático
OK — erro visível semântica
OK — erro visível execução
OK — aviso visível
OK — limpeza ao editar
OK — execuções independentes
R9.1: 23/23 testes aprovados.


## Resultado

O painel está pronto para uso interativo após **Executar tudo**. O notebook validou os callbacks automaticamente; a interação visual também foi conferida no JupyterLab.